# 01 — Preparação e construção inicial da tabela mestra

**Escopo do notebook:** inspeção das fontes, preparação das identificações e construção de um calendário diário inicial com contagens de alarmes. O alvo, as sequências e o modelo serão desenvolvidos em etapas posteriores. Os arquivos de origem permanecem intactos.

A futura unidade de análise será **estação + plataforma + porta + dia**, verificando se o tipo de ativo também precisa compor a chave. O alvo pretendido é a ocorrência de falha entre D+1 e D+7. ZM01 é evidência principal; ZM02 e ZM03 descrevem histórico de manutenção. Manutenção programada ou preventiva não equivale automaticamente a falha.

O notebook utiliza exclusivamente os quatro arquivos tratados em `data/`. As abas de auditoria são inspecionadas separadamente, sem concatená-las às bases principais.

**Estado atual:** as seções 47 a 54 preparam uma referência por registro ZM01 (proxy), não um alvo validado de falha real. As seções seguintes reúnem evidências de defeitos e alarmes para a revisão de episódios.

In [1]:
from pathlib import Path
import re
import sys
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 100)

# Funciona iniciando o Jupyter na raiz ou em uma subpasta do repositório.
candidatos = [Path.cwd(), *Path.cwd().parents]
RAIZ = next((p for p in candidatos
             if (p / "data/Corretiva Emergencial/processados/ZM01_limpa.xlsx").is_file()), None)
if RAIZ is None:
    raise FileNotFoundError("O notebook precisa ser executado a partir do repositório que contém data/.")

FONTES = {
    "Alarmes": (RAIZ / "data/Alarmes/alarmes_trata_limpos_consolidado.csv", None),
    "ZM01": (RAIZ / "data/Corretiva Emergencial/processados/ZM01_limpa.xlsx", "ZM01_limpa"),
    "ZM02": (RAIZ / "data/Corretiva Programada/processados/ZM02_limpa.xlsx", "ZM02_limpa"),
    "ZM03": (RAIZ / "data/Preventiva/preventiva_zm03_limpa.csv", None),
}
for nome, (arquivo, _) in FONTES.items():
    if not arquivo.is_file():
        raise FileNotFoundError(f"{nome}: {arquivo}")
print(f"Python: {sys.version.split()[0]} | pandas: {pd.__version__}")
print(f"Raiz: {RAIZ}")


Python: 3.13.14 | pandas: 2.3.1
Raiz: C:\Users\Inteli\Downloads\EC-M7\g04


## 1. Inventário e leitura

Os CSVs são lidos em UTF-8. As abas são escolhidas explicitamente para não confundir a base limpa com auditorias ou subconjuntos. A contagem de linhas abaixo corresponde aos registros lidos, não à quantidade de episódios de falha.

In [2]:
bases = {}
auditorias = {}
inventario = []
for nome, (arquivo, principal) in FONTES.items():
    if principal is None:
        df = pd.read_csv(arquivo, encoding="utf-8-sig", low_memory=False)
        bases[nome] = df
        inventario.append({"fonte": nome, "arquivo": str(arquivo.relative_to(RAIZ)),
                           "aba": "(CSV)", "papel": "principal",
                           "linhas": len(df), "colunas": len(df.columns)})
    else:
        with pd.ExcelFile(arquivo) as livro:
            for aba in livro.sheet_names:
                df = pd.read_excel(livro, sheet_name=aba)
                if aba == principal:
                    bases[nome] = df
                else:
                    auditorias[(nome, aba)] = df
                inventario.append({"fonte": nome, "arquivo": str(arquivo.relative_to(RAIZ)),
                                   "aba": aba, "papel": "principal" if aba == principal else "auditoria",
                                   "linhas": len(df), "colunas": len(df.columns)})
display(pd.DataFrame(inventario))
for (nome, aba), df in auditorias.items():
    if aba == "Decisoes_limpeza":
        print(f"Decisões de limpeza — {nome}")
        display(df)


,fonte,arquivo,aba,papel,linhas,colunas
0,Alarmes,data\Alarmes\alarmes_trata_limpos_consolidado.csv,(CSV),principal,742506,15
1,ZM01,data\Corretiva Emergencial\processados\ZM01_limpa.xlsx,ZM01_limpa,principal,10543,27
2,ZM01,data\Corretiva Emergencial\processados\ZM01_limpa.xlsx,Outliers_removidos,auditoria,103,27
3,ZM01,data\Corretiva Emergencial\processados\ZM01_limpa.xlsx,Qualidade_ausencias,auditoria,27,3
4,ZM01,data\Corretiva Emergencial\processados\ZM01_limpa.xlsx,Decisoes_limpeza,auditoria,7,2
5,ZM02,data\Corretiva Programada\processados\ZM02_limpa.xlsx,ZM02_limpa,principal,1163,27
6,ZM02,data\Corretiva Programada\processados\ZM02_limpa.xlsx,Duracoes_extremas,auditoria,12,27
7,ZM02,data\Corretiva Programada\processados\ZM02_limpa.xlsx,Duracoes_zero,auditoria,5,27
8,ZM02,data\Corretiva Programada\processados\ZM02_limpa.xlsx,Qualidade_ausencias,auditoria,27,3
9,ZM02,data\Corretiva Programada\processados\ZM02_limpa.xlsx,Decisoes_limpeza,auditoria,9,2


Decisões de limpeza — ZM01


,campo_ou_regra,decisao
0,iqm,Coluna removida por conter 100% de valores ausentes.
1,duplicatas,Removidas 1 linha(s) exatamente duplicada(s).
2,duracao_avaria_horas,"Removidos 103 registros acima do percentil 99 (25.00 horas), preservados separadamente para audi..."
3,fim_da_avaria,Ausências preservadas; não foi realizada imputação.
4,data_analise,Ausências preservadas; marca e data de análise não são equivalentes.
5,campos_catalogo,"Ausências de objeto, defeito, causa e ação foram preservadas."
6,ordens_ausentes,Registros sem ordem foram mantidos quando possuíam nota e dados úteis.


Decisões de limpeza — ZM02


,campo_ou_regra,decisao
0,falha,Removida por conter 100% de valores ausentes.
1,iqm,Removida por conter 100% de valores ausentes.
2,duplicatas,Nenhuma duplicata exata encontrada após a limpeza.
3,ordens_repetidas,Cinco ordens repetidas foram mantidas porque apresentam divergências em campos técnicos.
4,ordens_ausentes,"Cinco registros sem ordem foram mantidos por possuírem nota, descrição e localização."
5,valores_ausentes,As ausências foram preservadas sem imputação.
6,duracao_extrema,Foram sinalizados 12 registros acima do percentil 99 (6104.70 horas). Os registros foram mantido...
7,duracao_zero,"Foram sinalizados 5 registros com duração igual a zero, mantidos sem imputação."
8,colunas_esparsas,As colunas comentarios_classificacao_engenharia e analise_datetime foram removidas da base final...


## 2. Colunas, tipos e ausências

Os tipos são inferidos na leitura. Esta inspeção preserva os valores ausentes e todos os registros. As amostras exibem somente identificação e datas, evitando reproduzir textos longos das ordens.

In [3]:
AMOSTRAS = {
    "Alarmes": ["TAG", "estacao_extraida", "plataforma_extraida", "porta_extraida",
                "tipo_evento_extraido", "dt_alarme", "dt_normalizacao"],
    "ZM01": ["nota", "ordem", "local_de_instalacao", "inicio_avaria_datetime", "fim_avaria_datetime"],
    "ZM02": ["nota", "ordem", "local_de_instalacao", "inicio_avaria_datetime", "fim_avaria_datetime"],
    "ZM03": ["Ordem", "Local de instalação", "DT_Inicio", "DT_Fim", "cancelada"],
}
for nome, df in bases.items():
    print(f"\n{nome}")
    perfil = pd.DataFrame({
        "coluna": df.columns,
        "tipo_lido": df.dtypes.astype(str).values,
        "ausentes": df.isna().sum().values,
        "ausentes_pct": df.isna().mean().mul(100).round(2).values,
        "strings_vazias": [df[c].map(lambda v: isinstance(v, str) and not v.strip()).sum()
                           if df[c].dtype == "object" else 0 for c in df.columns],
    })
    display(perfil)
    display(df[AMOSTRAS[nome]].head(5))



Alarmes


,coluna,tipo_lido,ausentes,ausentes_pct,strings_vazias
0,TAG,object,0,0.00,0
1,Horário do Alarme,object,0,0.00,0
2,Estação,object,0,0.00,0
3,Porta,float64,170,0.02,0
4,Alarme,object,0,0.00,0
5,Horário da Normalizacao,object,0,0.00,0
6,Tempo de Normalização,object,0,0.00,0
7,tempo_normalizacao_seg,int64,0,0.00,0
8,dt_alarme,object,0,0.00,0
9,dt_normalizacao,object,0,0.00,0


,TAG,estacao_extraida,plataforma_extraida,porta_extraida,tipo_evento_extraido,dt_alarme,dt_normalizacao
0,PSD_FAL_FAL2_PDM09_OBS,FAL,FAL2,9.0,OBS,2026-05-12 08:16:52.657,2026-05-12 08:16:53.670
1,PSD_LUZ_LUZ1_PDM08_OBS,LUZ,LUZ1,8.0,OBS,2026-05-12 08:15:09.913,2026-05-12 08:15:10.917
2,PSD_FAL_FAL1_PDM15_OBS,FAL,FAL1,15.0,OBS,2026-05-12 08:14:06.657,2026-05-12 08:14:07.670
3,PSD_LUZ_LUZ1_PDM19_OBS,LUZ,LUZ1,19.0,OBS,2026-05-12 08:09:31.930,2026-05-12 08:09:32.917
4,PSD_FRE_FRE1_PDM14_OBS,FRE,FRE1,14.0,OBS,2026-05-12 08:05:05.830,2026-05-12 08:05:14.847



ZM01


,coluna,tipo_lido,ausentes,ausentes_pct,strings_vazias
0,linha_origem,int64,0,0.00,0
1,tipo_de_nota,object,0,0.00,0
2,nota,int64,0,0.00,0
3,ordem,float64,1,0.01,0
4,perfil_do_catalogo,object,0,0.00,0
5,descricao,object,0,0.00,0
6,local_de_instalacao,object,0,0.00,0
7,centro_de_trabalho_responsavel,object,0,0.00,0
8,inicio_avaria_datetime,datetime64[ns],0,0.00,0
9,fim_avaria_datetime,datetime64[ns],326,3.09,0


...

,coluna,tipo_lido,ausentes,ausentes_pct,strings_vazias
0,linha_origem,int64,0,0.00,0
1,tipo_de_nota,object,0,0.00,0
2,nota,int64,0,0.00,0
3,ordem,float64,5,0.43,0
4,perfil_do_catalogo,object,0,0.00,0
5,descricao,object,0,0.00,0
6,local_de_instalacao,object,0,0.00,0
7,centro_de_trabalho_responsavel,object,0,0.00,0
8,inicio_avaria_datetime,datetime64[ns],0,0.00,0
9,fim_avaria_datetime,datetime64[ns],39,3.35,0


,nota,ordem,local_de_instalacao,inicio_avaria_datetime,fim_avaria_datetime
0,15069252,54081268.0,SP04-6-PSD-BUT,2026-01-25 08:20:00,2026-01-25 09:30:00
1,14647619,53659484.0,SP04-6-PSD-BUT-P1S1,2025-05-30 09:30:00,2025-06-01 04:00:00
2,14682623,53694340.0,SP04-6-PSD-BUT-P2S1-PSSE2,2025-06-19 02:00:00,2025-06-19 04:30:00
3,14864269,53874165.0,SP04-6-PSD-BUT-P2S1-PSSE2,2025-09-30 09:00:00,2025-09-30 12:00:00
4,13390549,52423535.0,SP04-6-PSD-BUT-PLT1,2023-07-21 14:00:00,2023-07-27 13:50:56



ZM03


,coluna,tipo_lido,ausentes,ausentes_pct,strings_vazias
0,Ordem,int64,0,0.00,0
1,Tipo de ordem,object,0,0.00,0
2,Centro de trabalho,object,0,0.00,0
3,Local de instalação,object,0,0.00,0
4,Texto breve,object,0,0.00,0
5,Plano de manutenção,int64,0,0.00,0
6,Status usuário,object,0,0.00,0
7,Status do sistema,object,0,0.00,0
8,Data-base do início,object,0,0.00,0
9,Hr.início programada,object,0,0.00,0


,Ordem,Local de instalação,DT_Inicio,DT_Fim,cancelada
0,54164639,SP04-6-PSD-PTA,2026-03-23 21:30:00,2026-03-24 04:30:00,False
1,54164638,SP04-6-PSD-FAL,2026-03-24 21:30:00,2026-03-25 04:30:00,False
2,54152224,SP04-6-PSD-PIN,2026-03-17 21:30:00,2026-03-18 04:30:00,False
3,54152223,SP04-6-PSD-BUT,2026-03-16 21:30:00,2026-03-17 04:30:00,False
4,54143168,SP04-6-PSD-VSO-PLT1,2026-03-30 21:30:00,2026-04-02 04:30:00,False


## 3. Datas e período dos registros

A conversão ocorre em séries auxiliares; valores inválidos são contabilizados. Mínimo e máximo são **datas de eventos registrados**, não confirmação de cobertura contínua. Meses sem registros também não demonstram ausência de falhas.

Em ZM01/ZM02, `inicio_avaria_datetime` é um campo candidato a início da avaria, não prova de execução da manutenção ou de quando a informação ficou disponível. Em ZM03, a inspeção compara `DT_Inicio`/`DT_Fim` com os campos de programação: equivalência não comprova execução.

In [4]:
DATAS = {
    "Alarmes": ["dt_alarme", "dt_normalizacao"],
    "ZM01": ["inicio_avaria_datetime", "fim_avaria_datetime", "analise_datetime", "encerramento_datetime"],
    "ZM02": ["inicio_avaria_datetime", "fim_avaria_datetime", "encerramento_datetime"],
    "ZM03": ["DT_Inicio", "DT_Fim", "Data-base do início", "Data-base do fim"],
}
datas = {}
resumo_datas = []
for nome, colunas in DATAS.items():
    for coluna in colunas:
        original = bases[nome][coluna]
        convertida = pd.to_datetime(original, errors="coerce", format="mixed")
        datas[nome, coluna] = convertida
        resumo_datas.append({
            "fonte": nome, "coluna": coluna, "menor_data": convertida.min(),
            "maior_data": convertida.max(), "ausentes_origem": original.isna().sum(),
            "nao_convertidas": (original.notna() & convertida.isna()).sum(),
        })
display(pd.DataFrame(resumo_datas))

PARES = {"Alarmes": ("dt_alarme", "dt_normalizacao"),
         "ZM01": ("inicio_avaria_datetime", "fim_avaria_datetime"),
         "ZM02": ("inicio_avaria_datetime", "fim_avaria_datetime"),
         "ZM03": ("DT_Inicio", "DT_Fim")}
for nome, (inicio, fim) in PARES.items():
    a, b = datas[nome, inicio], datas[nome, fim]
    print(f"{nome}: fim anterior ao início = {(b < a).sum()}; fim ausente/inválido = {b.isna().sum()}")

mensais = {}
for nome, coluna in {"Alarmes": "dt_alarme", "ZM01": "inicio_avaria_datetime",
                     "ZM02": "inicio_avaria_datetime", "ZM03": "DT_Inicio"}.items():
    serie = datas[nome, coluna].dropna()
    if not serie.empty:
        contagem = serie.dt.to_period("M").value_counts().sort_index()
        mensais[nome] = contagem.reindex(pd.period_range(contagem.index.min(), contagem.index.max(), freq="M"), fill_value=0)
print("Registros por mês (ausente fora do intervalo de eventos de cada fonte):")
display(pd.DataFrame(mensais).sort_index().astype("Int64"))

for derivada, base in [("DT_Inicio", "Data-base do início"), ("DT_Fim", "Data-base do fim")]:
    a, b = datas["ZM03", derivada], datas["ZM03", base]
    validos = a.notna() & b.notna()
    print(f"ZM03: {derivada} e {base}: {validos.sum()} pares válidos; "
          f"{(a[validos].dt.normalize() == b[validos].dt.normalize()).sum()} no mesmo dia.")
display(bases["ZM03"][["Data-base do início", "Hr.início programada", "DT_Inicio",
                       "Data-base do fim", "Fim-base (hora)", "DT_Fim"]].head())


,fonte,coluna,menor_data,maior_data,ausentes_origem,nao_convertidas
0,Alarmes,dt_alarme,2023-02-27 06:26:56.897,2026-05-12 08:16:52.657,0,0
1,Alarmes,dt_normalizacao,2023-02-27 06:27:11.897,2026-05-12 08:16:53.670,0,0
2,ZM01,inicio_avaria_datetime,2023-01-01 02:39:00.000,2026-04-01 22:43:00.000,0,0
3,ZM01,fim_avaria_datetime,2023-01-01 03:39:00.000,2026-04-01 23:25:57.000,326,0
4,ZM01,analise_datetime,2024-11-14 09:59:15.000,2026-05-06 13:39:20.000,5756,0
5,ZM01,encerramento_datetime,2023-01-01 05:16:11.000,2026-04-30 14:01:20.000,1,0
6,ZM02,inicio_avaria_datetime,2023-01-03 03:55:59.000,2026-03-28 14:15:00.000,0,0
7,ZM02,fim_avaria_datetime,2023-01-07 11:30:00.000,2026-05-14 02:00:00.000,39,0
8,ZM02,encerramento_datetime,2023-01-07 17:07:19.000,2026-05-14 04:33:52.000,7,0
9,ZM03,DT_Inicio,2022-12-26 05:30:00.000,2026-03-30 21:30:00.000,0,0


Alarmes: fim anterior ao início = 0; fim ausente/inválido = 0
ZM01: fim anterior ao início = 0; fim ausente/inválido = 326
ZM02: fim anterior ao início = 0; fim ausente/inválido = 39
ZM03: fim anterior ao início = 0; fim ausente/inválido = 0


Registros por mês (ausente fora do intervalo de eventos de cada fonte):


,Alarmes,ZM01,ZM02,ZM03
2022-12,<NA>,<NA>,<NA>,1
2023-01,<NA>,147,51,22
2023-02,50,171,29,16
2023-03,43,178,40,16
2023-04,0,282,31,19
2023-05,0,293,27,18
2023-06,0,292,35,15
2023-07,0,249,18,19
2023-08,0,310,7,12
2023-09,0,199,37,17


ZM03: DT_Inicio e Data-base do início: 569 pares válidos; 569 no mesmo dia.
ZM03: DT_Fim e Data-base do fim: 569 pares válidos; 567 no mesmo dia.


,Data-base do início,Hr.início programada,DT_Inicio,Data-base do fim,Fim-base (hora),DT_Fim
0,2026-03-23,21:30:00,2026-03-23 21:30:00,2026-03-24,04:30:00,2026-03-24 04:30:00
1,2026-03-24,21:30:00,2026-03-24 21:30:00,2026-03-25,04:30:00,2026-03-25 04:30:00
2,2026-03-17,21:30:00,2026-03-17 21:30:00,2026-03-18,04:30:00,2026-03-18 04:30:00
3,2026-03-16,21:30:00,2026-03-16 21:30:00,2026-03-17,04:30:00,2026-03-17 04:30:00
4,2026-03-30,21:30:00,2026-03-30 21:30:00,2026-04-02,04:30:00,2026-04-02 04:30:00


## 4. Identificação dos ativos — extração exploratória

As regras abaixo são deliberadamente conservadoras e criam tabelas auxiliares. Formatos desconhecidos ficam para revisão, sem inferência por texto livre.

- SAP: `SP04-6-PSD-BUT-PLT1-PP05PDM` → BUT / 1 / 05 / PDM.
- Alarme: `PSD_BUT_BUT1_PDM05_OBS` → BUT / 1 / 05 / PDM.
- Registros apenas de estação ou plataforma permanecem nesse nível.
- PDM, PFX e outros tipos não são unificados automaticamente.

A chave completa é provisória e inclui o tipo. Esta etapa mantém os eventos separados e preserva o nível de identificação das manutenções coletivas.

In [5]:
def extrair_ativos(serie, alarme=False):
    texto = serie.astype("string").str.strip().str.upper()
    if alarme:
        padrao = r"^PSD_(?P<estacao>[A-Z]{3})_(?P<estacao_plataforma>[A-Z]{3})(?P<plataforma>\d+)_(?P<tipo_ativo>[A-Z]+)(?P<porta>\d+)_(?P<evento>[A-Z0-9]+)$"
        out = texto.str.extract(padrao)
        # Aceita PSD_P_BUT1_PDM07_FC e PSD_P_MBI1_PDM10FC.
        alternativo = texto.str.extract(
            r"^PSD_P_(?P<estacao>[A-Z]{3})(?P<plataforma>\d+)_(?P<tipo_ativo>[A-Z]+)(?P<porta>\d+)_?(?P<evento>[A-Z][A-Z0-9]*)$")
        alternativo["estacao_plataforma"] = alternativo["estacao"]
        out = out.combine_first(alternativo)
        # Tags com estações divergentes não recebem chave válida.
        divergente = out.estacao.ne(out.estacao_plataforma).fillna(False)
        out.loc[divergente, ["estacao", "plataforma", "porta", "tipo_ativo"]] = pd.NA
    else:
        out = texto.str.extract(
            r"^SP\d+-\d+-PSD-(?P<estacao>[A-Z]{3})(?:-|$)")
        out["plataforma"] = texto.str.extract(r"-PLT(\d+)(?:-|$)", expand=False)
        porta = texto.str.extract(r"-PP(?P<porta>\d+)(?P<tipo_ativo>[A-Z]+)$")
        out = out.join(porta)
    out = out[["estacao", "plataforma", "porta", "tipo_ativo"]].copy()
    for coluna in ["plataforma", "porta"]:
        out[coluna] = pd.to_numeric(out[coluna], errors="coerce").astype("Int64").astype("string")
    completa = out.notna().all(axis=1)
    # Nível identificável não significa que um subativo é uma porta.
    out["nivel_identificavel"] = "não reconhecido"
    out.loc[out.estacao.notna(), "nivel_identificavel"] = "estação"
    out.loc[out.estacao.notna() & out.plataforma.notna(), "nivel_identificavel"] = "plataforma"
    out.loc[completa, "nivel_identificavel"] = "porta e tipo"
    out["chave_candidata"] = out.estacao + "|" + out.plataforma + "|" + out.porta + "|" + out.tipo_ativo
    out["identificacao_original"] = serie
    return out

COLUNA_ATIVO = {"Alarmes": "TAG", "ZM01": "local_de_instalacao",
                "ZM02": "local_de_instalacao", "ZM03": "Local de instalação"}
ativos = {nome: extrair_ativos(df[COLUNA_ATIVO[nome]], alarme=nome == "Alarmes")
          for nome, df in bases.items()}
for nome, df in ativos.items():
    print(f"\n{nome} — quantidade de registros por nível identificável")
    display(df.nivel_identificavel.value_counts().rename_axis("nível").to_frame("registros"))
    print("Exemplos de identificação (incluindo formatos que exigem revisão):")
    display(df.drop_duplicates("identificacao_original").groupby("nivel_identificavel", sort=False).head(4))
    print("Tipos identificados:")
    display(df.tipo_ativo.value_counts(dropna=False).to_frame("registros"))



Alarmes — quantidade de registros por nível identificável


,registros
nível,
porta e tipo,742336
não reconhecido,170


Exemplos de identificação (incluindo formatos que exigem revisão):


,estacao,plataforma,porta,tipo_ativo,nivel_identificavel,chave_candidata,identificacao_original
0,FAL,2,9,PDM,porta e tipo,FAL|2|9|PDM,PSD_FAL_FAL2_PDM09_OBS
1,LUZ,1,8,PDM,porta e tipo,LUZ|1|8|PDM,PSD_LUZ_LUZ1_PDM08_OBS
2,FAL,1,15,PDM,porta e tipo,FAL|1|15|PDM,PSD_FAL_FAL1_PDM15_OBS
3,LUZ,1,19,PDM,porta e tipo,LUZ|1|19|PDM,PSD_LUZ_LUZ1_PDM19_OBS
104431,<NA>,<NA>,<NA>,<NA>,não reconhecido,<NA>,PSD_P_LUZ1_CBTC_BIM_FAULT_Value


Tipos identificados:


...

,registros
nível,
plataforma,313
estação,173
não reconhecido,83


Exemplos de identificação (incluindo formatos que exigem revisão):


,estacao,plataforma,porta,tipo_ativo,nivel_identificavel,chave_candidata,identificacao_original
0,PTA,<NA>,<NA>,<NA>,estação,<NA>,SP04-6-PSD-PTA
1,FAL,<NA>,<NA>,<NA>,estação,<NA>,SP04-6-PSD-FAL
2,PIN,<NA>,<NA>,<NA>,estação,<NA>,SP04-6-PSD-PIN
3,BUT,<NA>,<NA>,<NA>,estação,<NA>,SP04-6-PSD-BUT
4,VSO,1,<NA>,<NA>,plataforma,<NA>,SP04-6-PSD-VSO-PLT1
9,MBI,2,<NA>,<NA>,plataforma,<NA>,SP04-6-PSD-MBI-PLT2
10,MBI,1,<NA>,<NA>,plataforma,<NA>,SP04-6-PSD-MBI-PLT1
11,PTA,1,<NA>,<NA>,plataforma,<NA>,SP04-6-PSD-PTA-PLT1
260,<NA>,<NA>,<NA>,<NA>,não reconhecido,<NA>,SP04-6-PSD


Tipos identificados:


,registros
tipo_ativo,
<NA>,569


## 5. Compatibilidade das chaves

A interseção abaixo mede apenas a existência da mesma chave nas fontes, **sem associação temporal ou causal**. Uma chave ausente pode decorrer de cobertura, nomenclatura ou falta de eventos. A TAG é comparada aos campos já extraídos dos alarmes para conferir consistência.

In [6]:
a = bases["Alarmes"]
extraido = ativos["Alarmes"]
comparacoes = {
    "estacao": a["estacao_extraida"].astype("string").str.upper().str.strip(),
    "plataforma": a["plataforma_extraida"].astype("string").str.extract(r"(\d+)$", expand=False),
    "porta": pd.to_numeric(a["porta_extraida"], errors="coerce").astype("Int64").astype("string"),
}
for campo, existente in comparacoes.items():
    if campo == "plataforma":
        existente = pd.to_numeric(existente, errors="coerce").astype("Int64").astype("string")
    validos = extraido[campo].notna() & existente.notna()
    print(f"Alarmes — {campo}: {validos.sum()} comparáveis, "
          f"{(extraido.loc[validos, campo] != existente[validos]).sum()} divergências.")

chaves_alarmes = set(extraido.chave_candidata.dropna())
resumo_chaves = []
for nome, df in ativos.items():
    chaves = set(df.chave_candidata.dropna())
    resumo_chaves.append({"fonte": nome, "chaves_completas_distintas": len(chaves),
                          "presentes_nos_alarmes": len(chaves & chaves_alarmes),
                          "ausentes_nos_alarmes": len(chaves - chaves_alarmes)})
display(pd.DataFrame(resumo_chaves))

todos = pd.concat([df[["estacao", "plataforma", "porta", "tipo_ativo"]].dropna().drop_duplicates()
                   for df in ativos.values()]).drop_duplicates()
colisoes = todos.groupby(["estacao", "plataforma", "porta"]).tipo_ativo.nunique()
print("Combinações estação/plataforma/número com mais de um tipo:", colisoes.gt(1).sum())
display(colisoes[colisoes.gt(1)].rename("quantidade_tipos").head(10).to_frame())


Alarmes — estacao: 742336 comparáveis, 0 divergências.


Alarmes — plataforma: 742336 comparáveis, 0 divergências.


Alarmes — porta: 742336 comparáveis, 0 divergências.


,fonte,chaves_completas_distintas,presentes_nos_alarmes,ausentes_nos_alarmes
0,Alarmes,978,978,0
1,ZM01,589,522,67
2,ZM02,518,345,173
3,ZM03,0,0,0


Combinações estação/plataforma/número com mais de um tipo: 475


quantidade_tipos
estacao plataforma porta                  
BUT     1          1                     2
                   10                    2
                   11                    2
                   12                    2
                   13                    2
                   15                    3
                   16                    2
                   17                    2
                   18                    3
                   19                    2

## 6. Duplicidades e flags

Repetição de ordem ou nota não implica automaticamente duplicata: pode representar atualização ou vários registros legítimos. A contagem inclui todas as linhas envolvidas (`keep=False`) e também o excedente de duplicatas exatas. Todos os registros são preservados.

In [7]:
resumo_duplicatas = []
for nome, df in bases.items():
    resumo_duplicatas.append({"fonte": nome,
                              "linhas_em_duplicatas_exatas": df.duplicated(keep=False).sum(),
                              "excedente_duplicatas_exatas": df.duplicated().sum()})
display(pd.DataFrame(resumo_duplicatas))
for nome, colunas in {"ZM01": ["ordem", "nota"], "ZM02": ["ordem", "nota"], "ZM03": ["Ordem"]}.items():
    for coluna in colunas:
        serie = bases[nome][coluna].dropna()
        print(f"{nome}/{coluna}: {serie.duplicated(keep=False).sum()} linhas com identificador repetido; "
              f"{serie[serie.duplicated(keep=False)].nunique()} identificadores repetidos.")

chave_evento = ["TAG", "dt_alarme"]
validos = a[chave_evento].notna().all(axis=1)
print("Alarmes: linhas com TAG + dt_alarme repetidos:",
      a.loc[validos].duplicated(chave_evento, keep=False).sum())
for nome, df in bases.items():
    flags = [c for c in df.columns if c.endswith("_flag") or c == "cancelada"]
    for coluna in flags:
        print(f"{nome} — {coluna}")
        display(df[coluna].value_counts(dropna=False).rename_axis("valor").to_frame("registros"))

# Apenas diagnóstico: não reinserir registros excluídos automaticamente.
excluidos = auditorias.get(("ZM01", "Outliers_removidos"))
if excluidos is not None:
    print(f"ZM01: {len(excluidos)} registros preservados em Outliers_removidos.")
    print("Antes do alvo, revisar se são falhas válidas: excluir por duração pode retirar falhas relevantes.")


,fonte,linhas_em_duplicatas_exatas,excedente_duplicatas_exatas
0,Alarmes,0,0
1,ZM01,0,0
2,ZM02,0,0
3,ZM03,0,0


ZM01/ordem: 136 linhas com identificador repetido; 66 identificadores repetidos.
ZM01/nota: 213 linhas com identificador repetido; 104 identificadores repetidos.
ZM02/ordem: 10 linhas com identificador repetido; 5 identificadores repetidos.
ZM02/nota: 10 linhas com identificador repetido; 5 identificadores repetidos.
ZM03/Ordem: 0 linhas com identificador repetido; 0 identificadores repetidos.


Alarmes: linhas com TAG + dt_alarme repetidos: 449829
ZM01 — indisponivel_flag


,registros
valor,
True,6972
False,3571


ZM01 — falha_flag


...

ZM02 — duracao_zero_flag


,registros
valor,
False,1158
True,5


ZM03 — cancelada


,registros
valor,
False,559
True,10


ZM01: 103 registros preservados em Outliers_removidos.
Antes do alvo, revisar se são falhas válidas: excluir por duração pode retirar falhas relevantes.


## 7. Decisões necessárias antes da tabela mestra

1. **Ativos:** validar os formatos não reconhecidos, os tipos elegíveis (por exemplo, PDM) e se tipo deve integrar a chave. Não misturar portas homônimas de plataformas diferentes.
2. **Manutenções coletivas:** definir como históricos de estação/plataforma entram como contexto, sem convertê-los em intervenções individuais comprovadas.
3. **Semântica temporal:** confirmar a data efetiva de cada manutenção e quando o registro ficou disponível. Em especial, não tratar datas programadas da ZM03 como prova de execução. Status atuais e textos atualizados depois do evento não reconstituem automaticamente o estado histórico.
4. **Falha:** definir critérios para episódios ZM01 e eventual evidência de defeitos em ZM02, revisar os registros excluídos na limpeza ZM01 e depois estabelecer a regra de associação com alarmes. ZM03 não é alvo positivo por si só.
5. **Cobertura:** confirmar os intervalos completos de cada fonte e do cadastro de ativos; não deduzir início de operação da primeira ocorrência nem cobertura pelo último evento.
6. **Duplicidades:** revisar ordens/notas repetidas antes de contar intervenções ou episódios.

**Resultado da inspeção:** inventário, perfis de qualidade, períodos de eventos e chaves candidatas em memória. As seções seguintes apresentam a preparação das tabelas e o calendário inicial. A definição do alvo permanece pendente.

## 8. Preparar as quatro tabelas — passo a passo

Esta seção prepara os dados para a tabela mestra com operações simples do pandas, sem novas funções.

A preparação reutiliza as identificações verificadas na inspeção (`ativos`). A chave candidata tem o formato `BUT|1|5|PDM`: estação BUT, plataforma 1, número 5, tipo PDM. O tipo permanece na chave para distinguir, por exemplo, PDM e MDU enquanto a relação entre eles não for confirmada.

- `.copy()`: cria uma cópia para trabalhar sem modificar a tabela lida.
- `.rename()`: troca nomes de colunas.
- `.join()`: acrescenta as identificações da inspeção à mesma linha de origem. **Não é um cruzamento entre alarmes e manutenções.**

As linhas sem chave completa continuam nas tabelas. Esta etapa preserva os registros e as datas ausentes; a definição do alvo permanece pendente.

In [8]:
# Colunas de identificação verificadas na inspeção.
colunas_identificacao = [
    "estacao", "plataforma", "porta", "tipo_ativo",
    "nivel_identificavel", "chave_candidata",
]


### 8.1 Alarmes

A seleção inclui a TAG, o tipo do evento e seus horários. `data_alarme` representa o dia da ocorrência, usado mais adiante para contar alarmes.

A normalização pode acontecer depois do dia do alarme. Sua duração final não pode ser usada numa previsão feita antes de o alarme normalizar.

In [9]:
alarmes = bases["Alarmes"][[
    "TAG", "tipo_evento_extraido", "dt_alarme", "dt_normalizacao",
]].copy()

alarmes = alarmes.rename(columns={
    "TAG": "tag",
    "tipo_evento_extraido": "tipo_evento",
    "dt_alarme": "inicio_alarme",
    "dt_normalizacao": "fim_alarme",
})

alarmes = alarmes.join(ativos["Alarmes"][colunas_identificacao])

alarmes["inicio_alarme"] = pd.to_datetime(alarmes["inicio_alarme"], errors="coerce")
alarmes["fim_alarme"] = pd.to_datetime(alarmes["fim_alarme"], errors="coerce")
alarmes["data_alarme"] = alarmes["inicio_alarme"].dt.normalize()

display(alarmes.head())


,tag,tipo_evento,inicio_alarme,fim_alarme,estacao,plataforma,porta,tipo_ativo,nivel_identificavel,chave_candidata,data_alarme
0,PSD_FAL_FAL2_PDM09_OBS,OBS,2026-05-12 08:16:52.657,2026-05-12 08:16:53.670,FAL,2,9,PDM,porta e tipo,FAL|2|9|PDM,2026-05-12
1,PSD_LUZ_LUZ1_PDM08_OBS,OBS,2026-05-12 08:15:09.913,2026-05-12 08:15:10.917,LUZ,1,8,PDM,porta e tipo,LUZ|1|8|PDM,2026-05-12
2,PSD_FAL_FAL1_PDM15_OBS,OBS,2026-05-12 08:14:06.657,2026-05-12 08:14:07.670,FAL,1,15,PDM,porta e tipo,FAL|1|15|PDM,2026-05-12
3,PSD_LUZ_LUZ1_PDM19_OBS,OBS,2026-05-12 08:09:31.930,2026-05-12 08:09:32.917,LUZ,1,19,PDM,porta e tipo,LUZ|1|19|PDM,2026-05-12
4,PSD_FRE_FRE1_PDM14_OBS,OBS,2026-05-12 08:05:05.830,2026-05-12 08:05:14.847,FRE,1,14,PDM,porta e tipo,FRE|1|14|PDM,2026-05-12


### 8.2 ZM01 — corretiva emergencial

A tabela mantém nota, ordem, local e início/fim da avaria. `data_avaria` é uma candidata à data de falha, mas os critérios do alvo e os 103 registros separados na limpeza ainda precisam de revisão. `falha_flag` é preservada para essa revisão; não será usada automaticamente como informação histórica disponível no dia.

In [10]:
zm01 = bases["ZM01"][[
    "nota", "ordem", "local_de_instalacao",
    "inicio_avaria_datetime", "fim_avaria_datetime", "falha_flag",
]].copy()

zm01 = zm01.rename(columns={
    "local_de_instalacao": "local_original",
    "inicio_avaria_datetime": "inicio_avaria",
    "fim_avaria_datetime": "fim_avaria",
})

zm01 = zm01.join(ativos["ZM01"][colunas_identificacao])

zm01["inicio_avaria"] = pd.to_datetime(zm01["inicio_avaria"], errors="coerce")
zm01["fim_avaria"] = pd.to_datetime(zm01["fim_avaria"], errors="coerce")
zm01["data_avaria"] = zm01["inicio_avaria"].dt.normalize()

display(zm01.head())


,nota,ordem,local_original,inicio_avaria,fim_avaria,falha_flag,...,plataforma,porta,tipo_ativo,nivel_identificavel,chave_candidata,data_avaria
0,14943931,53954586.0,SP04-6-PSD-BUT,2025-11-14 06:35:28,NaT,True,...,<NA>,<NA>,<NA>,estação,<NA>,2025-11-14
1,15133619,54145839.0,SP04-6-PSD-BUT,2026-02-27 07:33:00,2026-02-27 12:00:00,False,...,<NA>,<NA>,<NA>,estação,<NA>,2026-02-27
2,13894133,52913343.0,SP04-6-PSD-BUT-P1S1,2024-04-18 06:00:00,2024-04-18 07:30:00,False,...,<NA>,<NA>,<NA>,estação,<NA>,2024-04-18
3,14713349,53724958.0,SP04-6-PSD-BUT-P1S1,2025-07-07 18:38:00,2025-07-08 03:00:00,False,...,<NA>,<NA>,<NA>,estação,<NA>,2025-07-07
4,14643768,53655728.0,SP04-6-PSD-BUT-P1S1-PSSE1,2025-05-28 09:10:00,2025-05-28 10:00:00,False,...,<NA>,<NA>,<NA>,estação,<NA>,2025-05-28


### 8.3 ZM02 — corretiva programada

A tabela preserva o nome `inicio_avaria`, conforme a identificação do campo na fonte. Sua interpretação como data de execução da manutenção exige confirmação. Um registro ZM02 não é classificado automaticamente como falha.

In [11]:
zm02 = bases["ZM02"][[
    "nota", "ordem", "local_de_instalacao",
    "inicio_avaria_datetime", "fim_avaria_datetime",
]].copy()

zm02 = zm02.rename(columns={
    "local_de_instalacao": "local_original",
    "inicio_avaria_datetime": "inicio_avaria",
    "fim_avaria_datetime": "fim_avaria",
})

zm02 = zm02.join(ativos["ZM02"][colunas_identificacao])

zm02["inicio_avaria"] = pd.to_datetime(zm02["inicio_avaria"], errors="coerce")
zm02["fim_avaria"] = pd.to_datetime(zm02["fim_avaria"], errors="coerce")
zm02["data_avaria"] = zm02["inicio_avaria"].dt.normalize()

display(zm02.head())


,nota,ordem,local_original,inicio_avaria,fim_avaria,estacao,plataforma,porta,tipo_ativo,nivel_identificavel,chave_candidata,data_avaria
0,15069252,54081268.0,SP04-6-PSD-BUT,2026-01-25 08:20:00,2026-01-25 09:30:00,BUT,<NA>,<NA>,<NA>,estação,<NA>,2026-01-25
1,14647619,53659484.0,SP04-6-PSD-BUT-P1S1,2025-05-30 09:30:00,2025-06-01 04:00:00,BUT,<NA>,<NA>,<NA>,estação,<NA>,2025-05-30
2,14682623,53694340.0,SP04-6-PSD-BUT-P2S1-PSSE2,2025-06-19 02:00:00,2025-06-19 04:30:00,BUT,<NA>,<NA>,<NA>,estação,<NA>,2025-06-19
3,14864269,53874165.0,SP04-6-PSD-BUT-P2S1-PSSE2,2025-09-30 09:00:00,2025-09-30 12:00:00,BUT,<NA>,<NA>,<NA>,estação,<NA>,2025-09-30
4,13390549,52423535.0,SP04-6-PSD-BUT-PLT1,2023-07-21 14:00:00,2023-07-27 13:50:56,BUT,1,<NA>,<NA>,plataforma,<NA>,2023-07-21


### 8.4 ZM03 — preventiva

Esta preparação trata `DT_Inicio` e `DT_Fim` como referências de programação, sujeitas à confirmação. O status `cancelada` é mantido para revisão: uma ordem não cancelada não comprova execução.

A inspeção não encontrou identificação individual completa de porta nessa base. Por isso, a preparação mantém as preventivas no nível de identificação disponível, sem atribuí-las a cada porta. Os registros sem estação reconhecida também permanecem disponíveis para revisão.

In [12]:
zm03 = bases["ZM03"][[
    "Ordem", "Local de instalação", "DT_Inicio", "DT_Fim", "cancelada",
]].copy()

zm03 = zm03.rename(columns={
    "Ordem": "ordem",
    "Local de instalação": "local_original",
    "DT_Inicio": "inicio_programado",
    "DT_Fim": "fim_programado",
})

zm03 = zm03.join(ativos["ZM03"][colunas_identificacao])

zm03["inicio_programado"] = pd.to_datetime(zm03["inicio_programado"], errors="coerce")
zm03["fim_programado"] = pd.to_datetime(zm03["fim_programado"], errors="coerce")
zm03["data_programada"] = zm03["inicio_programado"].dt.normalize()

display(zm03.head())


,ordem,local_original,inicio_programado,fim_programado,cancelada,estacao,plataforma,porta,tipo_ativo,nivel_identificavel,chave_candidata,data_programada
0,54164639,SP04-6-PSD-PTA,2026-03-23 21:30:00,2026-03-24 04:30:00,False,PTA,<NA>,<NA>,<NA>,estação,<NA>,2026-03-23
1,54164638,SP04-6-PSD-FAL,2026-03-24 21:30:00,2026-03-25 04:30:00,False,FAL,<NA>,<NA>,<NA>,estação,<NA>,2026-03-24
2,54152224,SP04-6-PSD-PIN,2026-03-17 21:30:00,2026-03-18 04:30:00,False,PIN,<NA>,<NA>,<NA>,estação,<NA>,2026-03-17
3,54152223,SP04-6-PSD-BUT,2026-03-16 21:30:00,2026-03-17 04:30:00,False,BUT,<NA>,<NA>,<NA>,estação,<NA>,2026-03-16
4,54143168,SP04-6-PSD-VSO-PLT1,2026-03-30 21:30:00,2026-04-02 04:30:00,False,VSO,1,<NA>,<NA>,plataforma,<NA>,2026-03-30


## 9. Conferência das identificações

Este resumo conta **registros**, não portas distintas ou falhas. A identificação completa é candidata: ainda depende da escolha dos tipos de ativo que entrarão no modelo.

O pequeno `for` repete a mesma conferência nas quatro tabelas. Os `assert` verificam se todas as linhas e suas posições foram preservadas.

In [13]:
tabelas_preparadas = {"Alarmes": alarmes, "ZM01": zm01, "ZM02": zm02, "ZM03": zm03}
resumo = []

for nome, tabela in tabelas_preparadas.items():
    assert len(tabela) == len(bases[nome]), f"A quantidade de linhas mudou em {nome}."
    assert tabela.index.equals(bases[nome].index), f"A ordem das linhas mudou em {nome}."

    resumo.append({
        "fonte": nome,
        "total_registros": len(tabela),
        "com_chave_completa": tabela["chave_candidata"].notna().sum(),
        "sem_chave_completa": tabela["chave_candidata"].isna().sum(),
    })

display(pd.DataFrame(resumo))


,fonte,total_registros,com_chave_completa,sem_chave_completa
0,Alarmes,742506,742336,170
1,ZM01,10543,9357,1186
2,ZM02,1163,1010,153
3,ZM03,569,0,569


## 10. Primeira contagem diária: alarmes

Esta etapa conta os **registros de alarmes por chave candidata e dia**, com `groupby()` e `size()`.

Por enquanto, a contagem mantém registros repetidos da fonte. Ela é exploratória: não representa episódios únicos e ainda não é a variável final `qtd_alarmes_1d`. As duplicidades e os tipos de ativo precisam de revisão antes da modelagem.

Linhas sem chave ou data ficam em `alarmes_para_revisar`. Não serão atribuídas a uma porta por aproximação. A tabela diária contém somente dias com registros; dias ausentes permanecem sem preenchimento enquanto a cobertura não for confirmada.

In [14]:
# A contagem por ativo exige identificação completa e data.
pode_agrupar = alarmes["chave_candidata"].notna() & alarmes["data_alarme"].notna()

alarmes_identificados = alarmes.loc[pode_agrupar].copy()
alarmes_para_revisar = alarmes.loc[~pode_agrupar].copy()

alarmes_diarios = (
    alarmes_identificados
    .groupby(["chave_candidata", "data_alarme"])
    .size()
    .reset_index(name="qtd_registros_alarmes")
)

# A conferência verifica se houve perda ou contagem duplicada no agrupamento.
assert len(alarmes_identificados) + len(alarmes_para_revisar) == len(alarmes)
assert alarmes_diarios["qtd_registros_alarmes"].sum() == len(alarmes_identificados)
assert not alarmes_diarios.duplicated(["chave_candidata", "data_alarme"]).any()

print("Registros usados na contagem:", len(alarmes_identificados))
print("Registros separados para revisão:", len(alarmes_para_revisar))
display(alarmes_diarios.head(10))


Registros usados na contagem: 742336
Registros separados para revisão: 170


,chave_candidata,data_alarme,qtd_registros_alarmes
0,BUT|1|10|MDU,2023-11-12,1
1,BUT|1|10|MDU,2023-11-13,1
2,BUT|1|10|MDU,2023-11-27,2
3,BUT|1|10|MDU,2023-12-06,11
4,BUT|1|10|MDU,2023-12-31,5
5,BUT|1|10|MDU,2024-01-01,3
6,BUT|1|10|MDU,2024-01-03,4
7,BUT|1|10|MDU,2024-01-08,19
8,BUT|1|10|MDU,2024-01-14,31
9,BUT|1|10|MDU,2024-01-20,5


### Resultado desta etapa

A preparação produz quatro tabelas em memória (`alarmes`, `zm01`, `zm02`, `zm03`) e uma primeira contagem diária (`alarmes_diarios`). Os arquivos de origem continuam intactos e nenhum CSV foi exportado.

**Continuação:** as próximas seções adotam um tipo de ativo e um recorte exploratório para construir o calendário diário e juntar as contagens de alarmes. A confirmação de cobertura, o alvo `falha_7d`, as janelas móveis e o treinamento permanecem para etapas posteriores.

## 11. Tipo de ativo e período inicial

O primeiro calendário considera **PDM**, sem agregar MDU ou outros tipos. Essa escolha inicial pode ser alterada na célula de configuração.

A leitura das TAGs também reconhece o formato de MBI sem separador entre número e evento, como `PSD_P_MBI1_PDM10FC`. Assim, esses registros passam pela mesma identificação dos demais alarmes.

O período exploratório vai de **17/10/2023 a 28/03/2026**. O início acompanha a retomada de registros em várias estações após meses vazios em 2023. O fim evita avançar além da última data de início de avaria registrada em ZM02, a mais antiga entre os limites finais dos quatro campos de referência. Essas datas **não comprovam cobertura completa**: servem apenas para construir e examinar a estrutura da tabela.

O resumo por estação ajuda a visualizar diferenças entre os períodos registrados. Primeiro e último alarme não representam início e fim de operação.

In [15]:
# Configuração simples do recorte exploratório.
TIPO_ATIVO = "PDM"
DATA_INICIO = pd.Timestamp("2023-10-17")
DATA_FIM = pd.Timestamp("2026-03-28")

assert DATA_INICIO <= DATA_FIM, "O início precisa ser anterior ou igual ao fim."

alarmes_pdm = alarmes.loc[alarmes["tipo_ativo"].eq(TIPO_ATIVO)].copy()

periodo_por_estacao = (
    alarmes_pdm.groupby("estacao")["data_alarme"]
    .agg(["min", "max", "count"])
    .rename(columns={"min": "primeiro_registro", "max": "ultimo_registro",
                     "count": "qtd_registros"})
)

display(periodo_por_estacao)


,primeiro_registro,ultimo_registro,qtd_registros
estacao,,,
BUT,2023-10-17,2026-05-12,84434
FAL,2023-10-17,2026-05-12,25011
FRA,2023-10-18,2026-04-14,145188
FRE,2023-02-27,2026-05-12,2111
HIG,2023-10-17,2026-05-12,85184
LUZ,2023-10-17,2026-05-12,39148
MBI,2024-06-05,2025-12-31,72913
PIN,2023-10-17,2025-12-31,114359
PTA,2023-10-24,2025-01-26,29048


## 12. Lista das portas presentes no recorte

A lista reúne identificações completas de PDM encontradas nos alarmes, na ZM01 ou na ZM02 durante o período escolhido. A inclusão das corretivas evita limitar a lista apenas às portas com alarmes.

- `.between()` seleciona datas entre os dois limites, incluindo ambos.
- `pd.concat()` reúne as listas.
- `.drop_duplicates()` mantém uma linha por identificação.

Esta é uma **lista exploratória de ativos registrados**, não um cadastro de portas em operação. O calendário pode conter datas anteriores à primeira identificação de uma porta. A definição de amostras de treino exigirá validar o cadastro e a cobertura, sem usar uma identificação futura como prova de operação no passado. A ZM03 permanece fora desta lista por não identificar portas individualmente.

In [16]:
colunas_porta = ["chave_candidata", "estacao", "plataforma", "porta", "tipo_ativo"]

portas_alarmes = alarmes_pdm.loc[
    alarmes_pdm["data_alarme"].between(DATA_INICIO, DATA_FIM),
    colunas_porta,
]

portas_zm01 = zm01.loc[
    zm01["tipo_ativo"].eq(TIPO_ATIVO)
    & zm01["data_avaria"].between(DATA_INICIO, DATA_FIM),
    colunas_porta,
]

portas_zm02 = zm02.loc[
    zm02["tipo_ativo"].eq(TIPO_ATIVO)
    & zm02["data_avaria"].between(DATA_INICIO, DATA_FIM),
    colunas_porta,
]

portas = pd.concat([portas_alarmes, portas_zm01, portas_zm02], ignore_index=True)
portas = portas.dropna(subset=colunas_porta)
portas = portas.drop_duplicates().sort_values("chave_candidata").reset_index(drop=True)

assert not portas.empty, "Nenhuma porta foi encontrada no recorte."
assert not portas["chave_candidata"].duplicated().any(), "Há identificações conflitantes."

print("Quantidade de portas na lista:", len(portas))
display(portas.head(10))


Quantidade de portas na lista: 528


,chave_candidata,estacao,plataforma,porta,tipo_ativo
0,BUT|1|10|PDM,BUT,1,10,PDM
1,BUT|1|11|PDM,BUT,1,11,PDM
2,BUT|1|12|PDM,BUT,1,12,PDM
3,BUT|1|13|PDM,BUT,1,13,PDM
4,BUT|1|14|PDM,BUT,1,14,PDM
5,BUT|1|15|PDM,BUT,1,15,PDM
6,BUT|1|16|PDM,BUT,1,16,PDM
7,BUT|1|17|PDM,BUT,1,17,PDM
8,BUT|1|18|PDM,BUT,1,18,PDM
9,BUT|1|19|PDM,BUT,1,19,PDM


## 13. Calendário: uma linha por porta e dia

`pd.date_range()` cria todos os dias do período. O `merge(how="cross")` combina cada porta com cada dia.

Por exemplo, uma lista com 2 portas e um calendário de 3 dias produz 6 linhas. A tabela ainda é uma estrutura exploratória; a existência de uma linha não significa que houve coleta de dados naquele dia.

In [17]:
calendario = pd.DataFrame({
    "data": pd.date_range(DATA_INICIO, DATA_FIM, freq="D")
})

grade_diaria = portas.merge(calendario, how="cross")

assert len(grade_diaria) == len(portas) * len(calendario)
assert not grade_diaria.duplicated(["chave_candidata", "data"]).any()

print("Portas:", len(portas))
print("Dias:", len(calendario))
print("Linhas do calendário:", len(grade_diaria))
display(grade_diaria.head())


Portas: 528
Dias: 894
Linhas do calendário: 472032


,chave_candidata,estacao,plataforma,porta,tipo_ativo,data
0,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-17
1,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-18
2,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-19
3,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-20
4,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-21


## 14. Inclusão da contagem diária de alarmes

O `merge(how="left")` mantém todas as linhas do calendário e procura a contagem correspondente à mesma chave e data. `validate="one_to_one"` verifica se cada combinação aparece no máximo uma vez em cada tabela.

A coluna `qtd_registros_alarmes` mantém valores ausentes (`<NA>`) quando não há correspondência. O preenchimento com zero depende de confirmar que a coleta estava completa. A coluna `tem_registro_alarme` indica apenas a presença de registros; `False` não comprova ausência de alarmes.

As contagens ainda incluem possíveis repetições da fonte. Esta versão não está pronta para treinamento.

In [18]:
contagens_recorte = alarmes_diarios.loc[
    alarmes_diarios["chave_candidata"].isin(portas["chave_candidata"])
    & alarmes_diarios["data_alarme"].between(DATA_INICIO, DATA_FIM)
].copy()

contagens_recorte = contagens_recorte.rename(columns={"data_alarme": "data"})

tabela_mestra_inicial = grade_diaria.merge(
    contagens_recorte,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)

# Int64, com I maiúsculo, permite números inteiros e valores ausentes.
tabela_mestra_inicial["qtd_registros_alarmes"] = (
    tabela_mestra_inicial["qtd_registros_alarmes"].astype("Int64")
)
tabela_mestra_inicial["tem_registro_alarme"] = (
    tabela_mestra_inicial["qtd_registros_alarmes"].notna()
)

display(tabela_mestra_inicial.head(10))


,chave_candidata,estacao,plataforma,porta,tipo_ativo,data,qtd_registros_alarmes,tem_registro_alarme
0,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-17,<NA>,False
1,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-18,<NA>,False
2,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-19,<NA>,False
3,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-20,<NA>,False
4,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-21,<NA>,False
5,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-22,<NA>,False
6,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-23,<NA>,False
7,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-24,<NA>,False
8,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-25,<NA>,False
9,BUT|1|10|PDM,BUT,1,10,PDM,2023-10-26,<NA>,False


## 15. Conferência da tabela inicial

As verificações conferem o tamanho do calendário, a unicidade de porta/dia e a conservação das contagens. O exemplo exibe dias de uma porta com registros no recorte, incluindo dias sem correspondência quando existirem.

In [19]:
assert len(tabela_mestra_inicial) == len(grade_diaria)
assert not tabela_mestra_inicial.duplicated(["chave_candidata", "data"]).any()
assert (
    tabela_mestra_inicial["qtd_registros_alarmes"].sum()
    == contagens_recorte["qtd_registros_alarmes"].sum()
)

print("Linhas da tabela inicial:", len(tabela_mestra_inicial))
print("Linhas com registros de alarmes:",
      tabela_mestra_inicial["tem_registro_alarme"].sum())
print("Linhas sem correspondência nos alarmes:",
      tabela_mestra_inicial["qtd_registros_alarmes"].isna().sum())

if not contagens_recorte.empty:
    primeiro_registro = contagens_recorte.iloc[0]
    exemplo = tabela_mestra_inicial.loc[
        tabela_mestra_inicial["chave_candidata"].eq(primeiro_registro["chave_candidata"])
        & tabela_mestra_inicial["data"].between(
            primeiro_registro["data"] - pd.Timedelta(days=3),
            primeiro_registro["data"] + pd.Timedelta(days=6),
        )
    ]
    display(exemplo)


Linhas da tabela inicial: 472032
Linhas com registros de alarmes: 47518
Linhas sem correspondência nos alarmes: 424514


,chave_candidata,estacao,plataforma,porta,tipo_ativo,data,qtd_registros_alarmes,tem_registro_alarme
21,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-07,<NA>,False
22,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-08,<NA>,False
23,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-09,<NA>,False
24,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-10,3,True
25,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-11,3,True
26,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-12,1,True
27,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-13,10,True
28,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-14,<NA>,False
29,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-15,<NA>,False
30,BUT|1|10|PDM,BUT,1,10,PDM,2023-11-16,<NA>,False


### Resultado e próxima etapa

A variável `tabela_mestra_inicial` contém uma linha por PDM e dia, com identificação e contagem de registros de alarmes. O resultado permanece em memória; nenhum CSV é exportado.

A próxima etapa deverá definir a contagem de registros de manutenção, tratando ordens repetidas e o nível de identificação de cada ocorrência. ZM01/ZM02 com identificação completa poderão ser associadas à mesma porta. Registros coletivos e ZM03 precisarão de regras próprias de contexto. A formação do alvo exigirá também a revisão dos registros ZM01 excluídos na limpeza e da cobertura das fontes.

O calendário inicial não confirma operação diária, coleta completa ou ausência de falhas. Essas condições precisam ser resolvidas antes do preenchimento de zeros, das janelas de 30 dias e do alvo de sete dias.

## 16. Regra de contagem das corretivas

Esta etapa acrescenta informações de ZM01 e ZM02 ao calendário de PDM. A unidade contada é a **ordem por porta e dia de início da avaria**. Essa contagem ainda não representa episódios de falha nem manutenções executadas.

Uma ordem repetida com o mesmo local e dia entra apenas uma vez na contagem. Quando a mesma ordem apresenta locais ou dias diferentes, seus registros ficam separados para revisão. Registros sem número de ordem também ficam separados; a nota é preservada na tabela de origem.

A identificação de divergências utiliza a base disponível inteira apenas para auditoria. Ela não representa o conhecimento que existia em cada dia histórico. Antes da modelagem, a versão e a disponibilidade temporal dos registros precisarão ser confirmadas.

O laço `for` aplica a mesma conferência às duas tabelas. `nunique(dropna=False)` conta valores distintos, incluindo ausências, para não esconder divergências.

In [20]:
corretivas = {"ZM01": zm01, "ZM02": zm02}
corretivas_para_revisar = {}
corretivas_sem_ambiguidade = {}
resumo_ordens = []

for nome, tabela in corretivas.items():
    variacoes = tabela.groupby("ordem")[["local_original", "data_avaria"]].nunique(dropna=False)
    ordens_divergentes = variacoes.index[
        (variacoes["local_original"] > 1) | (variacoes["data_avaria"] > 1)
    ]

    precisa_revisao = tabela["ordem"].isna() | tabela["ordem"].isin(ordens_divergentes)
    corretivas_para_revisar[nome] = tabela.loc[precisa_revisao].copy()
    corretivas_sem_ambiguidade[nome] = tabela.loc[~precisa_revisao].copy()

    assert len(corretivas_para_revisar[nome]) + len(corretivas_sem_ambiguidade[nome]) == len(tabela)

    resumo_ordens.append({
        "fonte": nome,
        "linhas_originais": len(tabela),
        "linhas_sem_ordem": tabela["ordem"].isna().sum(),
        "ordens_com_local_ou_dia_divergente": len(ordens_divergentes),
        "linhas_para_revisar": precisa_revisao.sum(),
    })

display(pd.DataFrame(resumo_ordens))
display(corretivas_para_revisar["ZM01"][[
    "ordem", "nota", "local_original", "data_avaria",
]].head(10))


,fonte,linhas_originais,linhas_sem_ordem,ordens_com_local_ou_dia_divergente,linhas_para_revisar
0,ZM01,10543,1,0,1
1,ZM02,1163,5,0,5


,ordem,nota,local_original,data_avaria
5263,NaN,14445462,SP04-6-PSD-MBI-PLT1-PP11PDM,2025-02-07


## 17. Contagens de ZM01 e ZM02 por porta e dia

A seleção considera identificações presentes na lista `portas` e datas dentro do recorte. Os demais tipos, registros coletivos e datas externas permanecem nas tabelas anteriores.

`.drop_duplicates()` gera uma referência por ordem/local/dia **somente para a contagem**. A operação não escolhe qual versão de texto, status ou diagnóstico da ordem está correta.

As colunas finais mantêm o termo `inicio_avaria` para distinguir essa referência de uma data de execução.

In [21]:
corretivas_no_recorte = {}
contagens_corretivas = {}
resumo_corretivas = []

for nome, tabela in corretivas_sem_ambiguidade.items():
    dentro_do_recorte = (
        tabela["chave_candidata"].isin(portas["chave_candidata"])
        & tabela["data_avaria"].between(DATA_INICIO, DATA_FIM)
    )
    selecionadas = tabela.loc[dentro_do_recorte].copy()
    ordens_unicas = selecionadas.drop_duplicates(["ordem", "local_original", "data_avaria"])
    corretivas_no_recorte[nome] = ordens_unicas

    contagem = (
        ordens_unicas.groupby(["chave_candidata", "data_avaria"])
        .size()
        .reset_index(name="qtd_ordens")
        .rename(columns={"data_avaria": "data"})
    )
    contagens_corretivas[nome] = contagem

    assert contagem["qtd_ordens"].sum() == len(ordens_unicas)
    resumo_corretivas.append({
        "fonte": nome,
        "linhas_selecionadas": len(selecionadas),
        "repeticoes_na_contagem": len(selecionadas) - len(ordens_unicas),
        "ordens_contadas": len(ordens_unicas),
    })

zm01_diaria = contagens_corretivas["ZM01"].rename(columns={
    "qtd_ordens": "qtd_ordens_zm01_inicio_avaria",
})
zm02_diaria = contagens_corretivas["ZM02"].rename(columns={
    "qtd_ordens": "qtd_ordens_zm02_inicio_avaria",
})

display(pd.DataFrame(resumo_corretivas))
display(zm01_diaria.head())
display(zm02_diaria.head())


,fonte,linhas_selecionadas,repeticoes_na_contagem,ordens_contadas
0,ZM01,7256,42,7214
1,ZM02,583,1,582


,chave_candidata,data,qtd_ordens_zm01_inicio_avaria
0,BUT|1|10|PDM,2023-11-10,2
1,BUT|1|10|PDM,2023-11-12,1
2,BUT|1|10|PDM,2023-11-13,2
3,BUT|1|10|PDM,2023-12-09,1
4,BUT|1|10|PDM,2023-12-20,1


,chave_candidata,data,qtd_ordens_zm02_inicio_avaria
0,BUT|1|10|PDM,2023-12-09,1
1,BUT|1|15|PDM,2025-08-07,1
2,BUT|1|18|PDM,2024-07-08,1
3,BUT|1|19|PDM,2024-12-04,1
4,BUT|1|23|PDM,2024-09-21,1


## 18. Junção das corretivas ao calendário

Cada junção usa a mesma porta e data. O `how="left"` preserva todas as linhas do calendário. O `validate="one_to_one"` impede a multiplicação de linhas caso uma contagem tenha chaves repetidas.

Valores sem correspondência permanecem ausentes. Em particular, a presença de ordens separadas para revisão e a cobertura ainda não confirmada impedem interpretar essas ausências automaticamente como zero falhas.

In [22]:
tabela_mestra_manutencoes = tabela_mestra_inicial.merge(
    zm01_diaria,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)

tabela_mestra_manutencoes = tabela_mestra_manutencoes.merge(
    zm02_diaria,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)

colunas_corretivas = [
    "qtd_ordens_zm01_inicio_avaria",
    "qtd_ordens_zm02_inicio_avaria",
]
tabela_mestra_manutencoes[colunas_corretivas] = (
    tabela_mestra_manutencoes[colunas_corretivas].astype("Int64")
)

assert len(tabela_mestra_manutencoes) == len(tabela_mestra_inicial)
assert tabela_mestra_manutencoes["qtd_ordens_zm01_inicio_avaria"].sum() == zm01_diaria["qtd_ordens_zm01_inicio_avaria"].sum()
assert tabela_mestra_manutencoes["qtd_ordens_zm02_inicio_avaria"].sum() == zm02_diaria["qtd_ordens_zm02_inicio_avaria"].sum()

display(tabela_mestra_manutencoes.loc[
    tabela_mestra_manutencoes[colunas_corretivas].notna().any(axis=1),
    ["chave_candidata", "data", *colunas_corretivas],
].head(10))


,chave_candidata,data,qtd_ordens_zm01_inicio_avaria,qtd_ordens_zm02_inicio_avaria
24,BUT|1|10|PDM,2023-11-10,2,<NA>
26,BUT|1|10|PDM,2023-11-12,1,<NA>
27,BUT|1|10|PDM,2023-11-13,2,<NA>
53,BUT|1|10|PDM,2023-12-09,1,1
64,BUT|1|10|PDM,2023-12-20,1,<NA>
304,BUT|1|10|PDM,2024-08-16,1,<NA>
323,BUT|1|10|PDM,2024-09-04,2,<NA>
371,BUT|1|10|PDM,2024-10-22,1,<NA>
407,BUT|1|10|PDM,2024-11-27,1,<NA>
494,BUT|1|10|PDM,2025-02-22,1,<NA>


## 19. ZM03: separação por estação e plataforma

A programação preventiva será tratada como **contexto coletivo**, sem indicar uma intervenção comprovada em cada porta.

A seleção reconhece somente os locais completos nos formatos:

- `SP04-6-PSD-BUT`: contexto da estação.
- `SP04-6-PSD-BUT-PLT1`: contexto da plataforma.

Um local genérico como `SP04-6-PSD` fica separado. A correspondência exata evita tratar um subcomponente como manutenção de toda a estação.

Nesta base, cada ordem ZM03 aparece uma vez. A verificação interrompe a contagem se uma versão futura contiver ordens repetidas, ausentes ou status de cancelamento desconhecidos. A coluna `cancelada` descreve o cadastro disponível: seu valor não comprova o status conhecido na data programada.

In [23]:
assert zm03["ordem"].notna().all(), "Há ordens ZM03 sem identificação."
assert not zm03["ordem"].duplicated().any(), "As ordens ZM03 repetidas precisam de revisão."
assert zm03["cancelada"].isin([True, False]).all(), "Há status de cancelamento desconhecido."

locais_zm03 = zm03["local_original"].astype("string").str.strip().str.upper()

local_estacao = locais_zm03.str.fullmatch(r"SP\d+-\d+-PSD-[A-Z]{3}", na=False)
local_plataforma = locais_zm03.str.fullmatch(r"SP\d+-\d+-PSD-[A-Z]{3}-PLT\d+", na=False)
data_no_recorte = zm03["data_programada"].between(DATA_INICIO, DATA_FIM)

zm03_estacao = zm03.loc[local_estacao & data_no_recorte].copy()
zm03_plataforma = zm03.loc[local_plataforma & data_no_recorte].copy()
zm03_sem_escopo = zm03.loc[~(local_estacao | local_plataforma)].copy()

print("Ordens de estação no recorte:", len(zm03_estacao))
print("Ordens de plataforma no recorte:", len(zm03_plataforma))
print("Ordens sem escopo reconhecido na base completa:", len(zm03_sem_escopo))
display(zm03_sem_escopo[["ordem", "local_original", "data_programada"]].head())


Ordens de estação no recorte: 128
Ordens de plataforma no recorte: 230
Ordens sem escopo reconhecido na base completa: 83


,ordem,local_original,data_programada
260,52907929,SP04-6-PSD,2024-07-29
261,52907928,SP04-6-PSD,2024-07-15
262,52907927,SP04-6-PSD,2024-07-08
263,52907926,SP04-6-PSD,2024-07-01
264,52907925,SP04-6-PSD,2024-06-24


## 20. Contagens da programação preventiva

A agregação separa ordens canceladas e não canceladas na versão disponível. **Não cancelada não significa executada.** As quatro colunas são de auditoria da programação e ainda não são entradas aprovadas para o modelo.

Uma mesma ordem de estação não entra também na contagem de plataforma. As duas tabelas preservam os níveis originais.

In [24]:
# Conversão explícita para booleano após a conferência do status.
zm03_estacao["cancelada"] = zm03_estacao["cancelada"].astype(bool)
zm03_estacao["nao_cancelada"] = ~zm03_estacao["cancelada"]

zm03_estacao_diaria = (
    zm03_estacao.groupby(["estacao", "data_programada"])
    .agg(
        qtd_zm03_nao_canceladas_estacao=("nao_cancelada", "sum"),
        qtd_zm03_canceladas_estacao=("cancelada", "sum"),
    )
    .reset_index()
    .rename(columns={"data_programada": "data"})
)

zm03_plataforma["cancelada"] = zm03_plataforma["cancelada"].astype(bool)
zm03_plataforma["nao_cancelada"] = ~zm03_plataforma["cancelada"]

zm03_plataforma_diaria = (
    zm03_plataforma.groupby(["estacao", "plataforma", "data_programada"])
    .agg(
        qtd_zm03_nao_canceladas_plataforma=("nao_cancelada", "sum"),
        qtd_zm03_canceladas_plataforma=("cancelada", "sum"),
    )
    .reset_index()
    .rename(columns={"data_programada": "data"})
)

assert zm03_estacao_diaria.filter(like="qtd_").sum().sum() == len(zm03_estacao)
assert zm03_plataforma_diaria.filter(like="qtd_").sum().sum() == len(zm03_plataforma)

display(zm03_estacao_diaria.head())
display(zm03_plataforma_diaria.head())


,estacao,data,qtd_zm03_nao_canceladas_estacao,qtd_zm03_canceladas_estacao
0,BUT,2023-10-30,1,0
1,BUT,2023-11-26,1,0
2,BUT,2023-12-28,1,0
3,BUT,2024-01-24,1,0
4,BUT,2024-02-23,1,0


,estacao,plataforma,data,qtd_zm03_nao_canceladas_plataforma,qtd_zm03_canceladas_plataforma
0,BUT,1,2024-01-02,1,0
1,BUT,1,2024-03-18,1,0
2,BUT,1,2024-06-03,1,0
3,BUT,1,2024-09-09,1,0
4,BUT,1,2024-12-05,1,0


## 21. Inclusão do contexto de ZM03

O contexto de estação é associado por estação e data. O de plataforma é associado por estação, plataforma e data. `validate="many_to_one"` permite que várias portas recebam o mesmo contexto, mas exige uma única linha na tabela coletiva para cada chave.

Por exemplo, uma ordem programada para BUT/PLT1 aparece como contexto nas linhas das PDM dessa plataforma naquele dia. Isso não representa uma ordem executada em cada PDM. Por esse motivo, essas colunas **não devem ser somadas entre portas** para obter o total de ordens.

As tabelas agregadas anteriores são a referência para totais de programação. Registros de estações ou plataformas sem PDM na lista continuam disponíveis nessas tabelas.

In [25]:
tabela_mestra_manutencoes = tabela_mestra_manutencoes.merge(
    zm03_estacao_diaria,
    on=["estacao", "data"],
    how="left",
    validate="many_to_one",
)

tabela_mestra_manutencoes = tabela_mestra_manutencoes.merge(
    zm03_plataforma_diaria,
    on=["estacao", "plataforma", "data"],
    how="left",
    validate="many_to_one",
)

colunas_preventivas = [
    "qtd_zm03_nao_canceladas_estacao",
    "qtd_zm03_canceladas_estacao",
    "qtd_zm03_nao_canceladas_plataforma",
    "qtd_zm03_canceladas_plataforma",
]
tabela_mestra_manutencoes[colunas_preventivas] = (
    tabela_mestra_manutencoes[colunas_preventivas].astype("Int64")
)

assert len(tabela_mestra_manutencoes) == len(tabela_mestra_inicial)
assert not tabela_mestra_manutencoes.duplicated(["chave_candidata", "data"]).any()
assert tabela_mestra_manutencoes["qtd_registros_alarmes"].equals(
    tabela_mestra_inicial["qtd_registros_alarmes"]
)

print("Linhas após as quatro junções:", len(tabela_mestra_manutencoes))
print("Colunas adicionadas:", len(colunas_corretivas + colunas_preventivas))
display(tabela_mestra_manutencoes.loc[
    tabela_mestra_manutencoes[colunas_preventivas].notna().any(axis=1),
    ["chave_candidata", "data", *colunas_preventivas],
].head(10))


Linhas após as quatro junções: 472032
Colunas adicionadas: 6


,chave_candidata,data,qtd_zm03_nao_canceladas_estacao,qtd_zm03_canceladas_estacao,qtd_zm03_nao_canceladas_plataforma,qtd_zm03_canceladas_plataforma
13,BUT|1|10|PDM,2023-10-30,1,0,<NA>,<NA>
40,BUT|1|10|PDM,2023-11-26,1,0,<NA>,<NA>
72,BUT|1|10|PDM,2023-12-28,1,0,<NA>,<NA>
77,BUT|1|10|PDM,2024-01-02,<NA>,<NA>,1,0
99,BUT|1|10|PDM,2024-01-24,1,0,<NA>,<NA>
129,BUT|1|10|PDM,2024-02-23,1,0,<NA>,<NA>
153,BUT|1|10|PDM,2024-03-18,1,0,1,0
183,BUT|1|10|PDM,2024-04-17,1,0,<NA>,<NA>
230,BUT|1|10|PDM,2024-06-03,<NA>,<NA>,1,0
326,BUT|1|10|PDM,2024-09-07,1,0,<NA>,<NA>


### Resultado da integração de manutenções

A variável `tabela_mestra_manutencoes` reúne o calendário de PDM, as contagens de registros de alarmes, as ordens ZM01/ZM02 por início da avaria e o contexto de programação ZM03. O resultado permanece em memória e os arquivos de origem permanecem intactos.

As tabelas `corretivas_para_revisar`, `zm03_sem_escopo` e a aba `Outliers_removidos` de ZM01 preservam casos que exigem análise antes da definição do alvo. As bases anteriores também mantêm registros coletivos e outros tipos de ativo que não entraram nas contagens individuais.

**Próximo passo:** revisar o critério de falha ZM01, os registros separados e o período de cobertura. Essas decisões permitem distinguir dias sem falha de dias sem informação. Somente depois poderão ser criados `falha_no_dia`, `falha_7d` e as variáveis históricas. Datas programadas de ZM03 e status atuais não serão usados automaticamente como histórico de execução.

## 22. Significado de `falha_flag`

O notebook de limpeza `eda/cleaning/limpeza-dados-ZM01.ipynb` transforma a marcação SAP `X` em `True`. Campos vazios são transformados em `False`. Portanto, **False significa ausência da marcação, não comprovação de ausência de falha**.

A comparação abaixo mantém dois critérios candidatos: toda ZM01 individual identificada e somente ZM01 com marcação de falha. Nenhum critério é adotado como alvo definitivo nesta etapa. Alarmes associados poderão complementar a análise, mas a falta de um alarme correspondente não invalida automaticamente uma ZM01.

In [26]:
resumo_flags = pd.DataFrame({
    "base_principal": bases["ZM01"]["falha_flag"].value_counts(dropna=False),
    "outliers_separados": auditorias[("ZM01", "Outliers_removidos")]["falha_flag"].value_counts(dropna=False),
}).rename_axis("marcacao_falha_sap")

display(resumo_flags)


,base_principal,outliers_separados
marcacao_falha_sap,,
False,7150,56
True,3393,47


## 23. Registros ZM01 separados pela duração

A aba `Outliers_removidos` contém registros acima do limite de duração usado na limpeza. Uma duração elevada não é, por si só, prova de erro. Esta auditoria identifica quais registros poderiam afetar o alvo de PDM no recorte.

A extração reutiliza a regra de identificação já presente na seção 4. Os registros continuam separados: sua inclusão dependerá da revisão do local, da data e da validade da ocorrência. A duração final e o texto atualizado da ordem não são utilizados como variáveis históricas.

In [27]:
zm01_outliers = auditorias[("ZM01", "Outliers_removidos")][[
    "ordem", "nota", "local_de_instalacao", "inicio_avaria_datetime",
    "fim_avaria_datetime", "duracao_avaria_horas", "falha_flag",
]].copy()

identificacao_outliers = extrair_ativos(zm01_outliers["local_de_instalacao"])
zm01_outliers = zm01_outliers.join(identificacao_outliers[colunas_identificacao])
zm01_outliers["data_avaria"] = pd.to_datetime(
    zm01_outliers["inicio_avaria_datetime"], errors="coerce"
).dt.normalize()

outliers_pdm_recorte = zm01_outliers.loc[
    zm01_outliers["tipo_ativo"].eq(TIPO_ATIVO)
    & zm01_outliers["chave_candidata"].notna()
    & zm01_outliers["data_avaria"].between(DATA_INICIO, DATA_FIM)
].copy()

print("Registros na aba de outliers:", len(zm01_outliers))
print("Registros PDM identificados dentro do período:", len(outliers_pdm_recorte))
print("Desses, sem porta no calendário atual:",
      (~outliers_pdm_recorte["chave_candidata"].isin(portas["chave_candidata"])).sum())

display(outliers_pdm_recorte[[
    "ordem", "chave_candidata", "data_avaria", "duracao_avaria_horas", "falha_flag",
]].head(10))


Registros na aba de outliers: 103
Registros PDM identificados dentro do período: 40
Desses, sem porta no calendário atual: 0


,ordem,chave_candidata,data_avaria,duracao_avaria_horas,falha_flag
8,53465108.0,BUT|2|4|PDM,2025-02-10,1204.500000,False
27,53909221.0,FRE|2|4|PDM,2025-10-19,25.500000,False
29,53643455.0,HIG|2|14|PDM,2025-05-20,28.333333,False
31,52960553.0,LUZ|1|4|PDM,2024-05-15,2854.288889,True
32,54227610.0,LUZ|1|23|PDM,2026-03-21,69.395278,False
33,52761904.0,LUZ|2|4|PDM,2024-01-26,1454.000000,True
34,54227707.0,LUZ|2|10|PDM,2026-03-26,278.603333,False
35,54227520.0,LUZ|2|13|PDM,2026-03-13,64.232500,False
36,54227621.0,LUZ|2|17|PDM,2026-03-26,273.609167,False
37,54227445.0,LUZ|2|23|PDM,2026-03-07,206.749167,False


## 24. Comparação dos critérios por porta e dia

A comparação utiliza a base principal ZM01, com ordem identificada e sem divergência de local/dia, conforme a seção 16. Registros separados para revisão e outliers ainda não entram nos candidatos.

O agrupamento reúne registros da mesma porta no mesmo dia. Uma marcação `True` em qualquer registro torna o dia positivo no critério restrito. Isso evita escolher arbitrariamente a marcação de uma versão repetida da ordem.

A coluna `tem_zm01_flag` descreve a presença de uma marcação; seu valor zero não é um rótulo de ausência de falha. A tabela não reconstrói episódios contínuos nem prova quando os dados foram disponibilizados.

In [28]:
zm01_candidatas = corretivas_sem_ambiguidade["ZM01"].loc[
    corretivas_sem_ambiguidade["ZM01"]["chave_candidata"].isin(portas["chave_candidata"])
    & corretivas_sem_ambiguidade["ZM01"]["data_avaria"].between(DATA_INICIO, DATA_FIM)
].copy()

assert zm01_candidatas["falha_flag"].isin([True, False]).all()

eventos_zm01_candidatos = (
    zm01_candidatas.groupby(["chave_candidata", "data_avaria"])
    .agg(tem_zm01_flag=("falha_flag", "max"))
    .reset_index()
)
eventos_zm01_candidatos["tem_zm01"] = 1
eventos_zm01_candidatos["tem_zm01_flag"] = eventos_zm01_candidatos["tem_zm01_flag"].astype(int)

comparacao_criterios = pd.DataFrame({
    "criterio": ["Toda ZM01 individual elegível", "ZM01 individual com marcação de falha"],
    "dias_porta_com_evidencia": [
        len(eventos_zm01_candidatos),
        eventos_zm01_candidatos["tem_zm01_flag"].sum(),
    ],
})

display(comparacao_criterios)
display(eventos_zm01_candidatos.head())


,criterio,dias_porta_com_evidencia
0,Toda ZM01 individual elegível,6228
1,ZM01 individual com marcação de falha,1962


,chave_candidata,data_avaria,tem_zm01_flag,tem_zm01
0,BUT|1|10|PDM,2023-11-10,1,1
1,BUT|1|10|PDM,2023-11-12,1,1
2,BUT|1|10|PDM,2023-11-13,1,1
3,BUT|1|10|PDM,2023-12-09,1,1
4,BUT|1|10|PDM,2023-12-20,1,1


## 25. Presença de registros de alarmes por estação e mês

O diagnóstico conta dias com pelo menos um registro PDM na estação. Ele ajuda a localizar períodos que precisam de investigação. **Dias sem registros não comprovam falha na coleta nem ausência de alarmes.** Mesmo um dia com registros em uma porta não comprova coleta completa nas demais.

Nesta tabela de auditoria, zero significa literalmente zero registros encontrados. Esse preenchimento não altera os valores ausentes das variáveis da tabela mestra. O denominador usa apenas os dias do recorte, inclusive nos meses inicial e final.

In [29]:
alarmes_recorte_auditoria = alarmes_pdm.loc[
    alarmes_pdm["data_alarme"].between(DATA_INICIO, DATA_FIM)
    & alarmes_pdm["estacao"].isin(portas["estacao"])
].copy()
alarmes_recorte_auditoria["mes"] = alarmes_recorte_auditoria["data_alarme"].dt.to_period("M")

presenca_mensal = (
    alarmes_recorte_auditoria.groupby(["estacao", "mes"])
    .agg(
        dias_com_registro=("data_alarme", "nunique"),
        qtd_registros=("data_alarme", "size"),
    )
    .reset_index()
)

dias_por_mes = (
    calendario["data"].dt.to_period("M").value_counts()
    .sort_index().rename_axis("mes").reset_index(name="dias_no_recorte")
)
estacoes = portas[["estacao"]].drop_duplicates()
auditoria_mensal = estacoes.merge(dias_por_mes, how="cross")
auditoria_mensal = auditoria_mensal.merge(
    presenca_mensal, on=["estacao", "mes"], how="left", validate="one_to_one"
)
auditoria_mensal[["dias_com_registro", "qtd_registros"]] = (
    auditoria_mensal[["dias_com_registro", "qtd_registros"]].fillna(0).astype(int)
)
auditoria_mensal["dias_sem_registro"] = (
    auditoria_mensal["dias_no_recorte"] - auditoria_mensal["dias_com_registro"]
)

assert auditoria_mensal["dias_sem_registro"].ge(0).all()
assert auditoria_mensal["qtd_registros"].sum() == len(alarmes_recorte_auditoria)

print("Combinações estação/mês sem registros:",
      auditoria_mensal["qtd_registros"].eq(0).sum())
display(auditoria_mensal.sort_values(
    ["dias_sem_registro", "estacao", "mes"], ascending=[False, True, True]
).head(20))


Combinações estação/mês sem registros: 78


,estacao,mes,dias_no_recorte,dias_com_registro,qtd_registros,dias_sem_registro
51,FAL,2025-07,31,0,0,31
52,FAL,2025-08,31,0,0,31
54,FAL,2025-10,31,0,0,31
56,FAL,2025-12,31,0,0,31
92,FRE,2023-12,31,0,0,31
93,FRE,2024-01,31,0,0,31
95,FRE,2024-03,31,0,0,31
97,FRE,2024-05,31,0,0,31
99,FRE,2024-07,31,0,0,31
100,FRE,2024-08,31,0,0,31


## 26. Prévia dos positivos nos próximos sete dias

A prévia apresenta os dois critérios em colunas separadas, sem escolher o alvo definitivo. Uma ZM01 candidata com início em 10/01 torna positivas as datas de referência entre 03/01 e 09/01: a ocorrência está entre D+1 e D+7. O próprio dia 10/01 não fica positivo por essa ocorrência.

O código combina cada dia com evidência com as antecedências de 1 a 7 dias. Depois, reúne as datas de referência iguais. Essa abordagem evita criar uma função ou usar janelas móveis nesta etapa.

**Os únicos valores da prévia são 1 e ausente.** Um valor 1 indica evidência pelo critério candidato; ausente não significa 0. Sem confirmação de cobertura e revisão dos critérios, não há base suficiente para gerar negativos confiáveis. Os últimos sete dias do calendário também permanecem ausentes para manter somente horizontes inteiramente dentro do recorte.

In [30]:
antecedencias = pd.DataFrame({"dias_antes": range(1, 8)})
referencias_candidatas = eventos_zm01_candidatos.merge(antecedencias, how="cross")

referencias_candidatas["data"] = (
    referencias_candidatas["data_avaria"]
    - pd.to_timedelta(referencias_candidatas["dias_antes"], unit="D")
)

# Apenas datas D com os sete dias seguintes dentro do calendário.
ultimo_dia_com_horizonte = DATA_FIM - pd.Timedelta(days=7)
referencias_candidatas = referencias_candidatas.loc[
    referencias_candidatas["data"].between(DATA_INICIO, ultimo_dia_com_horizonte)
].copy()

diferenca_dias = (
    referencias_candidatas["data_avaria"] - referencias_candidatas["data"]
).dt.days
assert diferenca_dias.between(1, 7).all()

positivos_candidatos = (
    referencias_candidatas.groupby(["chave_candidata", "data"])[
        ["tem_zm01", "tem_zm01_flag"]
    ]
    .max()
    .reset_index()
    .rename(columns={
        "tem_zm01": "previa_registro_zm01_7d",
        "tem_zm01_flag": "previa_registro_zm01_marcada_7d",
    })
)

# Sem marcação positiva no critério restrito, o resultado permanece desconhecido.
positivos_candidatos["previa_registro_zm01_marcada_7d"] = (
    positivos_candidatos["previa_registro_zm01_marcada_7d"]
    .where(positivos_candidatos["previa_registro_zm01_marcada_7d"].eq(1), pd.NA)
    .astype("Int64")
)

tabela_mestra_previa_alvo = tabela_mestra_manutencoes.merge(
    positivos_candidatos,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)
colunas_previa = ["previa_registro_zm01_7d", "previa_registro_zm01_marcada_7d"]
tabela_mestra_previa_alvo[colunas_previa] = tabela_mestra_previa_alvo[colunas_previa].astype("Int64")

assert len(tabela_mestra_previa_alvo) == len(tabela_mestra_manutencoes)
assert not tabela_mestra_previa_alvo.duplicated(["chave_candidata", "data"]).any()
assert tabela_mestra_previa_alvo[colunas_previa].stack().eq(1).all()
assert tabela_mestra_previa_alvo.loc[
    tabela_mestra_previa_alvo["data"] > ultimo_dia_com_horizonte, colunas_previa
].isna().all().all()
assert tabela_mestra_previa_alvo.loc[
    tabela_mestra_previa_alvo["previa_registro_zm01_marcada_7d"].eq(1),
    "previa_registro_zm01_7d",
].eq(1).all()

display(tabela_mestra_previa_alvo[colunas_previa].notna().sum().to_frame("positivos_candidatos"))
display(tabela_mestra_previa_alvo.loc[
    tabela_mestra_previa_alvo["previa_registro_zm01_7d"].eq(1),
    ["chave_candidata", "data", *colunas_previa],
].head(10))


,positivos_candidatos
previa_registro_zm01_7d,36715
previa_registro_zm01_marcada_7d,12539


,chave_candidata,data,previa_registro_zm01_7d,previa_registro_zm01_marcada_7d
17,BUT|1|10|PDM,2023-11-03,1,1
18,BUT|1|10|PDM,2023-11-04,1,1
19,BUT|1|10|PDM,2023-11-05,1,1
20,BUT|1|10|PDM,2023-11-06,1,1
21,BUT|1|10|PDM,2023-11-07,1,1
22,BUT|1|10|PDM,2023-11-08,1,1
23,BUT|1|10|PDM,2023-11-09,1,1
24,BUT|1|10|PDM,2023-11-10,1,1
25,BUT|1|10|PDM,2023-11-11,1,1
26,BUT|1|10|PDM,2023-11-12,1,1


### Resultado da revisão do alvo

A seção apresenta a comparação dos critérios ZM01, os outliers PDM candidatos à revisão, o diagnóstico mensal dos alarmes e uma prévia dos positivos de sete dias. A variável `tabela_mestra_previa_alvo` mantém as duas alternativas separadas.

A prévia **não é uma base de treinamento**: não contém negativos definidos. Valores ausentes não devem ser preenchidos com zero nesta etapa. Nenhuma correlação entre alarmes e manutenção foi presumida.

A definição do alvo final depende de três decisões: o critério de falha, o tratamento dos registros separados e o período em que o registro de falhas pode ser considerado completo. A cobertura dos alarmes precisa de avaliação própria para as entradas temporais. Até essas decisões, a prévia permanece exploratória e nenhuma base é exportada.

## 27. Referência provisória por registro ZM01

A primeira versão adota **toda ZM01 individual elegível como indicador de falha**, independentemente de `falha_flag`. Trata-se de uma definição operacional inicial: uma ordem emergencial serve como evidência, mas não comprova sozinha uma quebra física.

A elegibilidade segue as seções anteriores: PDM identificada, ordem presente, ausência de divergência de local/dia e início de avaria dentro do recorte. Os registros separados na auditoria ainda não entram nesse critério. ZM02 e ZM03 permanecem como fontes de histórico ou contexto, sem gerar positivos automaticamente.

A coluna `registro_zm01_7d` recebe a prévia correspondente a esse critério. Ela continua **parcial**: contém 1 quando há evidência elegível nos sete dias seguintes e ausente nos demais casos. A geração de zero depende de cobertura confirmada e da revisão dos registros separados.

As colunas de alvo e status descrevem o resultado futuro. Elas nunca devem integrar as entradas da LSTM.

In [31]:
tabela_mestra_historico = (
    tabela_mestra_previa_alvo
    .sort_values(["chave_candidata", "data"])
    .reset_index(drop=True)
    .copy()
)

tabela_mestra_historico["registro_zm01_7d"] = (
    tabela_mestra_historico["previa_registro_zm01_7d"].astype("Int64")
)

tabela_mestra_historico["status_rotulo"] = "sem negativo definido"
tabela_mestra_historico.loc[
    tabela_mestra_historico["registro_zm01_7d"].eq(1),
    "status_rotulo",
] = "positivo pelo critério ZM01"

tabela_mestra_historico.loc[
    tabela_mestra_historico["data"] > ultimo_dia_com_horizonte,
    "status_rotulo",
] = "horizonte ultrapassa o recorte"

display(tabela_mestra_historico["status_rotulo"].value_counts().to_frame("linhas"))


,linhas
status_rotulo,
sem negativo definido,431621
positivo pelo critério ZM01,36715
horizonte ultrapassa o recorte,3696


## 28. Significado de dias desde a última ZM01/ZM02

As duas primeiras variáveis históricas medem os dias desde o último **início de avaria registrado** em uma ordem elegível da mesma porta. Elas não medem tempo desde uma manutenção executada.

O cálculo considera uma previsão feita ao final de D. Por isso, uma avaria registrada em D produz valor zero nesse dia. Sem um novo registro, a diferença aumenta com a passagem dos dias:

| Data de referência | Último início de avaria registrado | Dias desde o registro |
|---|---|---:|
| 10/01 | 10/01 | 0 |
| 11/01 | 10/01 | 1 |
| 12/01 | 10/01 | 2 |

Sem histórico anterior elegível, o resultado permanece ausente. O valor zero não substitui histórico desconhecido.

O cálculo também aproveita registros anteriores a `DATA_INICIO`, evitando reiniciar o histórico artificialmente no primeiro dia do calendário. A sequência fica ordenada por porta e data.

Esta etapa usa as datas de ocorrência existentes na extração. Ainda será necessário confirmar quando cada registro ficou disponível: um início de avaria preenchido posteriormente não poderia ter sido usado numa previsão passada.

## 29. Dias desde a última ZM01 registrada

O código identifica a última avaria elegível anterior ao calendário, marca os dias que têm registros no recorte e propaga a última data apenas para a frente, dentro de cada porta.

- `groupby().max()` obtém a última data anterior ao recorte para cada porta.
- `where()` conserva a data somente nas linhas com um registro ZM01.
- `groupby().ffill()` propaga a última data conhecida dentro da mesma porta.
- `map()` recupera o histórico anterior quando ainda não houve registro no calendário.

Esse preenchimento aplica-se à **data da última ocorrência registrada**. Ele não preenche contagens ausentes de alarmes ou rótulos negativos.

In [32]:
# Histórico elegível anterior ao início do calendário.
historico_zm01 = corretivas_sem_ambiguidade["ZM01"]
zm01_anteriores = historico_zm01.loc[
    historico_zm01["chave_candidata"].isin(portas["chave_candidata"])
    & historico_zm01["data_avaria"].lt(DATA_INICIO)
]
ultima_zm01_antes = zm01_anteriores.groupby("chave_candidata")["data_avaria"].max()

# Marcação das datas com registros dentro do calendário.
tem_zm01_no_dia = tabela_mestra_historico["qtd_ordens_zm01_inicio_avaria"].gt(0)
tabela_mestra_historico["data_ultima_zm01_registrada"] = (
    tabela_mestra_historico["data"].where(tem_zm01_no_dia)
)

# Propagação para a frente, sempre dentro da mesma porta.
tabela_mestra_historico["data_ultima_zm01_registrada"] = (
    tabela_mestra_historico.groupby("chave_candidata")["data_ultima_zm01_registrada"].ffill()
)

# Uso da última avaria anterior ao calendário quando houver registro.
tabela_mestra_historico["data_ultima_zm01_registrada"] = (
    tabela_mestra_historico["data_ultima_zm01_registrada"].fillna(
        tabela_mestra_historico["chave_candidata"].map(ultima_zm01_antes)
    )
)

tabela_mestra_historico["dias_desde_ultima_zm01"] = (
    tabela_mestra_historico["data"]
    - tabela_mestra_historico["data_ultima_zm01_registrada"]
).dt.days.astype("Int64")

display(tabela_mestra_historico[[
    "chave_candidata", "data", "data_ultima_zm01_registrada", "dias_desde_ultima_zm01",
]].head(10))


,chave_candidata,data,data_ultima_zm01_registrada,dias_desde_ultima_zm01
0,BUT|1|10|PDM,2023-10-17,2023-07-16,93
1,BUT|1|10|PDM,2023-10-18,2023-07-16,94
2,BUT|1|10|PDM,2023-10-19,2023-07-16,95
3,BUT|1|10|PDM,2023-10-20,2023-07-16,96
4,BUT|1|10|PDM,2023-10-21,2023-07-16,97
5,BUT|1|10|PDM,2023-10-22,2023-07-16,98
6,BUT|1|10|PDM,2023-10-23,2023-07-16,99
7,BUT|1|10|PDM,2023-10-24,2023-07-16,100
8,BUT|1|10|PDM,2023-10-25,2023-07-16,101
9,BUT|1|10|PDM,2023-10-26,2023-07-16,102


## 30. Dias desde a última ZM02 registrada

O cálculo repete os passos da ZM01 para facilitar a comparação entre as duas fontes. O nome `dias_desde_ultima_zm02` mantém a nomenclatura proposta para o modelo, mas sua referência atual é o início da avaria da ZM02, e não a data programada ou executada da manutenção.

A variável `dias_desde_ultima_zm03` ainda não é criada: as datas disponíveis de ZM03 foram tratadas como programação, e não há identificação individual de porta ou confirmação da execução.

In [33]:
historico_zm02 = corretivas_sem_ambiguidade["ZM02"]
zm02_anteriores = historico_zm02.loc[
    historico_zm02["chave_candidata"].isin(portas["chave_candidata"])
    & historico_zm02["data_avaria"].lt(DATA_INICIO)
]
ultima_zm02_antes = zm02_anteriores.groupby("chave_candidata")["data_avaria"].max()

tem_zm02_no_dia = tabela_mestra_historico["qtd_ordens_zm02_inicio_avaria"].gt(0)
tabela_mestra_historico["data_ultima_zm02_registrada"] = (
    tabela_mestra_historico["data"].where(tem_zm02_no_dia)
)

tabela_mestra_historico["data_ultima_zm02_registrada"] = (
    tabela_mestra_historico.groupby("chave_candidata")["data_ultima_zm02_registrada"].ffill()
)

tabela_mestra_historico["data_ultima_zm02_registrada"] = (
    tabela_mestra_historico["data_ultima_zm02_registrada"].fillna(
        tabela_mestra_historico["chave_candidata"].map(ultima_zm02_antes)
    )
)

tabela_mestra_historico["dias_desde_ultima_zm02"] = (
    tabela_mestra_historico["data"]
    - tabela_mestra_historico["data_ultima_zm02_registrada"]
).dt.days.astype("Int64")

display(tabela_mestra_historico[[
    "chave_candidata", "data", "data_ultima_zm02_registrada", "dias_desde_ultima_zm02",
]].head(10))


,chave_candidata,data,data_ultima_zm02_registrada,dias_desde_ultima_zm02
0,BUT|1|10|PDM,2023-10-17,NaT,<NA>
1,BUT|1|10|PDM,2023-10-18,NaT,<NA>
2,BUT|1|10|PDM,2023-10-19,NaT,<NA>
3,BUT|1|10|PDM,2023-10-20,NaT,<NA>
4,BUT|1|10|PDM,2023-10-21,NaT,<NA>
5,BUT|1|10|PDM,2023-10-22,NaT,<NA>
6,BUT|1|10|PDM,2023-10-23,NaT,<NA>
7,BUT|1|10|PDM,2023-10-24,NaT,<NA>
8,BUT|1|10|PDM,2023-10-25,NaT,<NA>
9,BUT|1|10|PDM,2023-10-26,NaT,<NA>


## 31. Conferência das variáveis históricas

As verificações conferem que as datas utilizadas não ultrapassam D, que os dias desde a última ocorrência nunca são negativos e que um registro no próprio dia resulta em zero. Uma comparação no início do calendário confere o aproveitamento do histórico anterior.

Essas verificações validam o cálculo pelas datas de ocorrência. Elas não substituem a confirmação de disponibilidade histórica dos dados.

In [34]:
assert len(tabela_mestra_historico) == len(tabela_mestra_manutencoes)
assert not tabela_mestra_historico.duplicated(["chave_candidata", "data"]).any()
assert tabela_mestra_historico["registro_zm01_7d"].dropna().eq(1).all()

for tipo in ["zm01", "zm02"]:
    coluna_data = f"data_ultima_{tipo}_registrada"
    coluna_dias = f"dias_desde_ultima_{tipo}"
    coluna_contagem = f"qtd_ordens_{tipo}_inicio_avaria"

    tem_historico = tabela_mestra_historico[coluna_data].notna()
    tem_registro_hoje = tabela_mestra_historico[coluna_contagem].gt(0)

    assert (
        tabela_mestra_historico.loc[tem_historico, coluna_data]
        <= tabela_mestra_historico.loc[tem_historico, "data"]
    ).all()
    assert tabela_mestra_historico[coluna_dias].dropna().ge(0).all()
    assert tabela_mestra_historico.loc[tem_registro_hoje, coluna_dias].eq(0).all()

inicio_calendario = tabela_mestra_historico.loc[
    tabela_mestra_historico["data"].eq(DATA_INICIO)
].copy()
sem_zm01_no_inicio = inicio_calendario["qtd_ordens_zm01_inicio_avaria"].isna()
esperado_zm01 = inicio_calendario.loc[
    sem_zm01_no_inicio, "chave_candidata"
].map(ultima_zm01_antes)
pd.testing.assert_series_equal(
    inicio_calendario.loc[sem_zm01_no_inicio, "data_ultima_zm01_registrada"],
    esperado_zm01,
    check_names=False,
)

sem_zm02_no_inicio = inicio_calendario["qtd_ordens_zm02_inicio_avaria"].isna()
esperado_zm02 = inicio_calendario.loc[
    sem_zm02_no_inicio, "chave_candidata"
].map(ultima_zm02_antes)
pd.testing.assert_series_equal(
    inicio_calendario.loc[sem_zm02_no_inicio, "data_ultima_zm02_registrada"],
    esperado_zm02,
    check_names=False,
)

colunas_historico = ["dias_desde_ultima_zm01", "dias_desde_ultima_zm02"]
display(tabela_mestra_historico[colunas_historico].describe())
display(tabela_mestra_historico[colunas_historico].isna().sum().to_frame("linhas_sem_historico"))

print("Linhas da tabela:", len(tabela_mestra_historico))
print("Positivos pelo critério ZM01:", tabela_mestra_historico["registro_zm01_7d"].eq(1).sum())
print("Negativos definidos:", tabela_mestra_historico["registro_zm01_7d"].eq(0).sum())


,dias_desde_ultima_zm01,dias_desde_ultima_zm02
count,449751.0,228377.0
mean,109.416691,319.799796
std,128.96424,249.255303
min,0.0,0.0
25%,25.0,115.0
50%,66.0,266.0
75%,145.0,471.0
max,1092.0,1180.0


,linhas_sem_historico
dias_desde_ultima_zm01,22281
dias_desde_ultima_zm02,243655


Linhas da tabela: 472032
Positivos pelo critério ZM01: 36715
Negativos definidos: 0


### Resultado das primeiras variáveis históricas

A variável `tabela_mestra_historico` contém o alvo parcial pelo critério ZM01 e as variáveis `dias_desde_ultima_zm01` e `dias_desde_ultima_zm02`. As duas colunas auxiliares com a data do último registro permitem conferir o cálculo; não são novas variáveis numéricas para a LSTM.

O resultado continua exploratório. A base ainda não contém negativos definidos, a cobertura permanece pendente e os registros separados para revisão não foram incorporados. A coluna `status_rotulo` explicita essas limitações por linha.

O próximo cálculo será o histórico de alarmes em janelas de 7 e 14 dias. A definição de ausência de registros terá de permanecer explícita para não transformar falta de informação em zero eventos. A montagem das sequências e o treinamento dependem da resolução dessas questões.

## 32. Definição da unidade de contagem dos alarmes

A base contém linhas com a mesma TAG e o mesmo início de alarme, mas diferentes horários de normalização. A nova contagem define uma **ativação registrada** como um par único de TAG e início do alarme. A definição é operacional e poderá ser revista se houver um identificador de evento mais preciso.

O código seleciona somente identificação e início. Assim, `drop_duplicates()` não escolhe qual horário de normalização está correto. A análise da duração permanece pendente.

A coluna anterior `qtd_registros_alarmes` continua como auditoria das linhas da fonte. As novas colunas usam o nome `qtd_alarmes_registrados` e contam ativações distintas. A mudança de unidade de contagem não altera as tabelas anteriores.

In [35]:
colunas_ativacao = [
    "chave_candidata", "tag", "inicio_alarme", "data_alarme",
]

# Seleção das portas do calendário, preservando o histórico anterior ao recorte.
registros_portas = alarmes_identificados.loc[
    alarmes_identificados["chave_candidata"].isin(portas["chave_candidata"]),
    colunas_ativacao,
].copy()

# A mesma TAG e o mesmo instante de início devem identificar o mesmo ativo.
identificacoes_por_inicio = (
    registros_portas.groupby(["tag", "inicio_alarme"])["chave_candidata"].nunique()
)
assert identificacoes_por_inicio.le(1).all()

ativacoes_alarmes = registros_portas.drop_duplicates(["tag", "inicio_alarme"]).copy()

ativacoes_diarias = (
    ativacoes_alarmes.groupby(["chave_candidata", "data_alarme"])
    .size()
    .reset_index(name="qtd_alarmes_registrados_1d")
    .rename(columns={"data_alarme": "data"})
)

assert ativacoes_diarias["qtd_alarmes_registrados_1d"].sum() == len(ativacoes_alarmes)

print("Linhas de alarmes das portas selecionadas:", len(registros_portas))
print("Ativações distintas registradas:", len(ativacoes_alarmes))
print("Linhas excedentes na contagem de ativações:", len(registros_portas) - len(ativacoes_alarmes))
display(ativacoes_diarias.head())


Linhas de alarmes das portas selecionadas: 678166
Ativações distintas registradas: 304143
Linhas excedentes na contagem de ativações: 374023


,chave_candidata,data,qtd_alarmes_registrados_1d
0,BUT|1|10|PDM,2023-11-10,3
1,BUT|1|10|PDM,2023-11-11,3
2,BUT|1|10|PDM,2023-11-12,1
3,BUT|1|10|PDM,2023-11-13,10
4,BUT|1|10|PDM,2023-11-21,2


## 33. Calendário auxiliar com 13 dias anteriores

Uma janela de 14 dias em D usa D−13 até D. Por isso, o calendário auxiliar começa 13 dias antes de `DATA_INICIO`. A extensão permite calcular a primeira janela inteira do calendário principal, sem usar dias futuros.

Nesta tabela auxiliar, **zero significa que nenhuma ativação foi encontrada no arquivo para aquela porta e dia**. Isso não comprova ausência de alarmes na operação. As colunas de auditoria anteriores continuam com seus valores ausentes, e o alvo não recebe zeros.

A coluna `dia_com_registro` marca dias em que há pelo menos uma ativação. Ela indica presença de registros, não cobertura completa da coleta.

In [36]:
inicio_historico_alarmes = DATA_INICIO - pd.Timedelta(days=13)

calendario_alarmes = pd.DataFrame({
    "data": pd.date_range(inicio_historico_alarmes, DATA_FIM, freq="D")
})
historico_alarmes_diario = portas[["chave_candidata"]].merge(
    calendario_alarmes, how="cross"
)

historico_alarmes_diario = historico_alarmes_diario.merge(
    ativacoes_diarias,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)

historico_alarmes_diario["dia_com_registro"] = (
    historico_alarmes_diario["qtd_alarmes_registrados_1d"].notna().astype(int)
)

# Zero apenas na contagem de ativações encontradas no arquivo.
historico_alarmes_diario["qtd_alarmes_registrados_1d"] = (
    historico_alarmes_diario["qtd_alarmes_registrados_1d"].fillna(0).astype(int)
)

historico_alarmes_diario = (
    historico_alarmes_diario
    .sort_values(["chave_candidata", "data"])
    .reset_index(drop=True)
)

assert len(historico_alarmes_diario) == len(portas) * len(calendario_alarmes)
assert not historico_alarmes_diario.duplicated(["chave_candidata", "data"]).any()
display(historico_alarmes_diario.head())


,chave_candidata,data,qtd_alarmes_registrados_1d,dia_com_registro
0,BUT|1|10|PDM,2023-10-04,0,0
1,BUT|1|10|PDM,2023-10-05,0,0
2,BUT|1|10|PDM,2023-10-06,0,0
3,BUT|1|10|PDM,2023-10-07,0,0
4,BUT|1|10|PDM,2023-10-08,0,0


## 34. Contagens móveis de 7 e 14 dias

`rolling(7)` percorre grupos de sete linhas consecutivas. Como o calendário tem exatamente uma linha por porta e dia, sete linhas representam sete dias. O agrupamento por chave impede a mistura entre portas.

- Janela de 7 dias: D−6 até D.
- Janela de 14 dias: D−13 até D.
- `min_periods` exige que a janela tenha a quantidade completa de dias do calendário auxiliar.

A saída do agrupamento tem um índice adicional com a chave da porta. `reset_index(level=0, drop=True)` remove esse nível e permite alinhar os resultados às linhas originais.

A previsão é feita ao final de D, portanto os registros de D entram na janela. Os nomes deixam explícito que o cálculo representa alarmes **registrados**, cuja completude ainda não foi confirmada.

In [37]:
for dias in [7, 14]:
    historico_alarmes_diario[f"qtd_alarmes_registrados_{dias}d"] = (
        historico_alarmes_diario.groupby("chave_candidata")["qtd_alarmes_registrados_1d"]
        .rolling(window=dias, min_periods=dias)
        .sum()
        .reset_index(level=0, drop=True)
        .astype("Int64")
    )

    historico_alarmes_diario[f"dias_com_registro_{dias}d"] = (
        historico_alarmes_diario.groupby("chave_candidata")["dia_com_registro"]
        .rolling(window=dias, min_periods=dias)
        .sum()
        .reset_index(level=0, drop=True)
        .astype("Int64")
    )

display(historico_alarmes_diario.loc[
    historico_alarmes_diario["data"].eq(DATA_INICIO)
].head())


,chave_candidata,data,qtd_alarmes_registrados_1d,dia_com_registro,qtd_alarmes_registrados_7d,dias_com_registro_7d,qtd_alarmes_registrados_14d,dias_com_registro_14d
13,BUT|1|10|PDM,2023-10-17,0,0,0,0,0,0
920,BUT|1|11|PDM,2023-10-17,0,0,0,0,0,0
1827,BUT|1|12|PDM,2023-10-17,0,0,0,0,0,0
2734,BUT|1|13|PDM,2023-10-17,0,0,0,0,0,0
3641,BUT|1|14|PDM,2023-10-17,0,0,0,0,0,0


## 35. Comparação entre duas semanas consecutivas

A diferença entre a soma de 14 dias e a soma de 7 dias fornece a semana anterior: D−13 até D−7. A variação compara essa semana com D−6 até D.

Por exemplo, 12 ativações na semana atual e 8 na anterior produzem variação +4. Uma variação negativa indica redução de registros. O cálculo usa diferença absoluta, sem divisão.

A coluna `variacao_alarmes_registrados_7d` é a definição inicial proposta para a ideia de aceleração de alarmes. Ela pode mudar tanto por alteração nas ocorrências quanto por alteração na coleta. As colunas de dias com registro ajudam na investigação, mas não corrigem uma coleta incompleta.

In [38]:
historico_alarmes_diario["qtd_alarmes_registrados_7d_anteriores"] = (
    historico_alarmes_diario["qtd_alarmes_registrados_14d"]
    - historico_alarmes_diario["qtd_alarmes_registrados_7d"]
)

historico_alarmes_diario["variacao_alarmes_registrados_7d"] = (
    historico_alarmes_diario["qtd_alarmes_registrados_7d"]
    - historico_alarmes_diario["qtd_alarmes_registrados_7d_anteriores"]
)

colunas_janelas = [
    "qtd_alarmes_registrados_1d",
    "qtd_alarmes_registrados_7d",
    "qtd_alarmes_registrados_14d",
    "qtd_alarmes_registrados_7d_anteriores",
    "variacao_alarmes_registrados_7d",
    "dias_com_registro_7d",
    "dias_com_registro_14d",
]

janelas_alarmes = historico_alarmes_diario.loc[
    historico_alarmes_diario["data"].between(DATA_INICIO, DATA_FIM),
    ["chave_candidata", "data", *colunas_janelas],
].copy()

assert janelas_alarmes[colunas_janelas].notna().all().all()
display(janelas_alarmes.head(10))


,chave_candidata,data,qtd_alarmes_registrados_1d,qtd_alarmes_registrados_7d,qtd_alarmes_registrados_14d,qtd_alarmes_registrados_7d_anteriores,variacao_alarmes_registrados_7d,dias_com_registro_7d,dias_com_registro_14d
13,BUT|1|10|PDM,2023-10-17,0,0,0,0,0,0,0
14,BUT|1|10|PDM,2023-10-18,0,0,0,0,0,0,0
15,BUT|1|10|PDM,2023-10-19,0,0,0,0,0,0,0
16,BUT|1|10|PDM,2023-10-20,0,0,0,0,0,0,0
17,BUT|1|10|PDM,2023-10-21,0,0,0,0,0,0,0
18,BUT|1|10|PDM,2023-10-22,0,0,0,0,0,0,0
19,BUT|1|10|PDM,2023-10-23,0,0,0,0,0,0,0
20,BUT|1|10|PDM,2023-10-24,0,0,0,0,0,0,0
21,BUT|1|10|PDM,2023-10-25,0,0,0,0,0,0,0
22,BUT|1|10|PDM,2023-10-26,0,0,0,0,0,0,0


## 36. Inclusão das janelas na tabela mestra

A junção acrescenta as sete colunas à tabela histórica, por chave e data. O calendário principal permanece no mesmo período. Os 13 dias auxiliares servem somente ao cálculo das janelas e não acrescentam linhas à tabela final.

In [39]:
tabela_mestra_janelas = tabela_mestra_historico.merge(
    janelas_alarmes,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)

assert len(tabela_mestra_janelas) == len(tabela_mestra_historico)
assert not tabela_mestra_janelas.duplicated(["chave_candidata", "data"]).any()
assert tabela_mestra_janelas["registro_zm01_7d"].equals(tabela_mestra_historico["registro_zm01_7d"])
assert tabela_mestra_janelas["qtd_registros_alarmes"].equals(
    tabela_mestra_historico["qtd_registros_alarmes"]
)
assert tabela_mestra_janelas[colunas_janelas].notna().all().all()

print("Linhas da tabela:", len(tabela_mestra_janelas))
print("Novas colunas:", len(colunas_janelas))
display(tabela_mestra_janelas[colunas_janelas].describe())


Linhas da tabela: 472032
Novas colunas: 7


,qtd_alarmes_registrados_1d,qtd_alarmes_registrados_7d,qtd_alarmes_registrados_14d,qtd_alarmes_registrados_7d_anteriores,variacao_alarmes_registrados_7d,dias_com_registro_7d,dias_com_registro_14d
count,472032.000000,472032.0,472032.0,472032.0,472032.0,472032.0,472032.0
mean,0.641747,4.489842,8.970616,4.480775,0.009067,0.703266,1.401966
std,4.575306,21.599751,37.197844,21.599121,21.965039,1.320716,2.406354
min,0.000000,0.0,0.0,0.0,-1854.0,0.0,0.0
25%,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
50%,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
75%,0.000000,4.0,8.0,4.0,0.0,1.0,2.0
max,375.000000,1956.0,2501.0,1956.0,1744.0,7.0,14.0


## 37. Conferência das janelas por contagem direta

A conferência seleciona o início e o fim do calendário e uma linha com a maior contagem diária. Para cada caso, o código conta diretamente as ativações da mesma porta dentro do intervalo de datas, sem usar `rolling()`.

A soma de 14 dias deve ser igual à soma da semana atual com a semana anterior. Os dias com registros devem permanecer entre zero e o tamanho da janela.

In [40]:
assert (
    tabela_mestra_janelas["qtd_alarmes_registrados_14d"]
    == tabela_mestra_janelas["qtd_alarmes_registrados_7d"]
    + tabela_mestra_janelas["qtd_alarmes_registrados_7d_anteriores"]
).all()

assert tabela_mestra_janelas["dias_com_registro_7d"].between(0, 7).all()
assert tabela_mestra_janelas["dias_com_registro_14d"].between(0, 14).all()
assert tabela_mestra_janelas["qtd_alarmes_registrados_7d_anteriores"].ge(0).all()

indices_exemplo = [
    tabela_mestra_janelas.index[0],
    tabela_mestra_janelas.index[-1],
    tabela_mestra_janelas["qtd_alarmes_registrados_1d"].idxmax(),
]
amostras_janelas = tabela_mestra_janelas.loc[indices_exemplo].drop_duplicates(
    ["chave_candidata", "data"]
)

for _, linha in amostras_janelas.iterrows():
    registros_da_porta = ativacoes_alarmes.loc[
        ativacoes_alarmes["chave_candidata"].eq(linha["chave_candidata"])
    ]

    for dias in [1, 7, 14]:
        inicio_janela = linha["data"] - pd.Timedelta(days=dias - 1)
        quantidade_direta = registros_da_porta["data_alarme"].between(
            inicio_janela, linha["data"]
        ).sum()
        assert quantidade_direta == linha[f"qtd_alarmes_registrados_{dias}d"]

display(amostras_janelas[["chave_candidata", "data", *colunas_janelas]])
print("Contagens móveis conferidas com as ativações de origem.")


,chave_candidata,data,qtd_alarmes_registrados_1d,qtd_alarmes_registrados_7d,qtd_alarmes_registrados_14d,qtd_alarmes_registrados_7d_anteriores,variacao_alarmes_registrados_7d,dias_com_registro_7d,dias_com_registro_14d
0,BUT|1|10|PDM,2023-10-17,0,0,0,0,0,0,0
472031,VSO|2|9|PDM,2026-03-28,0,0,0,0,0,0,0
208046,HIG|2|2|PDM,2025-07-16,375,388,388,0,388,2,2


Contagens móveis conferidas com as ativações de origem.


### Resultado das janelas de alarmes

A variável `tabela_mestra_janelas` acrescenta contagens de ativações registradas em 1, 7 e 14 dias, a semana anterior, a variação semanal e os dias com registros. As contagens de linhas da fonte e os alvos parciais anteriores permanecem intactos.

A unidade de contagem agora é TAG + início do alarme. A ambiguidade dos horários de normalização ainda precisa de tratamento antes das variáveis de duração e anomalias de 15/30 segundos.

As novas contagens são candidatas a entradas temporais. Zero significa nenhuma ativação encontrada no arquivo; a interpretação operacional exige confirmação de cobertura. `dias_com_registro` não é uma certificação de coleta completa. O alvo continua sem negativos definidos, portanto o treinamento ainda não pode começar.

A próxima etapa poderá separar as contagens por tipo de alarme, como FO e LSD, mantendo a mesma unidade de ativação e as mesmas regras temporais.

## 38. Classificação das ativações por tipo de evento

Esta etapa prepara as variáveis `qtd_fo_7d` e `qtd_lsd_7d`, contando ativações registradas dos códigos FO e LSD. A unidade continua sendo **TAG + início do alarme**.

Parte dos registros de MBI está sem classificação preenchida no arquivo processado. Nesses casos, o código do evento pode ser recuperado da TAG. Por exemplo, `PSD_P_MBI1_PDM10FO` termina com FO. O padrão também aceita TAGs com separador, como `PSD_BUT_BUT1_PDM10_FO`.

A classificação preenchida é comparada à TAG quando ambas estão disponíveis. Uma divergência interrompe a execução para revisão. Os códigos representam tipos de evento, sem classificar automaticamente FO ou LSD como falha ou alarme grave.

In [41]:
mapa_tipos = alarmes_identificados.loc[
    alarmes_identificados["tag"].isin(ativacoes_alarmes["tag"]),
    ["tag", "tipo_evento"],
].drop_duplicates().copy()

mapa_tipos = mapa_tipos.rename(columns={"tipo_evento": "tipo_evento_processado"})
mapa_tipos["tipo_evento_processado"] = (
    mapa_tipos["tipo_evento_processado"].astype("string").str.strip().str.upper()
    .replace("", pd.NA)
)

# A extração lê somente o código após o número de PDM ou MDU.
mapa_tipos["tipo_evento_tag"] = (
    mapa_tipos["tag"].astype("string").str.strip().str.upper()
    .str.extract(r"_(?:PDM|MDU)\d+_?([A-Z][A-Z0-9]*)$", expand=False)
)

tem_duas_classificacoes = (
    mapa_tipos["tipo_evento_processado"].notna()
    & mapa_tipos["tipo_evento_tag"].notna()
)
assert (
    mapa_tipos.loc[tem_duas_classificacoes, "tipo_evento_processado"]
    == mapa_tipos.loc[tem_duas_classificacoes, "tipo_evento_tag"]
).all(), "Há divergência entre a classificação processada e a TAG."

mapa_tipos["tipo_evento"] = mapa_tipos["tipo_evento_processado"].fillna(
    mapa_tipos["tipo_evento_tag"]
)
mapa_tipos["classificacao_recuperada"] = (
    mapa_tipos["tipo_evento_processado"].isna() & mapa_tipos["tipo_evento"].notna()
)
assert not mapa_tipos["tag"].duplicated().any(), "Há mais de uma classificação por TAG."

ativacoes_classificadas = ativacoes_alarmes.merge(
    mapa_tipos[["tag", "tipo_evento", "classificacao_recuperada"]],
    on="tag",
    how="left",
    validate="many_to_one",
)
assert len(ativacoes_classificadas) == len(ativacoes_alarmes)
assert not ativacoes_classificadas.duplicated(["tag", "inicio_alarme"]).any()
assert ativacoes_classificadas["tipo_evento"].notna().all(), "Há ativações sem classificação."

print("Ativações com classificação recuperada da TAG:",
      ativacoes_classificadas["classificacao_recuperada"].sum())
display(ativacoes_classificadas["tipo_evento"].value_counts().to_frame("ativacoes"))
display(mapa_tipos.loc[
    mapa_tipos["classificacao_recuperada"],
    ["tag", "tipo_evento_processado", "tipo_evento"],
].head())


Ativações com classificação recuperada da TAG: 29583


,ativacoes
tipo_evento,
LSD,65135
OBS,56797
FC,55842
FO,55241
OA,38296
CA,16252
CSD,13314
RCE,3266


,tag,tipo_evento_processado,tipo_evento
241947,PSD_P_MBI1_PDM10FC,<NA>,FC
241949,PSD_P_MBI1_PDM10RCE,<NA>,RCE
241951,PSD_P_MBI1_PDM11FC,<NA>,FC
241954,PSD_P_MBI1_PDM11RCE,<NA>,RCE
241957,PSD_P_MBI1_PDM12CA,<NA>,CA


## 39. Contagens diárias de FO e LSD

Cada ativação recebe dois indicadores: `eh_fo` e `eh_lsd`. A soma desses indicadores fornece a quantidade diária de cada tipo. Uma ativação FO não é contada como LSD, e os demais tipos não entram nessas duas contagens.

A tabela auxiliar utiliza o mesmo calendário estendido da seção 33. Nela, zero significa nenhuma ativação **desse tipo encontrada no arquivo**. Não significa confirmação de ausência operacional do evento.

In [42]:
ativacoes_classificadas["eh_fo"] = ativacoes_classificadas["tipo_evento"].eq("FO").astype(int)
ativacoes_classificadas["eh_lsd"] = ativacoes_classificadas["tipo_evento"].eq("LSD").astype(int)

tipos_diarios = (
    ativacoes_classificadas.groupby(["chave_candidata", "data_alarme"])
    .agg(
        qtd_fo_1d=("eh_fo", "sum"),
        qtd_lsd_1d=("eh_lsd", "sum"),
    )
    .reset_index()
    .rename(columns={"data_alarme": "data"})
)

historico_tipos = historico_alarmes_diario[["chave_candidata", "data"]].merge(
    tipos_diarios,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)
historico_tipos[["qtd_fo_1d", "qtd_lsd_1d"]] = (
    historico_tipos[["qtd_fo_1d", "qtd_lsd_1d"]].fillna(0).astype(int)
)
historico_tipos = (
    historico_tipos.sort_values(["chave_candidata", "data"]).reset_index(drop=True)
)

dentro_do_historico = ativacoes_classificadas["data_alarme"].between(
    inicio_historico_alarmes, DATA_FIM
)
assert historico_tipos["qtd_fo_1d"].sum() == ativacoes_classificadas.loc[dentro_do_historico, "eh_fo"].sum()
assert historico_tipos["qtd_lsd_1d"].sum() == ativacoes_classificadas.loc[dentro_do_historico, "eh_lsd"].sum()

display(historico_tipos.loc[
    (historico_tipos["qtd_fo_1d"] > 0) | (historico_tipos["qtd_lsd_1d"] > 0)
].head(10))


,chave_candidata,data,qtd_fo_1d,qtd_lsd_1d
37,BUT|1|10|PDM,2023-11-10,0,2
38,BUT|1|10|PDM,2023-11-11,0,3
39,BUT|1|10|PDM,2023-11-12,1,0
40,BUT|1|10|PDM,2023-11-13,1,9
48,BUT|1|10|PDM,2023-11-21,0,2
51,BUT|1|10|PDM,2023-11-24,0,1
54,BUT|1|10|PDM,2023-11-27,2,0
65,BUT|1|10|PDM,2023-12-08,0,1
72,BUT|1|10|PDM,2023-12-15,1,1
88,BUT|1|10|PDM,2023-12-31,5,0


## 40. Janelas de sete dias por tipo

O cálculo utiliza os sete dias de D−6 até D, incluindo D porque a previsão será feita ao final do dia. O agrupamento mantém cada porta separada.

Os nomes `qtd_fo_7d` e `qtd_lsd_7d` seguem a lista inicial de variáveis do projeto. Seu significado nesta versão é **quantidade de ativações registradas** de cada código nos sete dias. A interpretação como total operacional depende da confirmação de cobertura.

In [43]:
for tipo in ["fo", "lsd"]:
    historico_tipos[f"qtd_{tipo}_7d"] = (
        historico_tipos.groupby("chave_candidata")[f"qtd_{tipo}_1d"]
        .rolling(window=7, min_periods=7)
        .sum()
        .reset_index(level=0, drop=True)
        .astype("Int64")
    )

colunas_tipos = ["qtd_fo_7d", "qtd_lsd_7d"]
janelas_tipos = historico_tipos.loc[
    historico_tipos["data"].between(DATA_INICIO, DATA_FIM),
    ["chave_candidata", "data", *colunas_tipos],
].copy()

assert janelas_tipos[colunas_tipos].notna().all().all()
assert janelas_tipos[colunas_tipos].ge(0).all().all()

tabela_mestra_tipos = tabela_mestra_janelas.merge(
    janelas_tipos,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)

assert len(tabela_mestra_tipos) == len(tabela_mestra_janelas)
assert not tabela_mestra_tipos.duplicated(["chave_candidata", "data"]).any()
assert tabela_mestra_tipos["registro_zm01_7d"].equals(tabela_mestra_janelas["registro_zm01_7d"])
assert tabela_mestra_tipos["qtd_alarmes_registrados_7d"].equals(
    tabela_mestra_janelas["qtd_alarmes_registrados_7d"]
)

print("Linhas da tabela:", len(tabela_mestra_tipos))
display(tabela_mestra_tipos[colunas_tipos].describe())


Linhas da tabela: 472032


,qtd_fo_7d,qtd_lsd_7d
count,472032.0,472032.0
mean,0.818565,0.964901
std,11.637091,14.830666
min,0.0,0.0
25%,0.0,0.0
50%,0.0,0.0
75%,0.0,0.0
max,988.0,1936.0


## 41. Conferência das contagens por tipo

FO e LSD são categorias distintas. A soma de suas contagens não pode ultrapassar a quantidade total de ativações registradas na mesma janela.

A conferência direta seleciona o início e o fim do calendário e as linhas de maior contagem FO e LSD. Para cada linha, a verificação conta as ativações da mesma porta entre D−6 e D, sem utilizar `rolling()`. Isso permite conferir a classificação, as datas e a separação entre portas.

In [44]:
assert (
    tabela_mestra_tipos["qtd_fo_7d"] + tabela_mestra_tipos["qtd_lsd_7d"]
    <= tabela_mestra_tipos["qtd_alarmes_registrados_7d"]
).all()

indices_tipos = [
    tabela_mestra_tipos.index[0],
    tabela_mestra_tipos.index[-1],
    tabela_mestra_tipos["qtd_fo_7d"].idxmax(),
    tabela_mestra_tipos["qtd_lsd_7d"].idxmax(),
]
amostras_tipos = tabela_mestra_tipos.loc[indices_tipos].drop_duplicates(
    ["chave_candidata", "data"]
)

for _, linha in amostras_tipos.iterrows():
    inicio_janela = linha["data"] - pd.Timedelta(days=6)
    ativacoes_da_janela = ativacoes_classificadas.loc[
        ativacoes_classificadas["chave_candidata"].eq(linha["chave_candidata"])
        & ativacoes_classificadas["data_alarme"].between(inicio_janela, linha["data"])
    ]

    for codigo in ["FO", "LSD"]:
        contagem_direta = ativacoes_da_janela["tipo_evento"].eq(codigo).sum()
        assert contagem_direta == linha[f"qtd_{codigo.lower()}_7d"]

display(amostras_tipos[[
    "chave_candidata", "data", "qtd_fo_7d", "qtd_lsd_7d",
    "qtd_alarmes_registrados_7d",
]])
print("Contagens FO e LSD conferidas com as ativações de origem.")


,chave_candidata,data,qtd_fo_7d,qtd_lsd_7d,qtd_alarmes_registrados_7d
0,BUT|1|10|PDM,2023-10-17,0,0,0
472031,VSO|2|9|PDM,2026-03-28,0,0,0
438550,VSO|1|1|PDM,2025-02-18,988,0,988
455432,VSO|2|15|PDM,2024-11-06,0,1936,1956


Contagens FO e LSD conferidas com as ativações de origem.


### Resultado dos históricos FO e LSD

A variável `tabela_mestra_tipos` acrescenta `qtd_fo_7d` e `qtd_lsd_7d`, preservando as contagens totais e o alvo parcial. O código recupera classificações ausentes a partir das TAGs e verifica sua consistência com os campos preenchidos.

As contagens continuam baseadas em ativações registradas, e os zeros não confirmam ausência de eventos na operação. Nenhum arquivo de dados é exportado e o treinamento permanece pendente dos rótulos negativos e da cobertura.

A próxima etapa deverá revisar os horários de normalização antes das variáveis de duração. Uma única ativação pode ter vários horários de normalização na fonte; selecionar um deles sem uma regra validada pode distorcer as anomalias de 15/30 segundos e as durações acumuladas.

## 42. Auditoria dos horários de normalização

Uma ativação continua sendo identificada por TAG + início. Esta seção conta quantos horários de normalização aparecem para cada ativação. As datas mínima e máxima são exibidas para auditoria, sem escolher arbitrariamente uma delas quando houver divergência.

A regra inicial admite o cálculo somente quando há **um fim distinto, nenhum fim ausente e nenhum fim anterior ao início**. Uma normalização posterior ao início da próxima ativação da mesma TAG também fica para revisão.

A classificação é feita sobre a extração disponível. Fim único significa consistência dentro desse arquivo, não comprovação da qualidade do registro nem de sua disponibilidade histórica.

In [45]:
registros_normalizacao = alarmes_identificados.loc[
    alarmes_identificados["chave_candidata"].isin(portas["chave_candidata"]),
    ["chave_candidata", "tag", "inicio_alarme", "fim_alarme"],
].copy()

registros_normalizacao["fim_ausente"] = registros_normalizacao["fim_alarme"].isna()
registros_normalizacao["fim_anterior_inicio"] = (
    registros_normalizacao["fim_alarme"] < registros_normalizacao["inicio_alarme"]
)

auditoria_duracoes = (
    registros_normalizacao.groupby(["chave_candidata", "tag", "inicio_alarme"])
    .agg(
        qtd_linhas=("fim_alarme", "size"),
        qtd_fins_distintos=("fim_alarme", "nunique"),
        primeiro_fim=("fim_alarme", "min"),
        ultimo_fim=("fim_alarme", "max"),
        tem_fim_ausente=("fim_ausente", "max"),
        tem_fim_anterior_inicio=("fim_anterior_inicio", "max"),
    )
    .reset_index()
    .sort_values(["tag", "inicio_alarme"])
    .reset_index(drop=True)
)

auditoria_duracoes["proximo_inicio"] = (
    auditoria_duracoes.groupby("tag")["inicio_alarme"].shift(-1)
)
auditoria_duracoes["fim_apos_proxima_ativacao"] = (
    auditoria_duracoes["primeiro_fim"] > auditoria_duracoes["proximo_inicio"]
)

auditoria_duracoes["fim_unico_utilizavel"] = (
    auditoria_duracoes["qtd_fins_distintos"].eq(1)
    & ~auditoria_duracoes["tem_fim_ausente"]
    & ~auditoria_duracoes["tem_fim_anterior_inicio"]
    & ~auditoria_duracoes["fim_apos_proxima_ativacao"]
)

assert len(auditoria_duracoes) == len(ativacoes_alarmes)
assert auditoria_duracoes["qtd_linhas"].sum() == len(registros_normalizacao)

duracoes_para_revisar = auditoria_duracoes.loc[
    ~auditoria_duracoes["fim_unico_utilizavel"]
].copy()

print("Ativações analisadas:", len(auditoria_duracoes))
print("Ativações com mais de um fim:", auditoria_duracoes["qtd_fins_distintos"].gt(1).sum())
print("Ativações com fim único utilizável:", auditoria_duracoes["fim_unico_utilizavel"].sum())
display(duracoes_para_revisar[[
    "tag", "inicio_alarme", "qtd_fins_distintos", "primeiro_fim", "ultimo_fim",
]].head())


Ativações analisadas: 304143
Ativações com mais de um fim: 54640
Ativações com fim único utilizável: 249503


,tag,inicio_alarme,qtd_fins_distintos,primeiro_fim,ultimo_fim
0,PSD_BUT_BUT1_PDM01_OBS,2024-01-21 23:12:46.257,33,2024-01-21 23:12:56.257,2024-02-20 18:02:26.770
2,PSD_BUT_BUT1_PDM01_OBS,2024-04-13 01:27:17.210,22,2024-04-18 01:14:28.663,2024-05-13 01:07:58.167
3,PSD_BUT_BUT1_PDM01_OBS,2024-05-23 18:15:43.987,12,2024-05-23 22:34:54.787,2024-06-03 13:29:14.897
5,PSD_BUT_BUT1_PDM01_OBS,2024-06-05 02:06:25.253,2,2024-06-05 04:53:13.300,2024-06-05 08:58:56.300
6,PSD_BUT_BUT1_PDM01_OBS,2024-06-08 11:16:31.223,2,2024-06-08 22:41:48.507,2024-06-09 01:02:46.830


## 43. Durações das ativações com fim único

A duração, em segundos, é calculada por fim − início, preservando as frações de segundo dos horários. A coluna de duração textual ou arredondada não é necessária para esse cálculo.

Nesta versão, uma anomalia de 15 segundos significa **duração maior ou igual a 15 segundos**. O mesmo critério é aplicado a 30 segundos. Esses limites são definições operacionais iniciais, não uma classificação automática de falha ou gravidade.

Durações elevadas permanecem no diagnóstico; sua magnitude não basta para classificá-las como erro. A tabela apresenta os percentis para orientar a revisão.

In [46]:
alarmes_fim_unico = auditoria_duracoes.loc[
    auditoria_duracoes["fim_unico_utilizavel"]
].copy()

# Com apenas um fim distinto, primeiro_fim e ultimo_fim são iguais.
assert alarmes_fim_unico["primeiro_fim"].equals(alarmes_fim_unico["ultimo_fim"])
alarmes_fim_unico["fim_alarme"] = alarmes_fim_unico["primeiro_fim"]
alarmes_fim_unico["duracao_seg"] = (
    alarmes_fim_unico["fim_alarme"] - alarmes_fim_unico["inicio_alarme"]
).dt.total_seconds()

alarmes_fim_unico["data_normalizacao"] = alarmes_fim_unico["fim_alarme"].dt.normalize()
alarmes_fim_unico["anomalia_15s"] = alarmes_fim_unico["duracao_seg"].ge(15).astype(int)
alarmes_fim_unico["anomalia_30s"] = alarmes_fim_unico["duracao_seg"].ge(30).astype(int)

assert alarmes_fim_unico["duracao_seg"].ge(0).all()
assert (alarmes_fim_unico["anomalia_30s"] <= alarmes_fim_unico["anomalia_15s"]).all()

display(alarmes_fim_unico["duracao_seg"].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
).to_frame("segundos"))


,segundos
count,2.495030e+05
mean,5.078661e+04
std,2.218203e+05
min,0.000000e+00
50%,8.000000e+00
90%,6.959157e+04
95%,2.751666e+05
99%,1.290313e+06
max,2.579599e+06


## 44. Agregação pelo dia da normalização

Um alarme iniciado em 10/01 e normalizado em 12/01 terá sua duração final contabilizada em **12/01**. A duração completa não poderia ser usada em uma previsão feita ao final de 10/01 ou 11/01.

As janelas desta etapa são, portanto, de **alarmes normalizados** em D−6…D ou D−13…D, e não de alarmes iniciados nesses períodos. Alarmes ainda abertos e casos ambíguos não entram nas durações calculadas.

A soma das durações pode contar intervalos sobrepostos e alarmes que não interrompem a operação. Por isso, ela recebe o nome `soma_duracoes_alarmes_7d` e **não substitui `tempo_parada_acumulado_7d`**.

Zero nas contagens e somas auxiliares significa nenhum caso elegível encontrado no arquivo. Máximos e médias sem casos permanecem ausentes.

In [47]:
duracoes_diarias = (
    alarmes_fim_unico.groupby(["chave_candidata", "data_normalizacao"])
    .agg(
        qtd_normalizados_fim_unico_1d=("duracao_seg", "size"),
        soma_duracoes_1d=("duracao_seg", "sum"),
        max_duracao_1d=("duracao_seg", "max"),
        qtd_anomalias_15s_1d=("anomalia_15s", "sum"),
        qtd_anomalias_30s_1d=("anomalia_30s", "sum"),
    )
    .reset_index()
    .rename(columns={"data_normalizacao": "data"})
)

historico_duracoes = historico_alarmes_diario[["chave_candidata", "data"]].merge(
    duracoes_diarias,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)

contagens_duracoes = [
    "qtd_normalizados_fim_unico_1d", "qtd_anomalias_15s_1d", "qtd_anomalias_30s_1d",
]
historico_duracoes[contagens_duracoes] = (
    historico_duracoes[contagens_duracoes].fillna(0).astype(int)
)
historico_duracoes["soma_duracoes_1d"] = historico_duracoes["soma_duracoes_1d"].fillna(0)

historico_duracoes = (
    historico_duracoes.sort_values(["chave_candidata", "data"]).reset_index(drop=True)
)

normalizados_no_historico = alarmes_fim_unico["data_normalizacao"].between(
    inicio_historico_alarmes, DATA_FIM
)
assert historico_duracoes["qtd_normalizados_fim_unico_1d"].sum() == normalizados_no_historico.sum()
display(historico_duracoes.loc[
    historico_duracoes["qtd_normalizados_fim_unico_1d"] > 0
].head())


,chave_candidata,data,qtd_normalizados_fim_unico_1d,soma_duracoes_1d,max_duracao_1d,qtd_anomalias_15s_1d,qtd_anomalias_30s_1d
37,BUT|1|10|PDM,2023-11-10,3,2464.983,2304.983,3,2
38,BUT|1|10|PDM,2023-11-11,3,24.970,14.984,0,0
39,BUT|1|10|PDM,2023-11-12,1,5.017,5.017,0,0
40,BUT|1|10|PDM,2023-11-13,10,640.004,470.000,4,2
48,BUT|1|10|PDM,2023-11-21,2,20.000,10.000,0,0


## 45. Janelas das durações e anomalias

As somas usam a mesma técnica `groupby().rolling()` das etapas anteriores. O máximo ignora dias sem casos elegíveis e permanece ausente quando toda a semana está sem casos.

A média é **soma das durações ÷ quantidade de alarmes normalizados elegíveis** na semana. Essa divisão evita dar o mesmo peso a um dia com um alarme e a outro com muitos alarmes.

Todas as variáveis desta seção usam apenas o subconjunto com fim único. A filtragem de qualidade foi feita na extração completa; o uso no modelo ainda exige confirmar quando os registros e suas revisões ficaram disponíveis.

In [48]:
# Cada linha define: coluna diária, tamanho da janela e nome do resultado.
calculos_duracao = [
    ("qtd_normalizados_fim_unico_1d", 7, "qtd_normalizados_fim_unico_7d"),
    ("qtd_normalizados_fim_unico_1d", 14, "qtd_normalizados_fim_unico_14d"),
    ("soma_duracoes_1d", 7, "soma_duracoes_alarmes_7d"),
    ("qtd_anomalias_15s_1d", 7, "qtd_anomalias_15s_7d"),
    ("qtd_anomalias_30s_1d", 14, "qtd_anomalias_30s_14d"),
]

for coluna_diaria, dias, coluna_resultado in calculos_duracao:
    historico_duracoes[coluna_resultado] = (
        historico_duracoes.groupby("chave_candidata")[coluna_diaria]
        .rolling(window=dias, min_periods=dias)
        .sum()
        .reset_index(level=0, drop=True)
    )

historico_duracoes["max_tempo_normalizacao_7d"] = (
    historico_duracoes.groupby("chave_candidata")["max_duracao_1d"]
    .rolling(window=7, min_periods=1)
    .max()
    .reset_index(level=0, drop=True)
)

quantidade_para_media = historico_duracoes["qtd_normalizados_fim_unico_7d"].where(
    historico_duracoes["qtd_normalizados_fim_unico_7d"] > 0
)
historico_duracoes["tempo_medio_por_alarme_7d"] = (
    historico_duracoes["soma_duracoes_alarmes_7d"] / quantidade_para_media
)

colunas_duracao = [
    "qtd_normalizados_fim_unico_7d", "qtd_normalizados_fim_unico_14d",
    "soma_duracoes_alarmes_7d", "qtd_anomalias_15s_7d", "qtd_anomalias_30s_14d",
    "max_tempo_normalizacao_7d", "tempo_medio_por_alarme_7d",
]

janelas_duracoes = historico_duracoes.loc[
    historico_duracoes["data"].between(DATA_INICIO, DATA_FIM),
    ["chave_candidata", "data", *colunas_duracao],
].copy()

for coluna in [
    "qtd_normalizados_fim_unico_7d", "qtd_normalizados_fim_unico_14d",
    "qtd_anomalias_15s_7d", "qtd_anomalias_30s_14d",
]:
    janelas_duracoes[coluna] = janelas_duracoes[coluna].astype("Int64")

tabela_mestra_duracoes = tabela_mestra_tipos.merge(
    janelas_duracoes,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)

assert len(tabela_mestra_duracoes) == len(tabela_mestra_tipos)
assert not tabela_mestra_duracoes.duplicated(["chave_candidata", "data"]).any()
assert tabela_mestra_duracoes["registro_zm01_7d"].equals(tabela_mestra_tipos["registro_zm01_7d"])

print("Linhas da tabela:", len(tabela_mestra_duracoes))
display(tabela_mestra_duracoes[colunas_duracao].describe())


Linhas da tabela: 472032


,qtd_normalizados_fim_unico_7d,qtd_normalizados_fim_unico_14d,soma_duracoes_alarmes_7d,qtd_anomalias_15s_7d,qtd_anomalias_30s_14d,max_tempo_normalizacao_7d,tempo_medio_por_alarme_7d
count,472032.0,472032.0,4.720320e+05,472032.0,472032.0,1.382750e+05,1.382750e+05
mean,3.680212,7.352254,1.879074e+05,1.282947,1.847362,1.493221e+05,1.130610e+05
std,21.056035,36.299365,1.001635e+06,7.124325,7.290014,3.551900e+05,3.197673e+05
min,0.0,0.0,0.000000e+00,0.0,0.0,0.000000e+00,0.000000e+00
25%,0.0,0.0,0.000000e+00,0.0,0.0,5.016000e+00,5.000000e+00
50%,0.0,0.0,0.000000e+00,0.0,0.0,5.198400e+01,2.000000e+01
75%,2.0,5.0,9.972000e+00,0.0,0.0,1.189566e+05,4.214516e+04
max,1956.0,2507.0,1.805719e+07,723.0,502.0,2.579599e+06,2.579599e+06


## 46. Conferência por contagem e cálculo direto

A conferência usa o início e o fim do calendário e o dia com maior soma semanal. Para cada exemplo, o código seleciona diretamente os alarmes cuja normalização ocorreu no intervalo da janela.

O resultado das anomalias deve ser menor ou igual à quantidade de casos elegíveis na respectiva janela. Quando não há normalizações elegíveis na semana, média e máximo devem permanecer ausentes.

In [49]:
from math import isclose

assert (
    tabela_mestra_duracoes["qtd_anomalias_15s_7d"]
    <= tabela_mestra_duracoes["qtd_normalizados_fim_unico_7d"]
).all()
assert (
    tabela_mestra_duracoes["qtd_anomalias_30s_14d"]
    <= tabela_mestra_duracoes["qtd_normalizados_fim_unico_14d"]
).all()

sem_normalizacoes = tabela_mestra_duracoes["qtd_normalizados_fim_unico_7d"].eq(0)
assert tabela_mestra_duracoes.loc[
    sem_normalizacoes, ["max_tempo_normalizacao_7d", "tempo_medio_por_alarme_7d"]
].isna().all().all()

indices_duracao = [
    tabela_mestra_duracoes.index[0],
    tabela_mestra_duracoes.index[-1],
    tabela_mestra_duracoes["soma_duracoes_alarmes_7d"].idxmax(),
]
amostras_duracao = tabela_mestra_duracoes.loc[indices_duracao].drop_duplicates(
    ["chave_candidata", "data"]
)

for _, linha in amostras_duracao.iterrows():
    mesma_porta = alarmes_fim_unico["chave_candidata"].eq(linha["chave_candidata"])
    semana = alarmes_fim_unico.loc[
        mesma_porta & alarmes_fim_unico["data_normalizacao"].between(
            linha["data"] - pd.Timedelta(days=6), linha["data"]
        )
    ]
    quatorze_dias = alarmes_fim_unico.loc[
        mesma_porta & alarmes_fim_unico["data_normalizacao"].between(
            linha["data"] - pd.Timedelta(days=13), linha["data"]
        )
    ]

    assert len(semana) == linha["qtd_normalizados_fim_unico_7d"]
    assert len(quatorze_dias) == linha["qtd_normalizados_fim_unico_14d"]
    assert semana["duracao_seg"].ge(15).sum() == linha["qtd_anomalias_15s_7d"]
    assert quatorze_dias["duracao_seg"].ge(30).sum() == linha["qtd_anomalias_30s_14d"]
    assert isclose(semana["duracao_seg"].sum(), linha["soma_duracoes_alarmes_7d"], abs_tol=0.000001)

    # O fim de cada caso selecionado precede o início do dia seguinte à previsão.
    assert (semana["fim_alarme"] < linha["data"] + pd.Timedelta(days=1)).all()
    if len(semana) > 0:
        assert isclose(semana["duracao_seg"].mean(), linha["tempo_medio_por_alarme_7d"], abs_tol=0.000001)
        assert isclose(semana["duracao_seg"].max(), linha["max_tempo_normalizacao_7d"], abs_tol=0.000001)

display(amostras_duracao[["chave_candidata", "data", *colunas_duracao]])
print("Durações e anomalias conferidas com os casos de fim único.")


,chave_candidata,data,qtd_normalizados_fim_unico_7d,qtd_normalizados_fim_unico_14d,soma_duracoes_alarmes_7d,qtd_anomalias_15s_7d,qtd_anomalias_30s_14d,max_tempo_normalizacao_7d,tempo_medio_por_alarme_7d
0,BUT|1|10|PDM,2023-10-17,0,0,0.000000e+00,0,0,NaN,NaN
472031,VSO|2|9|PDM,2026-03-28,0,0,0.000000e+00,0,0,NaN,NaN
306823,PIN|1|17|PDM,2024-04-15,7,7,1.805719e+07,7,7,2579598.71,2.579599e+06


Durações e anomalias conferidas com os casos de fim único.


### Resultado das durações

A variável `tabela_mestra_duracoes` acrescenta as contagens de casos normalizados com fim único, as anomalias de 15/30 segundos, a soma, o máximo e a média de duração. As unidades de duração são segundos, e as janelas se referem ao **dia da normalização**.

Os casos ambíguos permanecem em `duracoes_para_revisar`. Durações elevadas continuam identificáveis em `alarmes_fim_unico`; sua validade precisa de revisão antes do treinamento. A soma de durações não equivale ao tempo de parada da porta.

O modelo ainda depende da cobertura e dos negativos do alvo. Também permanecem pendentes a confirmação da execução de ZM03, a classificação de alarmes graves e a interpretação operacional do tempo de parada. As variáveis produzidas até aqui são candidatas, sem treinamento ou exportação de dados.

## 47. Referência experimental — presença de registro ZM01

Esta versão experimental prevê **a presença de uma ZM01 individual identificada entre D+1 e D+7**. O campo `registro_zm01_7d` segue essa definição operacional:

- **1:** existe pelo menos uma ocorrência ZM01 elegível nos sete dias seguintes.
- **0:** nenhuma ocorrência individual foi encontrada nesse intervalo na extração utilizada, sem ocorrência coletiva que torne a atribuição incerta.
- **Ausente:** o horizonte ultrapassa o recorte ou existe uma ocorrência coletiva sem evidência individual suficiente.

A cobertura completa não foi confirmada. Portanto, zero não comprova ausência de falha física. Os resultados futuros serão uma avaliação experimental da previsão de **registros ZM01**, condicionada à qualidade da extração. A coluna não deve ser apresentada como verdade operacional validada.

Os 103 registros separados pela duração são reintegrados à fonte do alvo: duração elevada não é motivo suficiente para excluir uma ocorrência emergencial. A elegibilidade utiliza PDM, identificação completa, data de início da avaria e pelo menos nota ou ordem. O alvo é por porta/dia; várias ordens no mesmo dia não geram múltiplos positivos.

A regra assume, para esta primeira versão retrospectiva, que os horários de ocorrência são referências utilizáveis. A disponibilidade histórica de registros e revisões continua sendo uma hipótese a validar antes do uso operacional.

In [50]:
DIAS_HISTORICO = 30
DIAS_ALVO = 7

configuracao_modelagem = {
    "versao": "experimental",
    "alvo_proxy": "registro de ZM01 individual nos próximos 7 dias",
    "alvo_final_validado": False,
    "cobertura_confirmada": False,
    "negativo": "ausência de registro individual na extração, sem ambiguidade coletiva",
    "outliers_zm01": "reintegrados; duração não é critério de exclusão de ocorrência",
    "janela_historico_dias": DIAS_HISTORICO,
    "horizonte_dias": DIAS_ALVO,
    "instante_previsao": "final do dia D",
    "disponibilidade_historica": "hipótese não confirmada: dados disponíveis pela data de ocorrência",
}

fonte_zm01_alvo = pd.concat([
    bases["ZM01"].assign(aba_origem="ZM01_limpa"),
    auditorias[("ZM01", "Outliers_removidos")].assign(aba_origem="Outliers_removidos"),
], ignore_index=True)

identificacao_zm01_alvo = extrair_ativos(fonte_zm01_alvo["local_de_instalacao"])
fonte_zm01_alvo = fonte_zm01_alvo.join(identificacao_zm01_alvo[colunas_identificacao])
fonte_zm01_alvo["data_avaria"] = pd.to_datetime(
    fonte_zm01_alvo["inicio_avaria_datetime"], errors="coerce"
).dt.normalize()

# A conferência evita resolver automaticamente ordens com versões conflitantes.
variacoes_zm01_alvo = fonte_zm01_alvo.groupby("ordem")[[
    "local_de_instalacao", "inicio_avaria_datetime"
]].nunique(dropna=False)
assert not variacoes_zm01_alvo.gt(1).any().any(), "Há ordens ZM01 divergentes para revisão."
assert fonte_zm01_alvo["data_avaria"].notna().all(), "Há ZM01 sem data de início."
assert fonte_zm01_alvo["estacao"].notna().all(), "Há ZM01 sem estação identificável."
assert (fonte_zm01_alvo["nota"].notna() | fonte_zm01_alvo["ordem"].notna()).all()

eventos_zm01_individuais = fonte_zm01_alvo.loc[
    fonte_zm01_alvo["tipo_ativo"].eq(TIPO_ATIVO)
    & fonte_zm01_alvo["chave_candidata"].notna()
    & fonte_zm01_alvo["data_avaria"].le(DATA_FIM),
    ["chave_candidata", "data_avaria"],
].drop_duplicates()

eventos_zm01_no_recorte = eventos_zm01_individuais.loc[
    eventos_zm01_individuais["data_avaria"].between(DATA_INICIO, DATA_FIM)
].copy()
assert eventos_zm01_no_recorte["chave_candidata"].isin(portas["chave_candidata"]).all()

print("Registros ZM01 reunidos:", len(fonte_zm01_alvo))
print("Dias/porta com ZM01 no recorte:", len(eventos_zm01_no_recorte))
display(pd.Series(configuracao_modelagem, name="definicao").to_frame())


Registros ZM01 reunidos: 10646
Dias/porta com ZM01 no recorte: 6261


,definicao
versao,experimental
alvo_proxy,registro de ZM01 individual nos próximos 7 dias
alvo_final_validado,False
cobertura_confirmada,False
negativo,"ausência de registro individual na extração, sem ambiguidade coletiva"
outliers_zm01,reintegrados; duração não é critério de exclusão de ocorrência
janela_historico_dias,30
horizonte_dias,7
instante_previsao,final do dia D
disponibilidade_historica,hipótese não confirmada: dados disponíveis pela data de ocorrência


## 48. Formação dos positivos e identificação das ocorrências coletivas

Cada dia com ZM01 gera até sete datas de referência anteriores. Uma ocorrência em 10/01 gera positivos de 03/01 a 09/01, sempre para a mesma porta.

Registros sem identificação individual completa são tratados como possíveis ocorrências coletivas. Uma plataforma identificada afeta apenas a revisão das portas daquela plataforma; sem plataforma, a revisão ocorre no nível da estação. Outros tipos de ativo com identificação individual completa não são convertidos em falhas de PDM.

O bloqueio de negativos coletivos é conservador e não distribui uma falha entre portas. Havendo uma ZM01 individual, o positivo é mantido.

In [51]:
antecedencias_alvo = pd.DataFrame({"dias_antes": range(1, DIAS_ALVO + 1)})

referencias_zm01 = eventos_zm01_no_recorte.merge(antecedencias_alvo, how="cross")
referencias_zm01["data"] = (
    referencias_zm01["data_avaria"]
    - pd.to_timedelta(referencias_zm01["dias_antes"], unit="D")
)
positivos_zm01_final = referencias_zm01[["chave_candidata", "data"]].drop_duplicates()
positivos_zm01_final["registro_zm01_7d"] = 1

coletivas_zm01 = fonte_zm01_alvo.loc[
    fonte_zm01_alvo["chave_candidata"].isna()
    & fonte_zm01_alvo["data_avaria"].between(DATA_INICIO, DATA_FIM),
    ["estacao", "plataforma", "data_avaria"],
].drop_duplicates()

referencias_coletivas = coletivas_zm01.merge(antecedencias_alvo, how="cross")
referencias_coletivas["data"] = (
    referencias_coletivas["data_avaria"]
    - pd.to_timedelta(referencias_coletivas["dias_antes"], unit="D")
)

coletivas_estacao = referencias_coletivas.loc[
    referencias_coletivas["plataforma"].isna(), ["estacao", "data"]
].drop_duplicates()
coletivas_estacao["zm01_coletiva_estacao_7d"] = True

coletivas_plataforma = referencias_coletivas.loc[
    referencias_coletivas["plataforma"].notna(), ["estacao", "plataforma", "data"]
].drop_duplicates()
coletivas_plataforma["zm01_coletiva_plataforma_7d"] = True

print("Datas de referência positivas, antes do corte do calendário:", len(positivos_zm01_final))
print("Ocorrências coletivas distintas para revisão:", len(coletivas_zm01))


Datas de referência positivas, antes do corte do calendário: 37207
Ocorrências coletivas distintas para revisão: 837


## 49. Tabela experimental com rótulos e exclusões explícitas

A tabela é reconstruída a partir da identificação e das contagens de ativações. As antigas prévias de alvo não são reutilizadas. Os campos de ocorrência coletiva e `status_rotulo` servem somente para auditoria e seleção das amostras; não serão entradas do modelo.

O recorte continua terminando em `DATA_FIM`. Os últimos sete dias não recebem rótulo supervisionado, mesmo quando houver algum registro futuro parcial.

In [52]:
colunas_identificacao_modelo = [
    "chave_candidata", "data", "estacao", "plataforma", "porta", "tipo_ativo",
]
features_alarmes_modelo = [
    "qtd_alarmes_registrados_1d",
    "qtd_alarmes_registrados_7d",
    "qtd_alarmes_registrados_14d",
    "variacao_alarmes_registrados_7d",
    "qtd_fo_7d", "qtd_lsd_7d",
    "dias_com_registro_7d", "dias_com_registro_14d",
]

base_modelagem_proxy = tabela_mestra_tipos[
    colunas_identificacao_modelo + features_alarmes_modelo
].copy()

base_modelagem_proxy = base_modelagem_proxy.merge(
    positivos_zm01_final, on=["chave_candidata", "data"], how="left", validate="one_to_one"
)
base_modelagem_proxy = base_modelagem_proxy.merge(
    coletivas_estacao, on=["estacao", "data"], how="left", validate="many_to_one"
)
base_modelagem_proxy = base_modelagem_proxy.merge(
    coletivas_plataforma, on=["estacao", "plataforma", "data"],
    how="left", validate="many_to_one"
)

base_modelagem_proxy["registro_zm01_7d"] = base_modelagem_proxy["registro_zm01_7d"].fillna(0).astype("Int64")
base_modelagem_proxy["status_rotulo"] = "negativo operacional: sem ZM01 individual encontrada"
base_modelagem_proxy.loc[base_modelagem_proxy["registro_zm01_7d"].eq(1), "status_rotulo"] = "positivo ZM01"

possivel_coletiva = base_modelagem_proxy[[
    "zm01_coletiva_estacao_7d", "zm01_coletiva_plataforma_7d"
]].notna().any(axis=1)
negativo_ambiguo = base_modelagem_proxy["registro_zm01_7d"].eq(0) & possivel_coletiva
base_modelagem_proxy.loc[negativo_ambiguo, "registro_zm01_7d"] = pd.NA
base_modelagem_proxy.loc[negativo_ambiguo, "status_rotulo"] = "excluído: ocorrência coletiva"

horizonte_incompleto = base_modelagem_proxy["data"] + pd.Timedelta(days=DIAS_ALVO) > DATA_FIM
base_modelagem_proxy.loc[horizonte_incompleto, "registro_zm01_7d"] = pd.NA
base_modelagem_proxy.loc[horizonte_incompleto, "status_rotulo"] = "excluído: horizonte incompleto"

base_modelagem_proxy = base_modelagem_proxy.sort_values(["chave_candidata", "data"]).reset_index(drop=True)
assert len(base_modelagem_proxy) == len(tabela_mestra_tipos)
assert not base_modelagem_proxy.duplicated(["chave_candidata", "data"]).any()
assert base_modelagem_proxy["registro_zm01_7d"].dropna().isin([0, 1]).all()
assert base_modelagem_proxy.loc[base_modelagem_proxy["data"] + pd.Timedelta(days=DIAS_ALVO) > DATA_FIM, "registro_zm01_7d"].isna().all()

display(base_modelagem_proxy["status_rotulo"].value_counts().to_frame("linhas"))


,linhas
status_rotulo,
negativo operacional: sem ZM01 individual encontrada,324743
excluído: ocorrência coletiva,106713
positivo ZM01,36880
excluído: horizonte incompleto,3696


## 50. Etapa 2 — seleção das variáveis da primeira versão

A primeira versão utiliza dez variáveis numéricas: oito contagens/indicadores de registros de alarmes e dois históricos de início de avaria. Dois indicadores binários adicionais registram ausência de histórico ZM01/ZM02. Estação e plataforma entram como informações estáticas categóricas.

As variáveis de duração ficam fora desta versão porque ainda dependem da associação correta entre início e normalização, da avaliação de durações elevadas e da disponibilidade histórica das revisões. ZM03, gravidade e tempo de parada também ficam fora por falta de definição ou evidência suficiente. Essa escolha fecha um conjunto inicial menor, sem inventar informações para completar a lista original.

Os nomes das contagens de alarmes continuam indicando registros encontrados. A classificação e a unidade TAG + início permanecem conforme as seções anteriores. Os históricos de ZM01/ZM02 representam início de avaria registrado, não execução da manutenção.

In [53]:
features_numericas = features_alarmes_modelo + [
    "dias_desde_ultima_zm01", "dias_desde_ultima_zm02",
]
features_indicadores = ["sem_historico_zm01", "sem_historico_zm02"]
features_estaticas = ["estacao", "plataforma"]

decisoes_variaveis = pd.DataFrame([
    ["Alarmes registrados e FO/LSD", "incluídos", "Contagens por TAG + início; cobertura não confirmada"],
    ["Históricos ZM01/ZM02", "incluídos", "Dias desde o início da avaria registrada"],
    ["Durações e anomalias de 15/30 segundos", "excluídas nesta versão", "Associação de normalizações e valores elevados pendentes"],
    ["Histórico de ZM03 executada", "excluído nesta versão", "Programação não comprova execução"],
    ["Gravidade dos alarmes", "excluída nesta versão", "Classificação operacional não definida"],
    ["Tempo de parada", "excluído nesta versão", "Soma de durações de alarmes não equivale a parada"],
], columns=["grupo", "decisao", "motivo"])

display(decisoes_variaveis)


,grupo,decisao,motivo
0,Alarmes registrados e FO/LSD,incluídos,Contagens por TAG + início; cobertura não confirmada
1,Históricos ZM01/ZM02,incluídos,Dias desde o início da avaria registrada
2,Durações e anomalias de 15/30 segundos,excluídas nesta versão,Associação de normalizações e valores elevados pendentes
3,Histórico de ZM03 executada,excluído nesta versão,Programação não comprova execução
4,Gravidade dos alarmes,excluída nesta versão,Classificação operacional não definida
5,Tempo de parada,excluído nesta versão,Soma de durações de alarmes não equivale a parada


### Atualização dos históricos com a regra consolidada

O histórico ZM01 é recalculado com a fonte reintegrada. Em ZM02, nota ou ordem identificam o registro; a falta do número de ordem, sozinha, não descarta uma nota identificada.

O cálculo mantém a data do último evento elegível até D e aproveita o histórico anterior ao calendário. A coluna auxiliar de data permite verificar que nenhum evento futuro entrou na diferença. Os indicadores de ausência serão preservados mesmo após a imputação.

In [54]:
eventos_zm02_individuais = zm02.loc[
    zm02["tipo_ativo"].eq(TIPO_ATIVO)
    & zm02["chave_candidata"].isin(portas["chave_candidata"])
    & (zm02["ordem"].notna() | zm02["nota"].notna())
    & zm02["data_avaria"].le(DATA_FIM),
    ["chave_candidata", "data_avaria"],
].drop_duplicates()

fontes_historico_final = {
    "zm01": eventos_zm01_individuais,
    "zm02": eventos_zm02_individuais,
}

for tipo, eventos in fontes_historico_final.items():
    anteriores = eventos.loc[eventos["data_avaria"] < DATA_INICIO]
    ultima_antes = anteriores.groupby("chave_candidata")["data_avaria"].max()

    datas_eventos = eventos.loc[
        eventos["data_avaria"].between(DATA_INICIO, DATA_FIM)
    ].rename(columns={"data_avaria": "data"}).copy()

    coluna_data = f"data_ultima_{tipo}"
    datas_eventos[coluna_data] = datas_eventos["data"]
    base_modelagem_proxy = base_modelagem_proxy.merge(
        datas_eventos, on=["chave_candidata", "data"], how="left", validate="one_to_one"
    )
    base_modelagem_proxy[coluna_data] = (
        base_modelagem_proxy.groupby("chave_candidata")[coluna_data].ffill()
    )
    base_modelagem_proxy[coluna_data] = base_modelagem_proxy[coluna_data].fillna(
        base_modelagem_proxy["chave_candidata"].map(ultima_antes)
    )
    base_modelagem_proxy[f"dias_desde_ultima_{tipo}"] = (
        base_modelagem_proxy["data"] - base_modelagem_proxy[coluna_data]
    ).dt.days.astype("Int64")
    base_modelagem_proxy[f"sem_historico_{tipo}"] = base_modelagem_proxy[coluna_data].isna().astype("int8")

    tem_data = base_modelagem_proxy[coluna_data].notna()
    assert (base_modelagem_proxy.loc[tem_data, coluna_data] <= base_modelagem_proxy.loc[tem_data, "data"]).all()
    assert base_modelagem_proxy[f"dias_desde_ultima_{tipo}"].dropna().ge(0).all()

assert base_modelagem_proxy[features_alarmes_modelo].notna().all().all()
assert "registro_zm01_7d" not in features_numericas + features_indicadores + features_estaticas
display(base_modelagem_proxy[features_numericas + features_indicadores].head())


,qtd_alarmes_registrados_1d,qtd_alarmes_registrados_7d,qtd_alarmes_registrados_14d,variacao_alarmes_registrados_7d,qtd_fo_7d,qtd_lsd_7d,dias_com_registro_7d,dias_com_registro_14d,dias_desde_ultima_zm01,dias_desde_ultima_zm02,sem_historico_zm01,sem_historico_zm02
0,0,0,0,0,0,0,0,0,93,<NA>,0,1
1,0,0,0,0,0,0,0,0,94,<NA>,0,1
2,0,0,0,0,0,0,0,0,95,<NA>,0,1
3,0,0,0,0,0,0,0,0,96,<NA>,0,1
4,0,0,0,0,0,0,0,0,97,<NA>,0,1


## 51. Etapa 3 — separação cronológica

Os cortes usam **70% dos dias para treino, 15% para validação e 15% para teste**, com as mesmas datas para todas as portas. As quantidades finais de amostras podem ter proporções diferentes devido às exclusões.

O alvo de uma amostra de treino deve terminar antes do primeiro dia da validação. A mesma regra vale entre validação e teste. Isso remove sete datas candidatas de alvo no fim de cada um desses períodos. Os últimos sete dias do teste já estão sem horizonte completo.

A primeira evidência de cada porta nos registros limita o início do seu histórico elegível. Uma amostra futura precisa de 30 dias de calendário desde essa referência. A primeira evidência não comprova início de operação ou coleta contínua: essa é uma restrição conservadora da primeira versão.

**As linhas do calendário não são removidas.** O campo `usar_como_alvo` indica quais datas poderão encerrar uma sequência supervisionada. Linhas excluídas como alvo podem continuar como histórico disponível de uma janela posterior. Nenhuma sequência é criada nesta etapa.

In [55]:
datas_modelagem = pd.date_range(DATA_INICIO, DATA_FIM, freq="D")
INICIO_VALIDACAO = datas_modelagem[int(len(datas_modelagem) * 0.70)]
INICIO_TESTE = datas_modelagem[int(len(datas_modelagem) * 0.85)]

base_modelagem_proxy["particao"] = "treino"
base_modelagem_proxy.loc[base_modelagem_proxy["data"] >= INICIO_VALIDACAO, "particao"] = "validacao"
base_modelagem_proxy.loc[base_modelagem_proxy["data"] >= INICIO_TESTE, "particao"] = "teste"

evidencias_portas = pd.concat([
    ativacoes_alarmes[["chave_candidata", "data_alarme"]].rename(columns={"data_alarme": "data"}),
    eventos_zm01_individuais.rename(columns={"data_avaria": "data"}),
    eventos_zm02_individuais.rename(columns={"data_avaria": "data"}),
], ignore_index=True)
primeira_evidencia = evidencias_portas.groupby("chave_candidata")["data"].min()

base_modelagem_proxy["inicio_historico_porta"] = base_modelagem_proxy["chave_candidata"].map(primeira_evidencia)
assert base_modelagem_proxy["inicio_historico_porta"].notna().all()
base_modelagem_proxy["inicio_historico_porta"] = base_modelagem_proxy["inicio_historico_porta"].clip(lower=DATA_INICIO)

base_modelagem_proxy["tem_30_dias_calendario"] = (
    base_modelagem_proxy["data"] - pd.Timedelta(days=DIAS_HISTORICO - 1)
    >= base_modelagem_proxy["inicio_historico_porta"]
)

fim_alvo = base_modelagem_proxy["data"] + pd.Timedelta(days=DIAS_ALVO)
alvo_atravessa_corte = (
    (base_modelagem_proxy["particao"].eq("treino") & fim_alvo.ge(INICIO_VALIDACAO))
    | (base_modelagem_proxy["particao"].eq("validacao") & fim_alvo.ge(INICIO_TESTE))
)

base_modelagem_proxy["usar_como_alvo"] = (
    base_modelagem_proxy["registro_zm01_7d"].notna()
    & base_modelagem_proxy["tem_30_dias_calendario"]
    & ~alvo_atravessa_corte
)

indices_alvo_proxy = {}
resumo_divisao = []
for particao in ["treino", "validacao", "teste"]:
    mascara = base_modelagem_proxy["particao"].eq(particao) & base_modelagem_proxy["usar_como_alvo"]
    indices_alvo_proxy[particao] = base_modelagem_proxy.index[mascara]
    amostras = base_modelagem_proxy.loc[mascara]
    assert len(amostras) > 0, f"A partição {particao} ficou sem amostras."
    assert set(amostras["registro_zm01_7d"].unique()) == {0, 1}, f"A partição {particao} não tem as duas classes."

    resumo_divisao.append({
        "particao": particao,
        "primeiro_alvo": amostras["data"].min(),
        "ultimo_alvo": amostras["data"].max(),
        "amostras_elegiveis": len(amostras),
        "positivos": int(amostras["registro_zm01_7d"].sum()),
        "negativos_operacionais": int(amostras["registro_zm01_7d"].eq(0).sum()),
    })

assert (base_modelagem_proxy.loc[indices_alvo_proxy["treino"], "data"] + pd.Timedelta(days=DIAS_ALVO) < INICIO_VALIDACAO).all()
assert (base_modelagem_proxy.loc[indices_alvo_proxy["validacao"], "data"] + pd.Timedelta(days=DIAS_ALVO) < INICIO_TESTE).all()
assert (base_modelagem_proxy.loc[indices_alvo_proxy["teste"], "data"] + pd.Timedelta(days=DIAS_ALVO) <= DATA_FIM).all()

configuracao_modelagem["inicio_validacao"] = str(INICIO_VALIDACAO.date())
configuracao_modelagem["inicio_teste"] = str(INICIO_TESTE.date())
print("Início da validação:", INICIO_VALIDACAO.date())
print("Início do teste:", INICIO_TESTE.date())
display(pd.DataFrame(resumo_divisao))


Início da validação: 2025-07-03
Início do teste: 2025-11-14


,particao,primeiro_alvo,ultimo_alvo,amostras_elegiveis,positivos,negativos_operacionais
0,treino,2023-11-15,2025-06-25,239150,23775,215375
1,validacao,2025-07-03,2025-11-06,50852,5731,45121
2,teste,2025-11-14,2026-03-21,50145,5554,44591


## 52. Imputação e normalização ajustadas somente no treino

O preenchimento usa a mediana de cada variável no treino. A normalização usa a média e o desvio-padrão desse mesmo conjunto. Os indicadores de histórico ausente permanecem binários, sem normalização.

O ajuste considera apenas datas do período de treino a partir da primeira evidência de cada porta. Ele inclui linhas de histórico, mesmo quando a linha não pode ser alvo, porque suas entradas podem compor as sequências. Validação e teste recebem somente `transform()`, sem novo ajuste.

As datas, os identificadores, os campos de auditoria e o alvo não entram no transformador numérico. O calendário completo é transformado para preservar o contexto das futuras janelas, inclusive nas fronteiras entre conjuntos.

In [56]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

mascara_ajuste_treino = (
    base_modelagem_proxy["particao"].eq("treino")
    & base_modelagem_proxy["data"].ge(base_modelagem_proxy["inicio_historico_porta"])
)
assert mascara_ajuste_treino.any()
assert base_modelagem_proxy.loc[mascara_ajuste_treino, "data"].max() < INICIO_VALIDACAO

numericas_treino = base_modelagem_proxy.loc[mascara_ajuste_treino, features_numericas].astype(float)
numericas_completas = base_modelagem_proxy[features_numericas].astype(float)

imputador = SimpleImputer(strategy="median", keep_empty_features=True)
numericas_treino_imputadas = imputador.fit_transform(numericas_treino)

normalizador = StandardScaler()
normalizador.fit(numericas_treino_imputadas)

dados_temporais_prontos = pd.DataFrame(
    normalizador.transform(imputador.transform(numericas_completas)).astype("float32"),
    columns=features_numericas,
    index=base_modelagem_proxy.index,
)
dados_temporais_prontos[features_indicadores] = base_modelagem_proxy[features_indicadores].astype("float32")

estatisticas_treino = pd.DataFrame({
    "variavel": features_numericas,
    "mediana_imputacao": imputador.statistics_,
    "media_normalizacao": normalizador.mean_,
    "escala_normalizacao": normalizador.scale_,
})
assert dados_temporais_prontos.notna().all().all()
assert dados_temporais_prontos.loc[mascara_ajuste_treino, features_numericas].mean().abs().max() < 0.00001

display(estatisticas_treino)
print("Formato das entradas temporais diárias:", dados_temporais_prontos.shape)


,variavel,mediana_imputacao,media_normalizacao,escala_normalizacao
0,qtd_alarmes_registrados_1d,0.0,0.723044,4.619428
1,qtd_alarmes_registrados_7d,0.0,5.045993,22.027461
2,qtd_alarmes_registrados_14d,0.0,10.043892,37.911120
3,variacao_alarmes_registrados_7d,0.0,0.048095,22.349764
4,qtd_fo_7d,0.0,0.851110,11.573535
5,qtd_lsd_7d,0.0,1.047459,15.361615
6,dias_com_registro_7d,0.0,0.800849,1.397303
7,dias_com_registro_14d,0.0,1.593694,2.549286
8,dias_desde_ultima_zm01,67.0,106.375388,117.838170
9,dias_desde_ultima_zm02,234.0,250.491359,133.476085


Formato das entradas temporais diárias: (472032, 12)


## 53. Codificação das informações estáticas

Estação e plataforma são tratadas como categorias, sem impor uma distância numérica entre elas. O `OneHotEncoder` aprende as categorias apenas nas linhas de ajuste do treino. Categorias novas em validação/teste são representadas por zeros no bloco correspondente, conforme `handle_unknown="ignore"`.

Porta e chave candidata permanecem como identificadores para separar as sequências. O tipo PDM é constante nesta versão e não acrescenta uma variável estática.

As tabelas prontas continuam alinhadas ao índice de `base_modelagem_proxy`. Os índices de alvo das três partições apontam para as linhas que poderão encerrar as futuras sequências de 30 dias.

In [57]:
estaticas = base_modelagem_proxy[features_estaticas].astype(str)
codificador_estatico = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype="float32")
codificador_estatico.fit(estaticas.loc[mascara_ajuste_treino])

nomes_estaticas = list(codificador_estatico.get_feature_names_out(features_estaticas))
dados_estaticos_prontos = pd.DataFrame(
    codificador_estatico.transform(estaticas),
    columns=nomes_estaticas,
    index=base_modelagem_proxy.index,
)

preprocessadores = {
    "imputador": imputador,
    "normalizador": normalizador,
    "codificador_estatico": codificador_estatico,
}

features_temporais_finais = features_numericas + features_indicadores
configuracao_modelagem["features_temporais"] = features_temporais_finais
configuracao_modelagem["features_estaticas_codificadas"] = nomes_estaticas

alvos_proxy_por_particao = {}
for particao, indices in indices_alvo_proxy.items():
    alvos_proxy_por_particao[particao] = base_modelagem_proxy.loc[indices, "registro_zm01_7d"].astype("int8")
    assert alvos_proxy_por_particao[particao].index.equals(indices)

assert dados_temporais_prontos.index.equals(base_modelagem_proxy.index)
assert dados_estaticos_prontos.index.equals(base_modelagem_proxy.index)
assert dados_estaticos_prontos.notna().all().all()
assert set(indices_alvo_proxy["treino"]).isdisjoint(indices_alvo_proxy["validacao"])
assert set(indices_alvo_proxy["treino"]).isdisjoint(indices_alvo_proxy["teste"])
assert set(indices_alvo_proxy["validacao"]).isdisjoint(indices_alvo_proxy["teste"])

display(dados_estaticos_prontos.head())
print("Variáveis temporais:", len(features_temporais_finais))
print("Variáveis estáticas codificadas:", len(nomes_estaticas))
print("Preparação da referência ZM01 concluída; alvo de falha real pendente.")


,estacao_BUT,estacao_FAL,estacao_FRA,estacao_FRE,estacao_HIG,estacao_LUZ,...,estacao_PIN,estacao_PTA,estacao_REP,estacao_VSO,plataforma_1,plataforma_2
0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0
1,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0
2,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0
3,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0
4,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0


Variáveis temporais: 12
Variáveis estáticas codificadas: 13
Preparação da referência ZM01 concluída; alvo de falha real pendente.


## 54. Conferência final do alvo e dos dados preparados

A verificação direta compara exemplos de positivos e negativos operacionais com as datas das ZM01 individuais da fonte consolidada, sem reutilizar a tabela de positivos expandida. Também confere que a preparação numérica preservou a ordem das colunas e não produziu valores infinitos.

Os índices e as tabelas diárias estão preparados, mas **não há tensores de sequências nem treinamento nesta etapa**.

In [58]:
import numpy as np

amostras_alvo = pd.concat([
    base_modelagem_proxy.loc[base_modelagem_proxy["usar_como_alvo"] & base_modelagem_proxy["registro_zm01_7d"].eq(1)].head(3),
    base_modelagem_proxy.loc[base_modelagem_proxy["usar_como_alvo"] & base_modelagem_proxy["registro_zm01_7d"].eq(0)].head(3),
])

for _, linha in amostras_alvo.iterrows():
    encontrou_zm01 = (
        eventos_zm01_individuais["chave_candidata"].eq(linha["chave_candidata"])
        & eventos_zm01_individuais["data_avaria"].gt(linha["data"])
        & eventos_zm01_individuais["data_avaria"].le(linha["data"] + pd.Timedelta(days=DIAS_ALVO))
    ).any()
    assert int(encontrou_zm01) == linha["registro_zm01_7d"]

assert list(dados_temporais_prontos.columns) == features_temporais_finais
assert np.isfinite(dados_temporais_prontos.to_numpy()).all()
assert np.isfinite(dados_estaticos_prontos.to_numpy()).all()
assert normalizador.n_features_in_ == len(features_numericas)
assert not base_modelagem_proxy.duplicated(["chave_candidata", "data"]).any()

display(amostras_alvo[["chave_candidata", "data", "registro_zm01_7d", "particao"]])
display(pd.DataFrame(resumo_divisao))


,chave_candidata,data,registro_zm01_7d,particao
46,BUT|1|10|PDM,2023-12-02,1,treino
47,BUT|1|10|PDM,2023-12-03,1,treino
48,BUT|1|10|PDM,2023-12-04,1,treino
29,BUT|1|10|PDM,2023-11-15,0,treino
30,BUT|1|10|PDM,2023-11-16,0,treino
31,BUT|1|10|PDM,2023-11-17,0,treino


,particao,primeiro_alvo,ultimo_alvo,amostras_elegiveis,positivos,negativos_operacionais
0,treino,2023-11-15,2025-06-25,239150,23775,215375
1,validacao,2025-07-03,2025-11-06,50852,5731,45121
2,teste,2025-11-14,2026-03-21,50145,5554,44591


### Entregáveis da referência por registro ZM01

- `base_modelagem_proxy`: calendário, alvo operacional, auditoria, partição e elegibilidade como alvo.
- `dados_temporais_prontos`: 12 variáveis diárias, com imputação e normalização ajustadas apenas no treino.
- `dados_estaticos_prontos`: estação e plataforma codificadas a partir do treino.
- `indices_alvo_proxy` e `alvos_proxy_por_particao`: referências supervisionadas separadas por data, com exclusão dos horizontes que atravessam os cortes.
- `preprocessadores` e `configuracao_modelagem`: transformações e definições utilizadas.

Os resultados permanecem em memória. Nenhum arquivo de origem é modificado, nenhuma sequência é criada e nenhum modelo é treinado.

**Limitação central:** a versão é experimental e usa ausência de registro ZM01 como negativo operacional. Cobertura e disponibilidade histórica não foram confirmadas. As próximas etapas poderão construir sequências e treinar esse indicador, mas a interpretação como risco real de falha depende de validação operacional dos dados.

Esses objetos não são um conjunto supervisionado validado de falhas reais. A separação temporal e as entradas podem ser reaproveitadas; os rótulos precisam ser reconstruídos após a definição dos episódios.

## 55. Evidências de ocorrências: ZM01 e ZM02

O objetivo final permanece **identificar o início de uma falha real entre D+1 e D+7**. ZM01 e ZM02 são fontes de evidência; seus registros não se tornam automaticamente falhas confirmadas. ZM03 permanece como contexto de programação preventiva.

A referência anterior foi renomeada para `registro_zm01_7d`. Os objetos `base_modelagem_proxy`, `indices_alvo_proxy` e `alvos_proxy_por_particao` pertencem a essa referência e não devem ser confundidos com rótulos validados de falhas reais.

Esta seção reúne identificação, instante de avaria e campos de sintoma, objeto, defeito, causa e ação. Um campo preenchido indica informação disponível para revisão; não confirma sozinho que a ocorrência atende ao conceito de falha desejado.

Registros da mesma porta no mesmo instante são agrupados em um **candidato a episódio**. Ocorrências em instantes diferentes permanecem separadas, mesmo se forem próximas. A consolidação de um episódio prolongado ou recorrente depende da revisão das evidências.

In [59]:
campos_evidencia = [
    "nota", "ordem", "inicio_avaria_datetime",
    "descricao", "sintoma", "objeto", "defeito", "causa", "acao",
]

evidencias_zm01 = fonte_zm01_alvo[
    campos_evidencia + colunas_identificacao + ["falha_flag", "aba_origem"]
].copy()
evidencias_zm01["fonte"] = "ZM01"

evidencias_zm02 = bases["ZM02"][campos_evidencia + ["local_de_instalacao"]].copy()
identificacao_zm02_evidencia = extrair_ativos(evidencias_zm02["local_de_instalacao"])
evidencias_zm02 = evidencias_zm02.join(identificacao_zm02_evidencia[colunas_identificacao])
evidencias_zm02["fonte"] = "ZM02"
evidencias_zm02["aba_origem"] = "ZM02_limpa"
evidencias_zm02["falha_flag"] = pd.NA

evidencias_manutencao = pd.concat([evidencias_zm01, evidencias_zm02], ignore_index=True)
evidencias_manutencao["inicio_avaria"] = pd.to_datetime(
    evidencias_manutencao["inicio_avaria_datetime"], errors="coerce"
)
for campo in ["descricao", "sintoma", "objeto", "defeito", "causa", "acao"]:
    evidencias_manutencao[campo] = (
        evidencias_manutencao[campo].astype("string").str.strip().replace("", pd.NA)
    )

evidencias_individuais = evidencias_manutencao.loc[
    evidencias_manutencao["tipo_ativo"].eq(TIPO_ATIVO)
    & evidencias_manutencao["chave_candidata"].isin(portas["chave_candidata"])
    & evidencias_manutencao["inicio_avaria"].ge(DATA_INICIO)
    & evidencias_manutencao["inicio_avaria"].lt(DATA_FIM + pd.Timedelta(days=1))
].copy()

evidencias_individuais["id_candidato"] = (
    evidencias_individuais["chave_candidata"] + "|"
    + evidencias_individuais["inicio_avaria"].dt.strftime("%Y%m%dT%H%M%S.%f")
)
evidencias_individuais["tem_zm01"] = evidencias_individuais["fonte"].eq("ZM01").astype(int)
evidencias_individuais["tem_zm02"] = evidencias_individuais["fonte"].eq("ZM02").astype(int)
evidencias_individuais["campo_defeito_preenchido"] = evidencias_individuais["defeito"].notna().astype(int)
evidencias_individuais["marcacao_falha_sap"] = evidencias_individuais["falha_flag"].eq(True).astype(int)

candidatos_episodios = (
    evidencias_individuais.groupby(["id_candidato", "chave_candidata", "inicio_avaria"])
    .agg(
        qtd_registros_manutencao=("fonte", "size"),
        tem_zm01=("tem_zm01", "max"),
        tem_zm02=("tem_zm02", "max"),
        tem_defeito_catalogado=("campo_defeito_preenchido", "max"),
        tem_marcacao_falha_sap=("marcacao_falha_sap", "max"),
    )
    .reset_index()
)
assert candidatos_episodios["id_candidato"].is_unique
assert candidatos_episodios["qtd_registros_manutencao"].sum() == len(evidencias_individuais)

print("Registros individuais para análise:", len(evidencias_individuais))
print("Candidatos a episódios:", len(candidatos_episodios))
display(candidatos_episodios.head())


Registros individuais para análise: 7882
Candidatos a episódios: 7815


,id_candidato,chave_candidata,inicio_avaria,qtd_registros_manutencao,tem_zm01,tem_zm02,tem_defeito_catalogado,tem_marcacao_falha_sap
0,BUT|1|10|PDM|20231110T143500.000000,BUT|1|10|PDM,2023-11-10 14:35:00,1,1,0,1,0
1,BUT|1|10|PDM|20231110T155600.000000,BUT|1|10|PDM,2023-11-10 15:56:00,1,1,0,1,1
2,BUT|1|10|PDM|20231112T104100.000000,BUT|1|10|PDM,2023-11-12 10:41:00,1,1,0,1,1
3,BUT|1|10|PDM|20231113T193200.000000,BUT|1|10|PDM,2023-11-13 19:32:00,1,1,0,1,1
4,BUT|1|10|PDM|20231113T213000.000000,BUT|1|10|PDM,2023-11-13 21:30:00,1,1,0,1,0


## 56. Associação candidata entre ocorrência e alarme

A janela inicial de busca é **24 horas antes e 24 horas depois** do início da avaria. Essa tolerância é exploratória e está concentrada em duas variáveis de configuração. Ela não foi validada como regra causal ou escolhida para maximizar métricas de teste.

O cruzamento exige a mesma chave de estação/plataforma/porta/tipo. A junção primeiro utiliza os dias possíveis e depois filtra os instantes exatos, evitando cruzar todo o histórico de cada porta.

Um alarme pode ser candidato a mais de uma ocorrência próxima. Essas relações permanecem identificáveis; nenhuma associação exclusiva é presumida. Alarmes posteriores podem representar consequência, teste ou intervenção. Todos os alarmes associados nesta etapa servem à revisão do episódio, **não a novas entradas históricas do modelo**.

In [60]:
HORAS_ANTES_AVARIA = 24
HORAS_DEPOIS_AVARIA = 24
assert isinstance(HORAS_ANTES_AVARIA, int) and HORAS_ANTES_AVARIA >= 0
assert isinstance(HORAS_DEPOIS_AVARIA, int) and HORAS_DEPOIS_AVARIA >= 0

dias_antes_busca = (HORAS_ANTES_AVARIA + 23) // 24
dias_depois_busca = (HORAS_DEPOIS_AVARIA + 23) // 24
deslocamentos_busca = pd.DataFrame({
    "deslocamento_dias": range(-dias_antes_busca, dias_depois_busca + 1)
})

dias_busca_episodios = candidatos_episodios[
    ["id_candidato", "chave_candidata", "inicio_avaria"]
].merge(deslocamentos_busca, how="cross")
dias_busca_episodios["data_alarme"] = (
    dias_busca_episodios["inicio_avaria"].dt.normalize()
    + pd.to_timedelta(dias_busca_episodios["deslocamento_dias"], unit="D")
)

associacoes_candidatas = dias_busca_episodios.merge(
    ativacoes_classificadas[[
        "chave_candidata", "data_alarme", "tag", "inicio_alarme", "tipo_evento",
    ]],
    on=["chave_candidata", "data_alarme"],
    how="inner",
    validate="many_to_many",
)
associacoes_candidatas["diferenca_horas"] = (
    associacoes_candidatas["inicio_alarme"] - associacoes_candidatas["inicio_avaria"]
).dt.total_seconds() / 3600

associacoes_candidatas = associacoes_candidatas.loc[
    associacoes_candidatas["diferenca_horas"].between(
        -HORAS_ANTES_AVARIA, HORAS_DEPOIS_AVARIA
    )
].copy()
associacoes_candidatas["antes_ou_no_instante"] = associacoes_candidatas["diferenca_horas"].le(0).astype(int)
associacoes_candidatas["depois_do_instante"] = associacoes_candidatas["diferenca_horas"].gt(0).astype(int)
associacoes_candidatas["distancia_horas"] = associacoes_candidatas["diferenca_horas"].abs()

assert not associacoes_candidatas.duplicated(["id_candidato", "tag", "inicio_alarme"]).any()
assert associacoes_candidatas["diferenca_horas"].between(
    -HORAS_ANTES_AVARIA, HORAS_DEPOIS_AVARIA
).all()
print("Associações candidatas encontradas:", len(associacoes_candidatas))
print("Ocorrências com pelo menos um alarme próximo:", associacoes_candidatas["id_candidato"].nunique())
display(associacoes_candidatas[[
    "id_candidato", "tag", "inicio_avaria", "inicio_alarme", "tipo_evento", "diferenca_horas",
]].head())


Associações candidatas encontradas: 94075
Ocorrências com pelo menos um alarme próximo: 4443


,id_candidato,tag,inicio_avaria,inicio_alarme,tipo_evento,diferenca_horas
0,BUT|1|10|PDM|20231110T143500.000000,PSD_P_BUT1_PDM10_LSD,2023-11-10 14:35:00,2023-11-10 16:51:24.910,LSD,2.273586
1,BUT|1|10|PDM|20231110T143500.000000,PSD_P_BUT1_PDM10_LSD,2023-11-10 14:35:00,2023-11-10 16:11:34.927,LSD,1.609702
2,BUT|1|10|PDM|20231110T143500.000000,PSD_P_BUT1_PDM10_CA,2023-11-10 14:35:00,2023-11-10 15:42:04.927,CA,1.118035
6,BUT|1|10|PDM|20231110T155600.000000,PSD_P_BUT1_PDM10_LSD,2023-11-10 15:56:00,2023-11-10 16:51:24.910,LSD,0.923586
7,BUT|1|10|PDM|20231110T155600.000000,PSD_P_BUT1_PDM10_LSD,2023-11-10 15:56:00,2023-11-10 16:11:34.927,LSD,0.259702


## 57. Quadro de evidências por candidato

O resumo informa quantos alarmes foram encontrados antes/depois da referência e quantos códigos distintos aparecem. A ausência de alarme próximo não invalida automaticamente uma ocorrência registrada em manutenção.

Os grupos de evidência organizam a revisão; não são classes finais do modelo. `tem_defeito_catalogado` significa somente que o campo defeito está preenchido. Os textos originais continuam em `evidencias_individuais`, ligados pelo identificador do candidato.

In [61]:
resumo_associacoes = (
    associacoes_candidatas.groupby("id_candidato")
    .agg(
        qtd_alarmes_proximos=("tag", "size"),
        qtd_alarmes_antes=("antes_ou_no_instante", "sum"),
        qtd_alarmes_depois=("depois_do_instante", "sum"),
        qtd_tipos_proximos=("tipo_evento", "nunique"),
        menor_distancia_horas=("distancia_horas", "min"),
    )
    .reset_index()
)

candidatos_episodios = candidatos_episodios.merge(
    resumo_associacoes, on="id_candidato", how="left", validate="one_to_one"
)
contagens_associacao = [
    "qtd_alarmes_proximos", "qtd_alarmes_antes", "qtd_alarmes_depois", "qtd_tipos_proximos",
]
candidatos_episodios[contagens_associacao] = (
    candidatos_episodios[contagens_associacao].fillna(0).astype(int)
)

candidatos_episodios["grupo_evidencias"] = "sem defeito catalogado e sem alarme próximo"
com_defeito = candidatos_episodios["tem_defeito_catalogado"].eq(1)
com_alarme = candidatos_episodios["qtd_alarmes_proximos"].gt(0)
candidatos_episodios.loc[com_defeito & ~com_alarme, "grupo_evidencias"] = "defeito catalogado, sem alarme próximo"
candidatos_episodios.loc[~com_defeito & com_alarme, "grupo_evidencias"] = "alarme próximo, sem defeito catalogado"
candidatos_episodios.loc[com_defeito & com_alarme, "grupo_evidencias"] = "defeito catalogado e alarme próximo"

assert candidatos_episodios["qtd_alarmes_proximos"].sum() == len(associacoes_candidatas)
assert (
    candidatos_episodios["qtd_alarmes_antes"] + candidatos_episodios["qtd_alarmes_depois"]
    == candidatos_episodios["qtd_alarmes_proximos"]
).all()

display(candidatos_episodios["grupo_evidencias"].value_counts().to_frame("candidatos"))

# A amostra mantém variantes de texto da mesma ocorrência para revisão.
ids_exemplo = candidatos_episodios.loc[com_defeito & com_alarme, "id_candidato"].head(5)
display(evidencias_individuais.loc[
    evidencias_individuais["id_candidato"].isin(ids_exemplo),
    ["id_candidato", "fonte", "nota", "ordem", "sintoma", "objeto", "defeito", "causa"],
].head(10))


,candidatos
grupo_evidencias,
defeito catalogado e alarme próximo,3985
"defeito catalogado, sem alarme próximo",3103
"alarme próximo, sem defeito catalogado",458
sem defeito catalogado e sem alarme próximo,269


,id_candidato,fonte,nota,ordem,sintoma,objeto,defeito,causa
123,BUT|1|10|PDM|20231110T143500.000000,ZM01,13599562,52626605.0,PDM NÃO FECHOU,SENSOR PORTA TRAVADA-HEADER BOX,SENSOR DESTRAVADO AVARIADO,DESGASTE
124,BUT|1|10|PDM|20231110T155600.000000,ZM01,13599659,52626722.0,PDM NÃO FECHOU,CABO/CONECTORES,BAIXA ISOLACAO,DESGASTE
125,BUT|1|10|PDM|20231112T104100.000000,ZM01,13602529,52629486.0,PDM NÃO FECHOU,LINGUETA DA TRAVA PDM,ATUADOR DA TRAVA DE PDM DESAJUSTADO,VIBRACAO
126,BUT|1|10|PDM|20231113T193200.000000,ZM01,13605732,52632510.0,PDM NÃO FECHOU,TRAVA EUCHNER PDM,TRAVA DE SEGURANCA PDM DESAJUSTADA,OPERACAO INDEVIDA
127,BUT|1|10|PDM|20231113T213000.000000,ZM01,13605974,52632765.0,PDM NÃO FECHOU,TRAVA EUCHNER PDM,SENSOR ABERTO DESAJUSTADO,DESGASTE


## 58. Códigos associados a ZM01 no período de treino

A frequência abaixo utiliza somente candidatos ZM01 cuja janela inteira de associação termina antes do início da validação. Cada candidato é contado uma vez por código de alarme, mesmo quando há várias ativações desse código.

O resultado descreve proximidade temporal no treino. Ele não comprova relação com o defeito, poder preditivo ou gravidade. Uma seleção futura de tipos de alarme precisará comparar períodos de controle e ser validada fora do treino. Nenhuma variável é selecionada automaticamente por essa frequência.

In [62]:
ids_zm01_treino = candidatos_episodios.loc[
    candidatos_episodios["tem_zm01"].eq(1)
    & (
        candidatos_episodios["inicio_avaria"] + pd.Timedelta(hours=HORAS_DEPOIS_AVARIA)
        < INICIO_VALIDACAO
    ),
    "id_candidato",
]
associacoes_zm01_treino = associacoes_candidatas.loc[
    associacoes_candidatas["id_candidato"].isin(ids_zm01_treino)
].copy()

frequencia_codigos_treino = (
    associacoes_zm01_treino.groupby("tipo_evento")
    .agg(
        candidatos_zm01_associados=("id_candidato", "nunique"),
        pares_alarme_ocorrencia=("id_candidato", "size"),
    )
    .sort_values("candidatos_zm01_associados", ascending=False)
)

assert (associacoes_zm01_treino["inicio_avaria"] < INICIO_VALIDACAO).all()
assert (associacoes_zm01_treino["inicio_alarme"] < INICIO_VALIDACAO).all()
display(frequencia_codigos_treino)


,candidatos_zm01_associados,pares_alarme_ocorrencia
tipo_evento,,
LSD,2074,28020
FC,2035,3697
OBS,1947,3570
CA,1870,3846
FO,1806,18831
OA,1382,5657
CSD,1363,1935
RCE,80,118


## 59. Registro da revisão dos episódios

A confirmação exige um critério de defeito compatível com o objetivo do projeto e a análise das evidências disponíveis. A tabela abaixo recebe o identificador do candidato, a decisão e a justificativa. Ela começa vazia, pois nenhuma confirmação foi presumida.

Uma decisão `True` confirma o candidato conforme a revisão; `False` rejeita esse candidato. **Rejeitar uma ocorrência não comprova ausência de outras falhas nos sete dias seguintes** e não gera um negativo de treinamento.

Ocorrências sem revisão continuam com `falha_confirmada` ausente. Se vários candidatos corresponderem ao mesmo episódio prolongado, a revisão também deverá resolver a data de início e os registros redundantes antes de consolidar o alvo.

In [63]:
# Tabela destinada às decisões fundamentadas da revisão.
revisoes_episodios = pd.DataFrame({
    "id_candidato": pd.Series(dtype="string"),
    "falha_confirmada": pd.Series(dtype="boolean"),
    "justificativa": pd.Series(dtype="string"),
})

assert revisoes_episodios["id_candidato"].is_unique
assert revisoes_episodios["id_candidato"].isin(candidatos_episodios["id_candidato"]).all()
assert revisoes_episodios["falha_confirmada"].notna().all()
assert revisoes_episodios["justificativa"].str.strip().str.len().gt(0).fillna(False).all()

episodios_em_revisao = candidatos_episodios.merge(
    revisoes_episodios, on="id_candidato", how="left", validate="one_to_one"
)
episodios_em_revisao["falha_confirmada"] = episodios_em_revisao["falha_confirmada"].astype("boolean")

episodios_confirmados = episodios_em_revisao.loc[
    episodios_em_revisao["falha_confirmada"].eq(True)
].copy()

print("Candidatos pendentes de revisão:", episodios_em_revisao["falha_confirmada"].isna().sum())
print("Candidatos confirmados:", len(episodios_confirmados))
print("Candidatos rejeitados:", episodios_em_revisao["falha_confirmada"].eq(False).sum())


Candidatos pendentes de revisão: 7815
Candidatos confirmados: 0
Candidatos rejeitados: 0


## 60. Separação entre referência ZM01 e alvo de falhas

O campo `falha_7d` passa a ser reservado ao alvo baseado em episódios revisados. A referência histórica continua em `base_modelagem_proxy["registro_zm01_7d"]` e seus rótulos não são copiados para esse campo.

A tabela abaixo apenas prevê onde os positivos de episódios confirmados entrarão. Como não há revisões confirmadas nem cobertura validada, o alvo de falha real permanece ausente. Negativos exigem uma regra de acompanhamento completo, além da revisão dos episódios.

Os dados temporais, a codificação estática e os cortes cronológicos permanecem disponíveis. Eles não autorizam o treinamento de falhas reais com os rótulos da referência por ZM01.

In [64]:
referencias_confirmadas = episodios_confirmados[[
    "chave_candidata", "inicio_avaria"
]].copy()
referencias_confirmadas["data_falha"] = referencias_confirmadas["inicio_avaria"].dt.normalize()
referencias_confirmadas = referencias_confirmadas.merge(antecedencias_alvo, how="cross")
referencias_confirmadas["data"] = (
    referencias_confirmadas["data_falha"]
    - pd.to_timedelta(referencias_confirmadas["dias_antes"], unit="D")
)
positivos_falhas_revisadas = referencias_confirmadas[[
    "chave_candidata", "data"
]].drop_duplicates()
positivos_falhas_revisadas["falha_7d"] = 1

base_alvo_falhas = base_modelagem_proxy[[
    "chave_candidata", "data", "particao", "tem_30_dias_calendario"
]].merge(
    positivos_falhas_revisadas,
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
)
base_alvo_falhas["falha_7d"] = base_alvo_falhas["falha_7d"].astype("Int64")
base_alvo_falhas.loc[
    base_alvo_falhas["data"] + pd.Timedelta(days=DIAS_ALVO) > DATA_FIM, "falha_7d"
] = pd.NA

configuracao_modelagem["objetivo_final"] = "início de falha real da porta entre D+1 e D+7"
configuracao_modelagem["alvo_final_validado"] = False
configuracao_modelagem["status_alvo_final"] = "episódios e cobertura pendentes de validação"

assert len(base_alvo_falhas) == len(base_modelagem_proxy)
assert not base_alvo_falhas.duplicated(["chave_candidata", "data"]).any()
assert base_alvo_falhas["falha_7d"].dropna().eq(1).all()
assert "falha_7d" not in base_modelagem_proxy.columns
if len(episodios_confirmados) == 0:
    assert base_alvo_falhas["falha_7d"].isna().all()

print("Rótulos positivos de falha revisada:", base_alvo_falhas["falha_7d"].eq(1).sum())
print("Negativos de falha real definidos:", base_alvo_falhas["falha_7d"].eq(0).sum())
print("Alvo de falha real validado:", configuracao_modelagem["alvo_final_validado"])


Rótulos positivos de falha revisada: 0
Negativos de falha real definidos: 0
Alvo de falha real validado: False


### Resultado do cruzamento de evidências

A tabela `candidatos_episodios` organiza ocorrências individuais de ZM01/ZM02, e `associacoes_candidatas` preserva os alarmes encontrados dentro da janela temporal da mesma porta. Os campos técnicos completos permanecem em `evidencias_individuais`.

Nenhuma associação temporal, marcação SAP ou campo preenchido foi convertido automaticamente em falha confirmada. A ausência de alarme próximo também não elimina um candidato.

A referência por registro ZM01 está explicitamente separada do objetivo final. O próximo passo é revisar os candidatos e aprovar o critério de defeito, a consolidação dos episódios e o acompanhamento necessário aos negativos. Até então, `base_alvo_falhas` não é um conjunto supervisionado pronto para treinamento.

## 61. Critério inicial para revisão de ZM01 e ZM02

A proposta inicial considera **perda ou comprometimento da função da porta**, documentados nos campos técnicos, como evidência para revisar uma possível falha. ZM01 e ZM02 recebem o mesmo tratamento. A modalidade de manutenção, isoladamente, não define o alvo.

A triagem utiliza listas explícitas de sintomas de abertura, fechamento, travamento e indicação. Um sintoma dessa lista acompanhado de defeito preenchido recebe prioridade de revisão. Isso ainda não confirma a ocorrência nem a data real de início.

Obstruções recebem uma categoria própria: uma porta impedida por objeto externo pode apresentar um sintoma sem ter um componente defeituoso. Ruído, dano estrutural, textos ausentes e outros sintomas permanecem para interpretação. Nenhuma dessas categorias significa automaticamente “sem falha”.

As listas abaixo são uma proposta exploratória baseada nos catálogos consultados. Elas não foram validadas por métricas. Antes da avaliação final do modelo, o critério deverá ser definido e fixado; as amostras de revisão desta etapa são retiradas somente do período de treino.


In [65]:
sintomas_funcionais = [
    "PDM NÃO FECHOU",
    "PDM NÃO ABRE",
    "PDM NÃO TRAVA FECHADA",
    "PDM NÃO DESTRAVA",
    "PDM FECHA PARCIALMENTE",
    "PDM ABRE PARCIALMENTE",
    "PDM NÃO INDICA ABERTA",
    "PDM NÃO INDICA FECHADA",
]
defeitos_obstrucao = [
    "OBSTRUCAO POR OBJETO ESTRANHO",
    "OBSTRUCAO DE FECHAMENTO",
    "OBSTRUCAO DE ABERTURA",
]

triagem_registros = evidencias_individuais.copy()
triagem_registros["sintoma_normalizado"] = (
    triagem_registros["sintoma"].astype("string").str.strip().str.upper()
)
triagem_registros["defeito_normalizado"] = (
    triagem_registros["defeito"].astype("string").str.strip().str.upper()
)

sintoma_funcional = triagem_registros["sintoma_normalizado"].isin(sintomas_funcionais)
defeito_preenchido = triagem_registros["defeito_normalizado"].notna()
obstrucao = triagem_registros["defeito_normalizado"].isin(defeitos_obstrucao)

triagem_registros["categoria_triagem"] = "outros registros para interpretação"
triagem_registros["motivo_triagem"] = "Sintoma fora da lista funcional inicial ou ausente."

triagem_registros.loc[sintoma_funcional, "categoria_triagem"] = "sintoma funcional sem defeito preenchido"
triagem_registros.loc[sintoma_funcional, "motivo_triagem"] = "Sintoma funcional registrado; o diagnóstico precisa de revisão."

triagem_registros.loc[sintoma_funcional & defeito_preenchido, "categoria_triagem"] = "sintoma funcional com defeito preenchido"
triagem_registros.loc[sintoma_funcional & defeito_preenchido, "motivo_triagem"] = "Sintoma e defeito documentados; a compatibilidade e a data precisam de revisão."

# A categoria de obstrução tem prioridade sobre as anteriores.
triagem_registros.loc[obstrucao, "categoria_triagem"] = "obstrução para interpretação"
triagem_registros.loc[obstrucao, "motivo_triagem"] = "A revisão deve distinguir impedimento externo de defeito da porta."

triagem_registros = triagem_registros.merge(
    candidatos_episodios[[
        "id_candidato", "qtd_alarmes_antes", "qtd_alarmes_depois", "grupo_evidencias",
    ]],
    on="id_candidato",
    how="left",
    validate="many_to_one",
)

assert len(triagem_registros) == len(evidencias_individuais)
assert set(triagem_registros["fonte"]) == {"ZM01", "ZM02"}
assert triagem_registros["categoria_triagem"].notna().all()


## 62. Participação de cada fonte e variantes do episódio

O resumo separa ZM01 e ZM02 e conta registros e candidatos distintos. Um candidato pode aparecer em mais de uma fonte ou categoria quando possui variantes de texto; por isso, a soma das linhas do resumo não representa necessariamente o total de episódios únicos.

A tabela de variantes destaca candidatos com classificações diferentes. A triagem preserva essas diferenças para revisão, sem escolher automaticamente o registro mais favorável à confirmação.


In [66]:
resumo_triagem = (
    triagem_registros.groupby(["fonte", "categoria_triagem"])
    .agg(
        registros=("id_candidato", "size"),
        candidatos_distintos=("id_candidato", "nunique"),
    )
    .reset_index()
)
display(resumo_triagem)

categorias_por_candidato = (
    triagem_registros.groupby("id_candidato")["categoria_triagem"].nunique()
)
ids_com_variantes = categorias_por_candidato.loc[categorias_por_candidato.gt(1)].index
variantes_para_revisar = triagem_registros.loc[
    triagem_registros["id_candidato"].isin(ids_com_variantes),
    ["id_candidato", "fonte", "nota", "sintoma", "defeito", "categoria_triagem"],
].sort_values(["id_candidato", "fonte"])

print("Candidatos com categorias diferentes:", len(ids_com_variantes))
display(variantes_para_revisar.head(12))


,fonte,categoria_triagem,registros,candidatos_distintos
0,ZM01,obstrução para interpretação,270,265
1,ZM01,outros registros para interpretação,98,94
2,ZM01,sintoma funcional com defeito preenchido,6216,6181
3,ZM01,sintoma funcional sem defeito preenchido,713,708
4,ZM02,obstrução para interpretação,5,5
5,ZM02,outros registros para interpretação,354,351
6,ZM02,sintoma funcional com defeito preenchido,211,210
7,ZM02,sintoma funcional sem defeito preenchido,15,15


Candidatos com categorias diferentes: 9


,id_candidato,fonte,nota,sintoma,defeito,categoria_triagem
1352,FRA|1|18|PDM|20251016T164500.000000,ZM01,14892406,LAÇO DE PSE ABERTO,ATUADOR DE PORTA AVARIADO,outros registros para interpretação
1353,FRA|1|18|PDM|20251016T164500.000000,ZM01,14892549,PDM NÃO FECHOU,<NA>,sintoma funcional sem defeito preenchido
1607,FRA|2|13|PDM|20250419T231800.000000,ZM01,14576764,PDM NÃO FECHOU,SENSOR FECHADO AVARIADO,sintoma funcional com defeito preenchido
1608,FRA|2|13|PDM|20250419T231800.000000,ZM01,14576768,PDM NÃO FECHOU,<NA>,sintoma funcional sem defeito preenchido
2085,FRE|2|4|PDM|20250831T064500.000000,ZM01,14809869,PDM NÃO FECHOU,<NA>,sintoma funcional sem defeito preenchido
2086,FRE|2|4|PDM|20250831T064500.000000,ZM01,14809974,PDM NÃO FECHOU,SENSOR FECHADO DESAJUSTADO,sintoma funcional com defeito preenchido
2567,HIG|1|15|PDM|20231202T072000.000000,ZM01,13640924,PDM NÃO FECHOU,SENSOR FECHADO AVARIADO,sintoma funcional com defeito preenchido
2568,HIG|1|15|PDM|20231202T072000.000000,ZM01,13641004,PDM NÃO FECHOU,<NA>,sintoma funcional sem defeito preenchido
4523,PIN|2|11|PDM|20240913T045100.000000,ZM01,14164863,PDM NÃO FECHOU,SENSOR ABERTO DESAJUSTADO,sintoma funcional com defeito preenchido
4524,PIN|2|11|PDM|20240913T045100.000000,ZM01,14164966,PDM NÃO FECHOU,<NA>,sintoma funcional sem defeito preenchido


## 63. Amostra de treino para avaliar o critério

A amostra seleciona até três candidatos por fonte e categoria, com semente fixa para reprodução. A seleção exige que as 24 horas posteriores à avaria também pertençam ao treino. Todas as variantes técnicas dos candidatos selecionados são exibidas.

A revisão deverá verificar se o sintoma corresponde à porta identificada, se o diagnóstico sustenta comprometimento funcional e se o início da avaria representa o início do episódio. Em ZM02, a data da intervenção programada não deve ser confundida com o início do defeito. A causa e a ação ajudam nessa interpretação.

Os alarmes próximos servem como apoio. Sua presença não é uma exigência de confirmação, e sua ausência não transforma o candidato em negativo. A tabela `revisoes_episodios`, na seção 59, continua reservada às decisões justificadas; esta triagem não a preenche automaticamente.


In [67]:
triagem_treino = triagem_registros.loc[
    triagem_registros["inicio_avaria"] + pd.Timedelta(hours=HORAS_DEPOIS_AVARIA)
    < INICIO_VALIDACAO
].copy()

candidatos_amostra = triagem_treino[
    ["fonte", "categoria_triagem", "id_candidato"]
].drop_duplicates()
candidatos_amostra = (
    candidatos_amostra.sample(frac=1, random_state=42)
    .groupby(["fonte", "categoria_triagem"], sort=False)
    .head(3)
)
amostra_revisao_criterios = triagem_treino.loc[
    triagem_treino["id_candidato"].isin(candidatos_amostra["id_candidato"]),
    [
        "id_candidato", "fonte", "nota", "ordem", "inicio_avaria",
        "sintoma", "objeto", "defeito", "causa", "acao",
        "categoria_triagem", "motivo_triagem",
        "qtd_alarmes_antes", "qtd_alarmes_depois",
    ],
].sort_values(["fonte", "categoria_triagem", "id_candidato"])

assert (
    amostra_revisao_criterios["inicio_avaria"]
    + pd.Timedelta(hours=HORAS_DEPOIS_AVARIA) < INICIO_VALIDACAO
).all()
assert set(amostra_revisao_criterios["fonte"]) == {"ZM01", "ZM02"}

print("Candidatos distintos na amostra:", amostra_revisao_criterios["id_candidato"].nunique())
with pd.option_context("display.max_rows", 100, "display.max_colwidth", 80):
    display(amostra_revisao_criterios)

print("A triagem não altera os rótulos de falha nem as entradas do modelo.")


Candidatos distintos na amostra: 24


,id_candidato,fonte,nota,ordem,inicio_avaria,sintoma,...,causa,acao,categoria_triagem,motivo_triagem,qtd_alarmes_antes,qtd_alarmes_depois
3244,LUZ|1|17|PDM|20240805T081700.000000,ZM01,140923,53106320.0,2024-08-05 08:17:00,ALAVANCA DE TRACAO PRESA,...,MAU USO,RESETADO E TESTADO,obstrução para interpretação,A revisão deve distinguir impedimento externo de defeito da porta.,2,2
3450,MBI|1|4|PDM|20240818T090800.000000,ZM01,14115702,53129500.0,2024-08-18 09:08:00,PDM NÃO FECHOU,...,OPERACAO INDEVIDA,RESETADO E TESTADO,obstrução para interpretação,A revisão deve distinguir impedimento externo de defeito da porta.,0,0
4357,PIN|2|2|PDM|20250414T130124.000000,ZM01,14568074,53580765.0,2025-04-14 13:01:24,PDM NÃO TRAVA FECHADA,...,MAU USO,ABERTO(A),obstrução para interpretação,A revisão deve distinguir impedimento externo de defeito da porta.,1,21
433,BUT|2|15|PDM|20240721T155300.000000,ZM01,14065493,53079807.0,2024-07-21 15:53:00,LAÇO DE PDM ABERTO SEM INDICACAO DO LOCA,...,DESGASTE,RESETADO E TESTADO,outros registros para interpretação,Sintoma fora da lista funcional inicial ou ausente.,0,0
434,BUT|2|15|PDM|20240721T155300.000000,ZM01,14065495,53079809.0,2024-07-21 15:53:00,LAÇO DE PDM ABERTO SEM INDICACAO DO LOCA,...,DESGASTE,SUBSTITUIDO(A),outros registros para interpretação,Sintoma fora da lista funcional inicial ou ausente.,0,0
1527,FRA|2|8|PDM|20240802T044600.000000,ZM01,14086893,53101136.0,2024-08-02 04:46:00,ATS SEM COMUNICAÇÃO COM PSD SERVER,...,ERRO DE COMUNICAÇÃO,PARAMETRIZACAO - PROGRAMACAO,outros registros para interpretação,Sintoma fora da lista funcional inicial ou ausente.,9,0
2110,FRE|2|6|PDM|20250403T212700.000000,ZM01,14548389,53561458.0,2025-04-03 21:27:00,INOPERANTE,...,ERRO DE COMUNICAÇÃO,RESETADO E TESTADO,outros registros para interpretação,Sintoma fora da lista funcional inicial ou ausente.,0,0
762,FAL|1|22|PDM|20240715T044200.000000,ZM01,14053578,53068043.0,2024-07-15 04:42:00,PDM NÃO FECHOU,...,ERRO DE COMUNICAÇÃO,RESETADO E TESTADO,sintoma funcional com defeito preenchido,Sintoma e defeito documentados; a compatibilidade e a data precisam de revisão.,1,12
3501,MBI|1|7|PDM|20240802T070800.000000,ZM01,14087017,53101228.0,2024-08-02 07:08:00,PDM NÃO FECHOU,...,ERRO DE SOFTWARE,RESETADO E TESTADO,sintoma funcional com defeito preenchido,Sintoma e defeito documentados; a compatibilidade e a data precisam de revisão.,11,10
5260,PTA|1|19|PDM|20240907T061500.000000,ZM01,14153203,53166697.0,2024-09-07 06:15:00,PDM NÃO ABRE,...,DESGASTE,SUBSTITUIDO(A),sintoma funcional com defeito preenchido,Sintoma e defeito documentados; a compatibilidade e a data precisam de revisão.,0,10


A triagem não altera os rótulos de falha nem as entradas do modelo.


### Resultado da triagem

ZM01 e ZM02 participam tanto do histórico de entradas quanto da análise documental de possíveis falhas. A triagem agora apresenta categorias, justificativas e uma amostra reproduzível para examinar o critério inicial.

A próxima etapa é revisar essa amostra e ajustar o critério, incluindo ocorrências repetidas e a confiabilidade da data de início. A definição dos negativos ainda depende do acompanhamento dos sete dias seguintes. O treinamento permanece pendente da consolidação desses rótulos.


## 64. Revisão documental dos 24 candidatos de treino

A leitura da amostra considera descrição, sintoma, objeto, defeito, causa e ação. O resultado abaixo registra uma **interpretação documental**, sem atribuir confirmação em campo ou validar o instante real de início.

A análise encontrou dois conflitos entre a descrição e a chave: a nota 14548389 menciona PDM03, mas está vinculada a FRE|2|6|PDM; a nota 14138889 menciona PDM02, mas está vinculada a MBI|1|6|PDM. Esses registros exigem resolução da identificação antes da atribuição de um alvo por porta.

Há também defeitos documentados em trolley, LED e alto-falante. Eles estão incluídos no escopo definido na seção 69, mesmo sem comprovação de perda de abertura ou fechamento. A identificação e as evidências do episódio ainda precisam ser consistentes.

A revisão é específica para os registros lidos. Ela não estende decisões para outros registros apenas porque compartilham um código de sintoma ou defeito. Nenhum registro inconclusivo recebe rótulo negativo.


In [68]:
# As conclusões se referem somente às notas efetivamente lidas na amostra.
pareceres_amostra = pd.DataFrame(
    [
    [
        "140923",
        "obstrução inconclusiva",
        "A descrição registra falha de fechamento; obstrução e mau uso não distinguem impedimento externo de defeito interno."
    ],
    [
        "14115702",
        "obstrução inconclusiva",
        "Obstrução de fechamento e operação indevida; o reset não esclarece a origem do impedimento."
    ],
    [
        "14568074",
        "obstrução inconclusiva",
        "Sintoma de trava com obstrução, mau uso e ação ABERTO(A); o diagnóstico permanece inconclusivo."
    ],
    [
        "14065493",
        "compatibilidade técnica pendente",
        "Há registro de falha de laço e trava desajustada; outra nota do mesmo candidato aponta sensor. A composição do episódio precisa de revisão."
    ],
    [
        "14065495",
        "compatibilidade técnica pendente",
        "Há sensor fechado desajustado e substituição; o mesmo candidato contém outra nota com diagnóstico e término diferentes."
    ],
    [
        "14086893",
        "escopo do equipamento pendente",
        "A descrição e o sintoma tratam de comunicação do sistema PSD/ATS; a atribuição à porta individual não está demonstrada."
    ],
    [
        "14548389",
        "identificação divergente",
        "A descrição menciona PDM03; a chave aponta FRE|2|6|PDM. O defeito OVPD também exige conferência do equipamento."
    ],
    [
        "14053578",
        "compatibilidade técnica pendente",
        "O sintoma é de fechamento, o objeto é sensor de porta aberta e o defeito é atuador desajustado. A combinação exige interpretação técnica."
    ],
    [
        "14087017",
        "defeito documentado",
        "A descrição menciona a PDM07, com falha de fechamento, MCP avariado e reset. O documento sustenta um defeito registrado, sem validar seu início real."
    ],
    [
        "14153203",
        "defeito documentado",
        "A PDM19 apresenta falha de abertura, sensor fechado avariado, desgaste e substituição registrados."
    ],
    [
        "14023973",
        "diagnóstico incompleto",
        "A descrição registra falha de fechamento da PDM20, mas objeto, defeito, causa e ação estão ausentes."
    ],
    [
        "13829284",
        "diagnóstico incompleto",
        "A descrição registra falha da trava da PDM5, mas faltam os campos de diagnóstico e ação."
    ],
    [
        "14392804",
        "diagnóstico incompleto",
        "A descrição registra falha da trava da PDM9, mas faltam os campos de diagnóstico e ação."
    ],
    [
        "14563897",
        "obstrução inconclusiva",
        "Trava desregulada, lubrificação insuficiente e ajuste sugerem problema mecânico; o código de obstrução não permite descartá-lo como externo."
    ],
    [
        "14308228",
        "obstrução inconclusiva",
        "Objeto estranho, falta de fixação e ajuste aparecem juntos; a origem do impedimento precisa de interpretação."
    ],
    [
        "13677211",
        "obstrução inconclusiva",
        "A descrição trata da fechadura e o sintoma de estrutura danificada; obstrução e reset não esclarecem o defeito."
    ],
    [
        "14320496",
        "defeito de componente documentado",
        "Trepidação, trolley avariado, desgaste e substituição sustentam defeito de componente; não comprovam interrupção de abertura ou fechamento."
    ],
    [
        "14320499",
        "defeito de componente documentado",
        "LED inoperante, LCB avariado e substituição sustentam defeito de sinalização; não comprovam interrupção do movimento."
    ],
    [
        "13558084",
        "defeito de componente documentado",
        "A descrição menciona speaker avariado; objeto alto-falante, baixa isolação e substituição sustentam defeito de componente."
    ],
    [
        "13693752",
        "defeito documentado",
        "Falha na trava, porta que não fechou, trava avariada e substituição sustentam defeito registrado."
    ],
    [
        "14257563",
        "compatibilidade técnica pendente",
        "A descrição e o defeito apontam sensor aberto, mas o objeto aponta atuador do sensor travado/destravado; a correspondência precisa de revisão."
    ],
    [
        "13890869",
        "compatibilidade técnica pendente",
        "Há fechamento parcial, objeto atuador, defeito AOU travado e ação ABERTO(A); a relação entre os campos precisa de revisão."
    ],
    [
        "14138889",
        "identificação divergente",
        "A descrição menciona PDM02, enquanto a chave aponta MBI|1|6|PDM; os campos de diagnóstico também estão ausentes."
    ],
    [
        "13876261",
        "diagnóstico incompleto",
        "A descrição solicita substituir sensor de porta aberta; sem diagnóstico e ação, a realização e o defeito permanecem sem confirmação documental suficiente."
    ],
    [
        "13944186",
        "diagnóstico incompleto",
        "A descrição registra falha da trava da PDM10, mas faltam os campos de diagnóstico e ação."
    ]
],
    columns=["nota_revisao", "resultado_documental", "justificativa_documental"],
)
assert pareceres_amostra["nota_revisao"].is_unique

revisao_documental_amostra = triagem_treino.loc[
    triagem_treino["id_candidato"].isin(amostra_revisao_criterios["id_candidato"]),
].copy()
revisao_documental_amostra["nota_revisao"] = (
    pd.to_numeric(revisao_documental_amostra["nota"], errors="raise")
    .astype("Int64").astype("string")
)
revisao_documental_amostra = revisao_documental_amostra.merge(
    pareceres_amostra, on="nota_revisao", how="left", validate="many_to_one"
)

assert revisao_documental_amostra["resultado_documental"].notna().all()
assert set(revisao_documental_amostra["nota_revisao"]) == set(pareceres_amostra["nota_revisao"])
assert revisao_documental_amostra["id_candidato"].nunique() == 24
assert (
    revisao_documental_amostra["inicio_avaria"]
    + pd.Timedelta(hours=HORAS_DEPOIS_AVARIA) < INICIO_VALIDACAO
).all()

resumo_revisao_documental = (
    revisao_documental_amostra.groupby(["fonte", "resultado_documental"])
    ["id_candidato"].nunique().unstack("fonte", fill_value=0)
)
display(resumo_revisao_documental)

# A apresentação mantém descrição, diagnóstico e conclusão visíveis.
with pd.option_context("display.max_columns", None, "display.max_rows", 100, "display.max_colwidth", 120):
    display(revisao_documental_amostra[[
        "id_candidato", "fonte", "nota", "descricao", "sintoma",
        "objeto", "defeito", "acao", "resultado_documental", "justificativa_documental",
    ]].sort_values(["resultado_documental", "fonte", "id_candidato"]))


fonte,ZM01,ZM02
resultado_documental,,
compatibilidade técnica pendente,2,2
defeito de componente documentado,0,3
defeito documentado,2,1
diagnóstico incompleto,3,2
escopo do equipamento pendente,1,0
identificação divergente,1,1
obstrução inconclusiva,3,3


,id_candidato,fonte,nota,descricao,sintoma,objeto,defeito,acao,resultado_documental,justificativa_documental
0,BUT|2|15|PDM|20240721T155300.000000,ZM01,14065493,Fallha de laço de PSD,LAÇO DE PDM ABERTO SEM INDICACAO DO LOCA,ATUADOR DE PORTA,TRAVA DESAJUSTADA,RESETADO E TESTADO,compatibilidade técnica pendente,Há registro de falha de laço e trava desajustada; outra nota do mesmo candidato aponta sensor. A composição do episó...
1,BUT|2|15|PDM|20240721T155300.000000,ZM01,14065495,Falha no fechamento,LAÇO DE PDM ABERTO SEM INDICACAO DO LOCA,SENSOR PORTA FECHADA-HEADER BOX,SENSOR FECHADO DESAJUSTADO,SUBSTITUIDO(A),compatibilidade técnica pendente,Há sensor fechado desajustado e substituição; o mesmo candidato contém outra nota com diagnóstico e término diferentes.
2,FAL|1|22|PDM|20240715T044200.000000,ZM01,14053578,PDM22 - Falha de fechamento,PDM NÃO FECHOU,SENSOR PORTA ABERTA-HEADER BOX,ATUADOR DE PORTA DESAJUSTADO,RESETADO E TESTADO,compatibilidade técnica pendente,"O sintoma é de fechamento, o objeto é sensor de porta aberta e o defeito é atuador desajustado. A combinação exige i..."
20,MBI|1|10|PDM|20241101T090000.000000,ZM02,14257563,Sensor aberto PDM 10 V1,PDM NÃO ABRE,ATUADOR DO SENSOR TRAVADO/DESTRAVADO-PSD,SENSOR ABERTO AVARIADO,SUBSTITUIDO(A),compatibilidade técnica pendente,"A descrição e o defeito apontam sensor aberto, mas o objeto aponta atuador do sensor travado/destravado; a correspon..."
22,REP|1|4|PDM|20240416T090000.000000,ZM02,13890869,Falha no fechamento da PDM-04 - PL0-01.,PDM FECHA PARCIALMENTE,ATUADOR DE PORTA,AOU TRAVADO,ABERTO(A),compatibilidade técnica pendente,"Há fechamento parcial, objeto atuador, defeito AOU travado e ação ABERTO(A); a relação entre os campos precisa de re..."
15,FRE|1|9|PDM|20241205T033000.000000,ZM02,14320496,PDM 09 - Porta com trepidação.,PORTA COM RUÍDO ANORMAL,TROLLEY DA PORTA,TROLLEY AVARIADO,SUBSTITUIDO(A),defeito de componente documentado,"Trepidação, trolley avariado, desgaste e substituição sustentam defeito de componente; não comprovam interrupção de ..."
16,FRE|2|15|PDM|20241205T040000.000000,ZM02,14320499,PDM 15 - Led inoperante,LAMPADA DE SINALIZAÇÃO DE PDM NAO ACENDE,LED -HEADER BOX,LCB AVARIADO,SUBSTITUIDO(A),defeito de componente documentado,"LED inoperante, LCB avariado e substituição sustentam defeito de sinalização; não comprovam interrupção do movimento."
17,HIG|2|6|PDM|20231020T010000.000000,ZM02,13558084,PLT2 PDM 06 - SPEAKER AVARIADO,PORTA COM RUÍDO ANORMAL,ALTO-FALANTE (SPEAKER),BAIXA ISOLACAO,SUBSTITUIDO(A),defeito de componente documentado,"A descrição menciona speaker avariado; objeto alto-falante, baixa isolação e substituição sustentam defeito de compo..."
7,MBI|1|7|PDM|20240802T070800.000000,ZM01,14087017,PDM 07 - Falha no fechamento,PDM NÃO FECHOU,MODULO DE CONTROLE DE PORTAS,MCP AVARIADO,RESETADO E TESTADO,defeito documentado,"A descrição menciona a PDM07, com falha de fechamento, MCP avariado e reset. O documento sustenta um defeito registr..."
11,PTA|1|19|PDM|20240907T061500.000000,ZM01,14153203,PDM19 - Falha na abertura,PDM NÃO ABRE,SENSOR PORTA FECHADA-HEADER BOX,SENSOR FECHADO AVARIADO,SUBSTITUIDO(A),defeito documentado,"A PDM19 apresenta falha de abertura, sensor fechado avariado, desgaste e substituição registrados."


## 65. Verificação das portas citadas na descrição da amostra

A conferência abaixo extrai somente referências explícitas como PDM03, PDM 03 ou PDM-03. Ela compara o número citado com o número da chave, sem tentar inferir uma porta quando o texto não a identifica.

O texto livre também pode conter erro. Uma divergência não autoriza substituir a chave automaticamente. A ausência de número na descrição tampouco comprova que a chave está correta.


In [69]:
conferencia_portas_amostra = revisao_documental_amostra[[
    "id_candidato", "fonte", "nota", "descricao", "porta",
]].copy()
conferencia_portas_amostra["porta_na_descricao"] = pd.to_numeric(
    conferencia_portas_amostra["descricao"].astype("string").str.upper()
    .str.extract(r"\bPDM[\s-]*(\d+)\b", expand=False),
    errors="coerce",
).astype("Int64")
conferencia_portas_amostra["porta_na_chave"] = pd.to_numeric(
    conferencia_portas_amostra["porta"], errors="raise"
).astype("Int64")
conferencia_portas_amostra["porta_divergente"] = (
    conferencia_portas_amostra["porta_na_descricao"].notna()
    & conferencia_portas_amostra["porta_na_descricao"].ne(
        conferencia_portas_amostra["porta_na_chave"]
    )
).fillna(False)

divergencias_portas_amostra = conferencia_portas_amostra.loc[
    conferencia_portas_amostra["porta_divergente"]
].copy()
assert set(divergencias_portas_amostra["id_candidato"]) == set(
    revisao_documental_amostra.loc[
        revisao_documental_amostra["resultado_documental"].eq("identificação divergente"),
        "id_candidato",
    ]
)
display(divergencias_portas_amostra)
print("Candidatos com número de porta divergente:", divergencias_portas_amostra["id_candidato"].nunique())


,id_candidato,fonte,nota,descricao,porta,porta_na_descricao,porta_na_chave,porta_divergente
4,FRE|2|6|PDM|20250403T212700.000000,ZM01,14548389,PDM03 - Falha de abertura,6,3,6,True
18,MBI|1|6|PDM|20240830T120000.000000,ZM02,14138889,PDM02 - PLT01 - Falha na abertura,6,2,6,True


Candidatos com número de porta divergente: 2


### Resultado da revisão documental

A amostra contém seis candidatos com defeito documentado: três associados a falha funcional e três a componentes de sinalização ou movimento. Os demais exigem esclarecimento de diagnóstico, compatibilidade técnica, obstrução ou identificação. Essas conclusões descrevem o conteúdo das planilhas; não constituem validação de campo.

A revisão mantém os cortes de treino, validação e teste. Nenhuma decisão da amostra foi obtida a partir de exemplos de validação ou teste nesta etapa. Diagnósticos e ações de manutenção permanecem evidências para o alvo, sem serem adicionados como entradas anteriores à falha.

A abrangência dos defeitos está definida na seção 69. Para consolidar `falha_7d`, ainda é necessário resolver identificação e episódios repetidos e estabelecer o acompanhamento que permite atribuir zero. A data `inicio_avaria` é a referência disponível no documento; sua equivalência ao início real permanece uma hipótese a verificar.


## 66. Datas de término e registros possivelmente repetidos

Um registro ZM01 e outro ZM02 podem descrever etapas da mesma ocorrência. Antes da contagem de falhas, a conferência procura novos registros da mesma porta iniciados durante um intervalo de avaria já registrado.

A análise utiliza somente candidatos do treino. Os términos iguais ou posteriores ao início da validação não são utilizados para estabelecer sobreposições nesta etapa. Términos ausentes ou anteriores ao início também não recebem preenchimento artificial.

O término de avaria disponível na planilha é uma referência documental. Ele não comprova, sozinho, o instante de restabelecimento da porta. Uma sobreposição indica um caso para revisão: pode haver continuidade do defeito, registros redundantes ou defeitos diferentes ocorrendo ao mesmo tempo.


In [70]:
colunas_intervalo = [
    "nota", "ordem", "local_de_instalacao",
    "inicio_avaria_datetime", "fim_avaria_datetime",
]
registros_intervalos = pd.concat([
    fonte_zm01_alvo[colunas_intervalo].assign(fonte="ZM01"),
    bases["ZM02"][colunas_intervalo].assign(fonte="ZM02"),
], ignore_index=True)

chaves_intervalos = extrair_ativos(registros_intervalos["local_de_instalacao"])
registros_intervalos["chave_candidata"] = chaves_intervalos["chave_candidata"]
registros_intervalos["inicio_avaria"] = pd.to_datetime(
    registros_intervalos["inicio_avaria_datetime"], errors="coerce"
)
registros_intervalos["fim_avaria"] = pd.to_datetime(
    registros_intervalos["fim_avaria_datetime"], errors="coerce"
)
registros_intervalos["id_candidato"] = (
    registros_intervalos["chave_candidata"] + "|"
    + registros_intervalos["inicio_avaria"].dt.strftime("%Y%m%dT%H%M%S.%f")
)

ids_intervalos_treino = candidatos_episodios.loc[
    candidatos_episodios["inicio_avaria"] < INICIO_VALIDACAO, "id_candidato"
]
intervalos_treino = registros_intervalos.loc[
    registros_intervalos["id_candidato"].isin(ids_intervalos_treino)
].copy()

intervalos_treino["situacao_fim"] = "término utilizável no treino"
intervalos_treino.loc[
    intervalos_treino["fim_avaria"].isna(), "situacao_fim"
] = "término ausente"
intervalos_treino.loc[
    intervalos_treino["fim_avaria"] < intervalos_treino["inicio_avaria"], "situacao_fim"
] = "término anterior ao início"
intervalos_treino.loc[
    intervalos_treino["fim_avaria"] >= INICIO_VALIDACAO, "situacao_fim"
] = "término fora do treino"

intervalos_treino["fim_utilizavel"] = intervalos_treino["fim_avaria"].where(
    intervalos_treino["situacao_fim"].eq("término utilizável no treino")
)
assert (intervalos_treino["inicio_avaria"] < INICIO_VALIDACAO).all()
assert (intervalos_treino["fim_utilizavel"].dropna() < INICIO_VALIDACAO).all()

display(pd.crosstab(intervalos_treino["situacao_fim"], intervalos_treino["fonte"]))


fonte,ZM01,ZM02
situacao_fim,,
término ausente,108,13
término fora do treino,0,10
término utilizável no treino,4881,342


## 67. Candidatos com intervalos sobrepostos

Para cada candidato, o resumo conserva a quantidade de registros e de fontes, além da quantidade de términos distintos. Quando há términos diferentes, o maior término utilizável serve apenas para ampliar a busca de possíveis sobreposições; ele não é escolhido como término definitivo do episódio.

O cruzamento exige a mesma chave da porta. Cada par aparece uma única vez, com o início mais antigo à esquerda. Inícios exatamente iguais já pertencem ao mesmo candidato e são identificados separadamente pelo número de registros.

O uso de `<=` também sinaliza intervalos que se encontram no mesmo instante, pois a precisão das datas não permite concluir automaticamente se houve recuperação entre eles.


In [71]:
resumo_intervalos_treino = (
    intervalos_treino.groupby(["id_candidato", "chave_candidata", "inicio_avaria"])
    .agg(
        qtd_registros=("fonte", "size"),
        qtd_fontes=("fonte", "nunique"),
        qtd_fins_utilizaveis=("fim_utilizavel", "count"),
        qtd_fins_distintos=("fim_utilizavel", "nunique"),
        maior_fim_utilizavel=("fim_utilizavel", "max"),
    )
    .reset_index()
)
resumo_intervalos_treino["tem_fim_pendente"] = (
    resumo_intervalos_treino["qtd_fins_utilizaveis"]
    < resumo_intervalos_treino["qtd_registros"]
)

inicio_dos_intervalos = resumo_intervalos_treino[[
    "id_candidato", "chave_candidata", "inicio_avaria",
    "maior_fim_utilizavel", "qtd_fins_distintos", "tem_fim_pendente",
]].rename(columns={
    "id_candidato": "id_anterior",
    "inicio_avaria": "inicio_anterior",
})

novos_inicios = resumo_intervalos_treino[[
    "id_candidato", "chave_candidata", "inicio_avaria",
]].rename(columns={
    "id_candidato": "id_seguinte",
    "inicio_avaria": "inicio_seguinte",
})

pares_intervalos_treino = inicio_dos_intervalos.merge(
    novos_inicios, on="chave_candidata", how="inner", validate="many_to_many"
)
sobreposicoes_treino = pares_intervalos_treino.loc[
    pares_intervalos_treino["inicio_seguinte"].gt(pares_intervalos_treino["inicio_anterior"])
    & pares_intervalos_treino["inicio_seguinte"].le(pares_intervalos_treino["maior_fim_utilizavel"])
].copy()
del pares_intervalos_treino

assert not sobreposicoes_treino.duplicated(["id_anterior", "id_seguinte"]).any()
assert (sobreposicoes_treino["inicio_seguinte"] < INICIO_VALIDACAO).all()

ids_sobrepostos = pd.concat([
    sobreposicoes_treino["id_anterior"],
    sobreposicoes_treino["id_seguinte"],
]).unique()
resumo_intervalos_treino["tem_sobreposicao_documental"] = (
    resumo_intervalos_treino["id_candidato"].isin(ids_sobrepostos)
)

print("Candidatos analisados no treino:", len(resumo_intervalos_treino))
print("Candidatos com mais de um registro no mesmo início:",
      resumo_intervalos_treino["qtd_registros"].gt(1).sum())
print("Candidatos com ZM01 e ZM02 no mesmo início:",
      resumo_intervalos_treino["qtd_fontes"].gt(1).sum())
print("Pares com intervalos sobrepostos:", len(sobreposicoes_treino))
print("Candidatos envolvidos nas sobreposições:", len(ids_sobrepostos))
print("Candidatos com algum término pendente:",
      resumo_intervalos_treino["tem_fim_pendente"].sum())

display(sobreposicoes_treino.sort_values(["chave_candidata", "inicio_anterior"]).head(10))


Candidatos analisados no treino: 5306
Candidatos com mais de um registro no mesmo início: 42
Candidatos com ZM01 e ZM02 no mesmo início: 5
Pares com intervalos sobrepostos: 412
Candidatos envolvidos nas sobreposições: 746
Candidatos com algum término pendente: 127


,id_anterior,chave_candidata,inicio_anterior,maior_fim_utilizavel,qtd_fins_distintos,tem_fim_pendente,id_seguinte,inicio_seguinte
97,BUT|1|10|PDM|20231209T150000.000000,BUT|1|10|PDM,2023-12-09 15:00:00,2024-01-29 11:00:00,1,False,BUT|1|10|PDM|20231220T141800.000000,2023-12-20 14:18:00
1495,BUT|1|18|PDM|20240708T142000.000000,BUT|1|18|PDM,2024-07-08 14:20:00,2024-07-08 16:30:00,1,False,BUT|1|18|PDM|20240708T155400.000000,2024-07-08 15:54:00
3382,BUT|2|4|PDM|20250210T223000.000000,BUT|2|4|PDM,2025-02-10 22:30:00,2025-04-02 03:00:00,1,False,BUT|2|4|PDM|20250221T185600.000000,2025-02-21 18:56:00
3383,BUT|2|4|PDM|20250210T223000.000000,BUT|2|4|PDM,2025-02-10 22:30:00,2025-04-02 03:00:00,1,False,BUT|2|4|PDM|20250320T165100.000000,2025-03-20 16:51:00
3982,FAL|1|13|PDM|20240312T133700.000000,FAL|1|13|PDM,2024-03-12 13:37:00,2024-03-12 16:40:36,1,False,FAL|1|13|PDM|20240312T141519.000000,2024-03-12 14:15:19
4327,FAL|1|14|PDM|20250104T214400.000000,FAL|1|14|PDM,2025-01-04 21:44:00,2025-01-04 22:30:00,1,False,FAL|1|14|PDM|20250104T214445.000000,2025-01-04 21:44:45
4642,FAL|1|15|PDM|20231129T020000.000000,FAL|1|15|PDM,2023-11-29 02:00:00,2023-12-26 10:00:00,1,False,FAL|1|15|PDM|20231212T061000.000000,2023-12-12 06:10:00
4643,FAL|1|15|PDM|20231129T020000.000000,FAL|1|15|PDM,2023-11-29 02:00:00,2023-12-26 10:00:00,1,False,FAL|1|15|PDM|20231222T151400.000000,2023-12-22 15:14:00
4644,FAL|1|15|PDM|20231129T020000.000000,FAL|1|15|PDM,2023-11-29 02:00:00,2023-12-26 10:00:00,1,False,FAL|1|15|PDM|20231222T152320.000000,2023-12-22 15:23:20
4645,FAL|1|15|PDM|20231129T020000.000000,FAL|1|15|PDM,2023-11-29 02:00:00,2023-12-26 10:00:00,1,False,FAL|1|15|PDM|20231222T183231.000000,2023-12-22 18:32:31


## 68. Efeito das sobreposições na amostra revisada

A junção abaixo acrescenta a situação dos intervalos aos 24 candidatos já lidos. Isso permite conferir se um defeito documentado também apresenta dúvidas sobre a separação do episódio.

A ausência de sobreposição detectada não confirma que se trata de uma falha nova: términos ausentes, falhas anteriores ao recorte e erros de registro podem ocultar continuidades. Nenhum candidato é descartado ou unido automaticamente nesta conferência.


In [72]:
amostra_com_intervalos = revisao_documental_amostra.merge(
    resumo_intervalos_treino[[
        "id_candidato", "qtd_registros", "qtd_fontes",
        "qtd_fins_distintos", "tem_fim_pendente", "tem_sobreposicao_documental",
    ]],
    on="id_candidato", how="left", validate="many_to_one",
)
assert len(amostra_com_intervalos) == len(revisao_documental_amostra)
assert amostra_com_intervalos["qtd_registros"].notna().all()

with pd.option_context("display.max_columns", None, "display.max_rows", 100):
    display(amostra_com_intervalos[[
        "id_candidato", "fonte", "nota", "resultado_documental",
        "qtd_registros", "qtd_fontes", "qtd_fins_distintos",
        "tem_fim_pendente", "tem_sobreposicao_documental",
    ]])

print("Candidatos da amostra com sobreposição:",
      amostra_com_intervalos.loc[
          amostra_com_intervalos["tem_sobreposicao_documental"], "id_candidato"
      ].nunique())


,id_candidato,fonte,nota,resultado_documental,qtd_registros,qtd_fontes,qtd_fins_distintos,tem_fim_pendente,tem_sobreposicao_documental
0,BUT|2|15|PDM|20240721T155300.000000,ZM01,14065493,compatibilidade técnica pendente,2,1,2,False,False
1,BUT|2|15|PDM|20240721T155300.000000,ZM01,14065495,compatibilidade técnica pendente,2,1,2,False,False
2,FAL|1|22|PDM|20240715T044200.000000,ZM01,14053578,compatibilidade técnica pendente,1,1,1,False,False
3,FRA|2|8|PDM|20240802T044600.000000,ZM01,14086893,escopo do equipamento pendente,1,1,1,False,False
4,FRE|2|6|PDM|20250403T212700.000000,ZM01,14548389,identificação divergente,1,1,1,False,False
5,LUZ|1|17|PDM|20240805T081700.000000,ZM01,140923,obstrução inconclusiva,1,1,1,False,False
6,MBI|1|4|PDM|20240818T090800.000000,ZM01,14115702,obstrução inconclusiva,1,1,1,False,False
7,MBI|1|7|PDM|20240802T070800.000000,ZM01,14087017,defeito documentado,1,1,1,False,True
8,PIN|2|2|PDM|20250414T130124.000000,ZM01,14568074,obstrução inconclusiva,1,1,1,False,False
9,PIN|2|5|PDM|20240315T085332.000000,ZM01,13829284,diagnóstico incompleto,1,1,1,False,False


Candidatos da amostra com sobreposição: 2


### Resultado da conferência temporal

A análise agora sinaliza registros com o mesmo início, intervalos sobrepostos e términos pendentes, reunindo ZM01 e ZM02. A contagem de linhas de manutenção não equivale à contagem de novas falhas.

Essas tabelas servem à consolidação do alvo e não foram adicionadas às entradas do modelo, pois usam términos conhecidos posteriormente. Os cortes cronológicos permanecem iguais. Defeitos de componentes como LED e alto-falante estão incluídos na definição da seção 69. A regra de acompanhamento suficiente para atribuir negativos ainda depende da consolidação dos rótulos.


## 69. Definição de falha adotada no projeto

**Considera-se falha uma ocorrência de perda ou comprometimento de função da porta, ou um defeito em um de seus componentes que justifique manutenção corretiva, mesmo que a porta continue abrindo e fechando.**

A definição inclui defeitos de componentes conforme a decisão de escopo do projeto. Esta seção passa a ser a referência para a consolidação do alvo. As triagens anteriores permanecem como análises exploratórias; suas listas de sintomas não limitam o conceito de falha.

### Situações incluídas

| Situação | Exemplos |
| --- | --- |
| Comprometimento do movimento | Porta que não abre, não fecha, abre parcialmente ou apresenta dificuldade de movimento por defeito. |
| Comprometimento da trava ou dos sensores | Trava avariada, sensor defeituoso ou desajustado, indicação incorreta de porta aberta ou fechada. |
| Defeito mecânico ou estrutural | Trolley, correia, atuador, motor, fechadura ou estrutura com dano ou desajuste que justifique correção. |
| Defeito elétrico, eletrônico ou de controle | Componente avariado, defeito de isolação, configuração inadequada ou falha de comunicação atribuível à porta ou a seus componentes. |
| Defeito de sinalização ou aviso | LED, lâmpada ou alto-falante defeituoso, mesmo com abertura e fechamento disponíveis. |

Um defeito não precisa causar parada total, gerar alarme ou já ter sido reparado para fazer parte do escopo. A substituição de uma peça, por si só, não comprova defeito: o motivo registrado também precisa sustentar a necessidade de correção.

### Evidências e fontes

**ZM01 e ZM02 recebem o mesmo critério de inclusão.** A classificação como corretiva emergencial ou programada não determina, isoladamente, a existência de falha. A avaliação considera em conjunto descrição, sintoma, objeto, defeito, causa e ação, vinculados à porta correta.

Os alarmes servem como evidência complementar. Proximidade temporal não comprova causa, e ausência de alarme não exclui um defeito documentado. A marcação SAP também não substitui a análise dos registros.

ZM03 permanece como contexto de manutenção preventiva. A existência de uma programação preventiva não gera um episódio de falha.

### Situações que não geram uma nova falha automaticamente

| Situação | Tratamento |
| --- | --- |
| Inspeção, prevenção ou troca programada sem defeito identificado | Não caracteriza falha por si só. |
| Obstrução exclusivamente externa, com evidência de que não há defeito na porta | Fica fora da definição de defeito da porta. Se a origem não estiver esclarecida ou houver dano, permanece para análise. |
| Ruído ou outro sintoma sem diagnóstico suficiente | Permanece inconclusivo; não recebe automaticamente confirmação nem rejeição. |
| Falha de sistema ou de equipamento coletivo sem atribuição individual confiável | Não é distribuída automaticamente para todas as portas. |
| Divergência de porta, diagnóstico conflitante ou informação insuficiente | Permanece pendente de esclarecimento. |
| Vários registros sobre o mesmo episódio | Representam um único início de falha após a consolidação, e não uma falha por nota ou ordem. |

A rejeição de um registro como episódio não demonstra ausência de outras falhas no horizonte de previsão.

### Definição do alvo temporal

O alvo permanece **“a porta apresentará uma nova falha ou um novo defeito em seus componentes nos próximos sete dias?”**, com previsão ao final do dia D.

- **1:** existe pelo menos um início de episódio que atende à definição entre D+1 e D+7, inclusive.
- **0:** não existe início de episódio nesse intervalo e há acompanhamento suficiente e confiável para sustentar essa ausência.
- **Ausente:** as informações não permitem decidir, incluindo horizonte incompleto ou ambiguidades relevantes para a porta e o período.

O dia D não pertence ao horizonte do alvo. A continuidade de um episódio iniciado em D ou antes não é contada como um novo início nos dias seguintes. Uma nova ocorrência distinta pode gerar positivo, mesmo que outro episódio já esteja em andamento.

A data de referência é o início do episódio, não a data da intervenção programada, da substituição da peça ou do encerramento da ordem. O campo `inicio_avaria` fornece a referência documental disponível; casos em que ele não representa confiavelmente o início precisam de tratamento na consolidação.

### Encerramento da etapa 1

**A abrangência da falha está definida: inclui falhas funcionais e defeitos de componentes da porta.** Os exemplos de trolley, LED e alto-falante encontrados na amostra estão dentro desse escopo, desde que a identificação e as evidências do episódio sejam consistentes.

A definição do conceito não equivale à validação de todos os registros. A próxima etapa é aplicar esse critério à consolidação dos episódios e dos rótulos, resolvendo duplicidades, datas, identificações e acompanhamento dos negativos. O campo `falha_7d` e os cortes cronológicos não são alterados nesta etapa.


## 70. Consolidação documental de ZM01 e ZM02

A definição da seção 69 é aplicada a ZM01 e ZM02, incluindo componentes da porta. Os positivos desta etapa representam **falhas sustentadas por diagnóstico documental**, com a data de início declarada na avaria. Isso não equivale a uma validação de campo.

O texto longo e os identificadores originais permanecem disponíveis para auditoria. Registros com a mesma porta e o mesmo instante de início formam um episódio documental. Inícios diferentes com intervalos sobrepostos são agrupados somente como **grupo temporal incerto**, sem presumir que todos correspondam a uma única falha.

As regras abaixo são fixas e conservadoras: exigem componente e defeito compatíveis, sintoma ou descrição disponível e identificação sem conflito detectado. Defeitos de LED, alto-falante e outros componentes são elegíveis. Combinações não contempladas ficam pendentes, sem receber zero.

As regras não foram escolhidas para melhorar métricas de validação ou teste. O campo `falha_flag` e a proximidade de alarmes não são condições obrigatórias. A execução de uma manutenção também não é condição obrigatória; ordens canceladas ficam pendentes para interpretação.


In [73]:
campos_consolidacao = [
    "nota", "ordem", "linha_origem", "local_de_instalacao",
    "inicio_avaria_datetime", "fim_avaria_datetime",
    "descricao", "sintoma", "objeto", "defeito", "causa", "acao",
    "texto_longo_da_ordem", "status_do_sistema_da_ordem", "status_do_usuario_da_ordem",
]
registros_consolidacao = pd.concat([
    fonte_zm01_alvo.reindex(columns=campos_consolidacao).assign(fonte="ZM01"),
    bases["ZM02"].reindex(columns=campos_consolidacao).assign(fonte="ZM02"),
], ignore_index=True)
registros_consolidacao["id_registro"] = registros_consolidacao.index
registros_consolidacao = registros_consolidacao.join(
    extrair_ativos(registros_consolidacao["local_de_instalacao"])[colunas_identificacao]
)
for campo in ["inicio_avaria", "fim_avaria"]:
    registros_consolidacao[campo] = pd.to_datetime(
        registros_consolidacao[campo + "_datetime"], errors="coerce"
    )
for campo in ["descricao", "sintoma", "objeto", "defeito", "texto_longo_da_ordem"]:
    registros_consolidacao[campo] = (
        registros_consolidacao[campo].astype("string").str.strip().replace("", pd.NA)
    )
registros_consolidacao["id_candidato"] = (
    registros_consolidacao["chave_candidata"] + "|"
    + registros_consolidacao["inicio_avaria"].dt.strftime("%Y%m%dT%H%M%S.%f")
)

# O exame alcança o histórico anterior ao recorte para detectar continuidades.
registros_consolidacao = registros_consolidacao.loc[
    registros_consolidacao["inicio_avaria"].isna()
    | (registros_consolidacao["inicio_avaria"] < DATA_FIM + pd.Timedelta(days=1))
].copy()

# Referências explícitas na descrição e no texto longo são confrontadas com a chave.
registros_consolidacao["conflito_porta"] = False
registros_consolidacao["conflito_plataforma"] = False
for campo in ["descricao", "texto_longo_da_ordem"]:
    texto_identificacao = registros_consolidacao[campo].str.upper().fillna("")
    referencias_porta = texto_identificacao.str.findall(r"\bPDM[\s-]*(\d+)\b").explode()
    referencias_porta = pd.to_numeric(referencias_porta, errors="coerce")
    porta_da_chave = pd.to_numeric(
        registros_consolidacao.loc[referencias_porta.index, "porta"], errors="coerce"
    )
    conflito = referencias_porta.notna() & referencias_porta.ne(porta_da_chave)
    conflito = conflito.groupby(level=0).any()
    registros_consolidacao["conflito_porta"] |= conflito.reindex(
        registros_consolidacao.index, fill_value=False
    )

    referencias_plt = texto_identificacao.str.findall(r"\bPLT[\s-]*(\d+)\b").explode()
    referencias_plt = pd.to_numeric(referencias_plt, errors="coerce")
    plt_da_chave = pd.to_numeric(
        registros_consolidacao.loc[referencias_plt.index, "plataforma"], errors="coerce"
    )
    conflito = referencias_plt.notna() & referencias_plt.ne(plt_da_chave)
    registros_consolidacao["conflito_plataforma"] |= conflito.groupby(level=0).any().reindex(
        registros_consolidacao.index, fill_value=False
    )

# Uma nota ou ordem associada a locais ou inícios diferentes não é resolvida por descarte.
registros_consolidacao["conflito_identificador"] = False
for campo in ["nota", "ordem"]:
    variacoes_identificador = registros_consolidacao.groupby(campo)[[
        "local_de_instalacao", "inicio_avaria",
    ]].nunique(dropna=False)
    ids_divergentes = variacoes_identificador.index[
        variacoes_identificador.gt(1).any(axis=1)
    ]
    registros_consolidacao["conflito_identificador"] |= registros_consolidacao[campo].isin(ids_divergentes)

status_ordem = (
    registros_consolidacao["status_do_sistema_da_ordem"].astype("string").fillna("")
    + " " + registros_consolidacao["status_do_usuario_da_ordem"].astype("string").fillna("")
).str.upper()
registros_consolidacao["cancelamento_pendente"] = status_ordem.str.contains(
    r"\b(?:CANC|CANCELADA|CANCELADO)\b", regex=True
)
registros_consolidacao["identificacao_pendente"] = registros_consolidacao[[
    "conflito_porta", "conflito_plataforma", "conflito_identificador",
]].any(axis=1)


## 71. Critérios de diagnóstico compatível

Cada linha da tabela liga uma família de componentes a diagnósticos compatíveis presentes no catálogo. As expressões são aplicadas aos campos completos de objeto e defeito; palavras isoladas no texto livre não confirmam falhas.

Este primeiro conjunto não cobre todas as combinações possíveis. Ausência de correspondência significa **pendência de revisão**, não ausência de defeito. Obstruções, ruído isolado e atribuições a equipamentos coletivos não são aprovados por essas regras.

Quando o mesmo episódio contém variantes de objeto ou defeito, a decisão permanece pendente. Campos ausentes em uma nota complementar podem ser supridos por outra nota do mesmo episódio, desde que não haja valores conflitantes.


In [74]:
regras_diagnostico = pd.DataFrame([
    ("sensor fechado", r"SENSOR PORTA FECHADA-HEADER BOX", r"SENSOR FECHADO (?:AVARIADO|DESAJUSTADO)"),
    ("sensor aberto", r"SENSOR PORTA ABERTA-HEADER BOX", r"SENSOR ABERTO (?:AVARIADO|DESAJUSTADO)"),
    ("sensor travado", r"SENSOR PORTA TRAVADA-HEADER BOX", r"SENSOR TRAVADO (?:AVARIADO|DESAJUSTADO)"),
    ("sensor destravado", r"SENSOR PORTA DESTRAVADA-HEADER BOX", r"SENSOR DESTRAVADO (?:AVARIADO|DESAJUSTADO)"),
    ("controle da porta", r"(?:MODULO DE CONTROLE DE PORTAS|DONGLE PROGRAMAVEL DO MCP)",
     r"(?:MCP (?:AVARIADO|TRAVADO|DESCONFIGURADO)|PERDA DE COMUNICACAO)"),
    ("trava e fechadura", r"(?:TRAVA DA PORTA|TRAVA EUCHNER PDM|TRAVA DE SEGURANCA PDM|MECANISMO DE FECHADURA DA PDM|FECHADURA DA PORTA|LINGUETA DA TRAVA PDM|SOLENOIDE DA TRAVA PDM)",
     r"TRAVA(?: EUCHNER| DE SEGURANCA PDM)? (?:AVARIAD[OA]|DESAJUSTAD[OA])"),
    ("atuador", r"ATUADOR (?:DE PORTA|DO SENSOR ABERTO/FECHADO - PSD|DO SENSOR TRAVADO/DESTRAVADO-PSD|DO SENSOR MAGNÉTICO)",
     r"ATUADOR (?:DE PORTA|DA TRAVA DE PDM) (?:AVARIADO|DESAJUSTADO)"),
    ("motor", r"MOTOR DC(?: DE PDM|-HEADER BOX)", r"(?:MOTOR AVARIADO|BAIXA ISOLACAO)"),
    ("trolley", r"TROLLEY DA PORTA", r"TROLLEY (?:AVARIADO|DESAJUSTADO)"),
    ("correia", r"(?:CORREIA DE TRACAO DE PORTAS|CORREIA DENTADA-HEADER BOX)", r"CORREIA DE TRACAO (?:DANIFICADA|DESAJUSTADA)"),
    ("soleira", r"SOLEIRA", r"SOLEIRA (?:AVARIADA|DESAJUSTADA)"),
    ("sinalização luminosa", r"(?:LED -HEADER BOX|INDICADOR LED HEADER BOX|LCB PDM)",
     r"(?:LCB AVARIADO|BAIXA ISOLACAO|SEM CONEXAO ELETRICA)"),
    ("sinalização sonora", r"(?:SPEAKER PDM|ALTO-FALANTE \(SPEAKER\))",
     r"(?:BAIXA ISOLACAO|SEM CONEXAO ELETRICA)"),
    ("cabeamento", r"(?:CABEAMENTO|CABO/CONECTORES|CABO/CONECTORES -HEADER BOX)",
     r"(?:CABO-FIO ELETRICO ROMPIDO|BAIXA ISOLACAO|SEM CONEXAO ELETRICA|FIBRA OPTICA AVARIADA)"),
    ("folha da porta", r"PORTA PDM (?:DIREITA|ESQUERDA) PINTSCH BAMAG",
     r"PORTA (?:DESAJUSTADA|CEDIDA)"),
    ("vidro", r"(?:VIDRO|VIDRO TEMPERADO COM PELICULA DE PROTECAO)", r"VIDRO (?:QUEBRADO|RISCADO)"),
    ("vedação", r"BORRACHA DE VEDAÇÃO - PDM", r"BORRACHA DE VEDACAO DE PORTA AVARIADA"),
    ("tampa", r"TAMPA-HEADER BOX", r"TAMPA HEADER BOX QUEBRADA"),
], columns=["familia", "padrao_objeto", "padrao_defeito"])

registros_consolidacao["regra_diagnostico"] = pd.Series(
    pd.NA, index=registros_consolidacao.index, dtype="string"
)
for regra in regras_diagnostico.itertuples(index=False):
    compativel = (
        registros_consolidacao["objeto"].str.upper().str.fullmatch(regra.padrao_objeto, na=False)
        & registros_consolidacao["defeito"].str.upper().str.fullmatch(regra.padrao_defeito, na=False)
    )
    registros_consolidacao.loc[compativel, "regra_diagnostico"] = regra.familia

registros_consolidacao["diagnostico_compativel"] = (
    registros_consolidacao["regra_diagnostico"].notna()
    & (registros_consolidacao["sintoma"].notna() | registros_consolidacao["descricao"].notna())
)
registros_consolidacao["fim_invalido_ou_ausente"] = (
    registros_consolidacao["fim_avaria"].isna()
    | registros_consolidacao["fim_avaria"].lt(registros_consolidacao["inicio_avaria"])
)
display(regras_diagnostico[["familia", "padrao_defeito"]])
display(pd.crosstab(registros_consolidacao["fonte"], registros_consolidacao["diagnostico_compativel"]))


,familia,padrao_defeito
0,sensor fechado,SENSOR FECHADO (?:AVARIADO|DESAJUSTADO)
1,sensor aberto,SENSOR ABERTO (?:AVARIADO|DESAJUSTADO)
2,sensor travado,SENSOR TRAVADO (?:AVARIADO|DESAJUSTADO)
3,sensor destravado,SENSOR DESTRAVADO (?:AVARIADO|DESAJUSTADO)
4,controle da porta,(?:MCP (?:AVARIADO|TRAVADO|DESCONFIGURADO)|PERDA DE COMUNICACAO)
5,trava e fechadura,TRAVA(?: EUCHNER| DE SEGURANCA PDM)? (?:AVARIAD[OA]|DESAJUSTAD[OA])
6,atuador,ATUADOR (?:DE PORTA|DA TRAVA DE PDM) (?:AVARIADO|DESAJUSTADO)
7,motor,(?:MOTOR AVARIADO|BAIXA ISOLACAO)
8,trolley,TROLLEY (?:AVARIADO|DESAJUSTADO)
9,correia,CORREIA DE TRACAO (?:DANIFICADA|DESAJUSTADA)


diagnostico_compativel,False,True
fonte,,
ZM01,5268,5341
ZM02,762,401


## 72. Episódios documentais e grupos temporais incertos

A primeira consolidação reúne registros com a mesma chave e início exato, preservando a ligação com notas, ordens e textos. A existência de valores técnicos diferentes ou de conflito de identificação impede a aprovação automática.

A segunda conferência forma grupos de intervalos conectados na mesma porta. Um intervalo pode se sobrepor a outro por meio de um terceiro; por isso, a busca considera o maior término alcançado pelo grupo. Grupos com inícios diferentes ficam pendentes, sem escolher arbitrariamente um início definitivo.

Para um término ausente ou inválido, a incerteza é estendida até o final do recorte. Isso é uma medida conservadora para os rótulos, **não um preenchimento de término de avaria**. Não se presume restabelecimento sem informação. A revisão posterior poderá reduzir esses intervalos.


In [75]:
registros_portas_consolidacao = registros_consolidacao.loc[
    registros_consolidacao["chave_candidata"].isin(portas["chave_candidata"])
    & registros_consolidacao["inicio_avaria"].notna()
].copy()

episodios_consolidados = (
    registros_portas_consolidacao.groupby(["id_candidato", "chave_candidata", "inicio_avaria"])
    .agg(
        qtd_registros=("id_registro", "size"),
        qtd_fontes=("fonte", "nunique"),
        qtd_objetos=("objeto", "nunique"),
        qtd_defeitos=("defeito", "nunique"),
        qtd_fins=("fim_avaria", "nunique"),
        fim_documental=("fim_avaria", "max"),
        identificacao_pendente=("identificacao_pendente", "max"),
        cancelamento_pendente=("cancelamento_pendente", "max"),
        tem_diagnostico_compativel=("diagnostico_compativel", "max"),
        fim_pendente=("fim_invalido_ou_ausente", "max"),
    )
    .reset_index()
    .sort_values(["chave_candidata", "inicio_avaria"])
    .reset_index(drop=True)
)
episodios_consolidados["diagnostico_divergente"] = (
    episodios_consolidados["qtd_objetos"].gt(1)
    | episodios_consolidados["qtd_defeitos"].gt(1)
)
episodios_consolidados["fim_pendente"] |= episodios_consolidados["qtd_fins"].gt(1)
episodios_consolidados["limite_busca"] = episodios_consolidados["fim_documental"].where(
    ~episodios_consolidados["fim_pendente"], DATA_FIM + pd.Timedelta(days=1)
).clip(upper=DATA_FIM + pd.Timedelta(days=1))

# O laço percorre os intervalos em ordem e conserva o maior término do grupo.
episodios_consolidados["grupo_temporal"] = -1
numero_grupo = -1
for chave, intervalos in episodios_consolidados.groupby("chave_candidata", sort=False):
    fim_grupo = pd.NaT
    for indice, episodio in intervalos.iterrows():
        if pd.isna(fim_grupo) or episodio["inicio_avaria"] > fim_grupo:
            numero_grupo += 1
            fim_grupo = episodio["limite_busca"]
        else:
            fim_grupo = max(fim_grupo, episodio["limite_busca"])
        episodios_consolidados.loc[indice, "grupo_temporal"] = numero_grupo

grupos_temporais = (
    episodios_consolidados.groupby("grupo_temporal")
    .agg(
        qtd_inicios=("id_candidato", "size"),
        inicio_grupo=("inicio_avaria", "min"),
        limite_grupo=("limite_busca", "max"),
    )
)
episodios_consolidados = episodios_consolidados.join(grupos_temporais, on="grupo_temporal")
episodios_consolidados["episodio_apto"] = (
    episodios_consolidados["tem_diagnostico_compativel"]
    & ~episodios_consolidados["identificacao_pendente"]
    & ~episodios_consolidados["cancelamento_pendente"]
    & ~episodios_consolidados["diagnostico_divergente"]
    & ~episodios_consolidados["fim_pendente"]
    & episodios_consolidados["qtd_inicios"].eq(1)
)
episodios_consolidados["status_episodio"] = "pendente"
episodios_consolidados.loc[
    episodios_consolidados["episodio_apto"], "status_episodio"
] = "falha documental"

# A ligação mantém todos os registros originais e os motivos da decisão.
rastreabilidade_episodios = registros_portas_consolidacao.merge(
    episodios_consolidados[[
        "id_candidato", "grupo_temporal", "status_episodio", "episodio_apto",
    ]],
    on="id_candidato", how="left", validate="many_to_one",
)
assert episodios_consolidados["id_candidato"].is_unique
assert len(rastreabilidade_episodios) == len(registros_portas_consolidacao)
assert episodios_consolidados["qtd_registros"].sum() == len(registros_portas_consolidacao)

display(episodios_consolidados["status_episodio"].value_counts().to_frame("episodios"))
display(pd.crosstab(rastreabilidade_episodios["fonte"], rastreabilidade_episodios["status_episodio"]))


,episodios
status_episodio,
pendente,6562
falha documental,3369


status_episodio,falha documental,pendente
fonte,,
ZM01,3206,6074
ZM02,185,586


## 73. Portas e períodos afetados pelas incertezas

Um episódio pendente bloqueia negativos nas janelas futuras que interceptam seu intervalo de incerteza. Em conflitos de número de porta, o bloqueio alcança a plataforma informada; em conflitos de plataforma, alcança a estação. Quando um identificador aparece em localizações divergentes, o bloqueio é ampliado para o conjunto de portas.

Registros coletivos ou sem chave individual não são copiados como falhas para todas as portas. Eles apenas bloqueiam negativos no escopo que pode ser afetado. Datas ausentes e localizações não reconhecidas recebem tratamento conservador. Equipamentos explicitamente identificados como tipos diferentes de PDM ficam fora do alvo PDM.

Uma falha documental individual suficiente para afirmar “existe uma falha” pode gerar positivo mesmo se houver outra ocorrência incerta no horizonte. A incerteza impede concluir ausência, mas não apaga essa evidência positiva.


In [76]:
ids_pendentes = episodios_consolidados.loc[
    ~episodios_consolidados["episodio_apto"], "id_candidato"
]
pendencias_rotulo = registros_consolidacao.loc[
    registros_consolidacao["id_candidato"].isin(ids_pendentes)
    | (
        (registros_consolidacao["tipo_ativo"].eq(TIPO_ATIVO) | registros_consolidacao["tipo_ativo"].isna())
        & (
            registros_consolidacao["chave_candidata"].isna()
            | registros_consolidacao["inicio_avaria"].isna()
            | registros_consolidacao["identificacao_pendente"]
        )
    )
].copy()
pendencias_rotulo["inicio_incerteza"] = pendencias_rotulo["inicio_avaria"].dt.normalize().fillna(DATA_INICIO)
pendencias_rotulo["fim_incerteza"] = pendencias_rotulo["fim_avaria"].dt.normalize()
pendencias_rotulo.loc[
    pendencias_rotulo["fim_invalido_ou_ausente"] | pendencias_rotulo["inicio_avaria"].isna(),
    "fim_incerteza",
] = DATA_FIM
pendencias_rotulo = pendencias_rotulo.merge(
    episodios_consolidados[["id_candidato", "inicio_grupo", "limite_grupo"]],
    on="id_candidato", how="left", validate="many_to_one",
)
pendencias_rotulo["inicio_incerteza"] = (
    pendencias_rotulo["inicio_grupo"].dt.normalize()
    .combine_first(pendencias_rotulo["inicio_incerteza"])
)
pendencias_rotulo["fim_incerteza"] = (
    pendencias_rotulo["limite_grupo"].dt.normalize()
    .combine_first(pendencias_rotulo["fim_incerteza"]).clip(upper=DATA_FIM)
)

# A expansão utiliza apenas o escopo conhecido, sem atribuir uma falha às portas.
intervalos_bloqueio = []
for pendencia in pendencias_rotulo.itertuples(index=False):
    escopo = portas.copy()
    if not pendencia.conflito_identificador:
        if pd.notna(pendencia.estacao):
            escopo = escopo.loc[escopo["estacao"].eq(pendencia.estacao)]
        if pd.notna(pendencia.plataforma) and not pendencia.conflito_plataforma:
            escopo = escopo.loc[escopo["plataforma"].eq(pendencia.plataforma)]
        if pd.notna(pendencia.chave_candidata) and not pendencia.identificacao_pendente:
            escopo = escopo.loc[escopo["chave_candidata"].eq(pendencia.chave_candidata)]
    intervalo = escopo[["chave_candidata"]].copy()
    intervalo["inicio_bloqueio"] = max(DATA_INICIO, pendencia.inicio_incerteza - pd.Timedelta(days=DIAS_ALVO))
    intervalo["fim_bloqueio"] = min(DATA_FIM, pendencia.fim_incerteza - pd.Timedelta(days=1))
    intervalos_bloqueio.append(intervalo)

bloqueios_rotulo = pd.concat(
    intervalos_bloqueio or [pd.DataFrame(columns=["chave_candidata", "inicio_bloqueio", "fim_bloqueio"])],
    ignore_index=True,
).drop_duplicates()
bloqueios_rotulo = bloqueios_rotulo.loc[
    bloqueios_rotulo["inicio_bloqueio"] <= bloqueios_rotulo["fim_bloqueio"]
].copy()

# Dois marcadores por intervalo evitam expandir cada pendência em centenas de dias.
entradas_bloqueio = bloqueios_rotulo[["chave_candidata", "inicio_bloqueio"]].rename(
    columns={"inicio_bloqueio": "data"}
).assign(variacao_pendencias=1)
saidas_bloqueio = bloqueios_rotulo[["chave_candidata", "fim_bloqueio"]].rename(
    columns={"fim_bloqueio": "data"}
).assign(variacao_pendencias=-1)
saidas_bloqueio["data"] = saidas_bloqueio["data"] + pd.Timedelta(days=1)
marcadores_pendencias = pd.concat([entradas_bloqueio, saidas_bloqueio], ignore_index=True)
marcadores_pendencias = (
    marcadores_pendencias.groupby(["chave_candidata", "data"])["variacao_pendencias"]
    .sum().reset_index()
)

rotulos_consolidados = base_modelagem_proxy[[
    "chave_candidata", "data", "particao", "tem_30_dias_calendario",
]].merge(marcadores_pendencias, on=["chave_candidata", "data"], how="left", validate="one_to_one")
rotulos_consolidados = rotulos_consolidados.sort_values(["chave_candidata", "data"]).reset_index(drop=True)
rotulos_consolidados["variacao_pendencias"] = rotulos_consolidados["variacao_pendencias"].fillna(0).astype(int)
rotulos_consolidados["pendencias_no_horizonte"] = (
    rotulos_consolidados.groupby("chave_candidata")["variacao_pendencias"].cumsum()
)
assert rotulos_consolidados["pendencias_no_horizonte"].ge(0).all()


## 74. Acompanhamento suficiente e geração dos rótulos

A ausência de manutenção ou de alarme em uma planilha não comprova ausência de falha. As abas de qualidade descrevem limpeza e ausências; não certificam que todas as falhas foram registradas em cada porta e período.

A tabela `periodos_acompanhamento_confirmado` recebe períodos em que há evidência de operação/acompanhamento da porta e de completude dos registros de falhas de **ambas as fontes ZM01 e ZM02**, com justificativa rastreável. Ela começa vazia porque essa confirmação não foi fornecida. Não é preenchida pela simples presença diária de alarmes.

O zero exige sete dias confirmados, de D+1 a D+7, sem falha documental e sem pendências no horizonte. Um positivo exige episódio documental apto no horizonte. Os últimos sete dias do recorte permanecem sem rótulo, pois o horizonte está incompleto.

Esta etapa não converte os negativos da referência antiga `registro_zm01_7d` em negativos de falha.


In [77]:
periodos_acompanhamento_confirmado = pd.DataFrame({
    "chave_candidata": pd.Series(dtype="string"),
    "inicio": pd.Series(dtype="datetime64[ns]"),
    "fim": pd.Series(dtype="datetime64[ns]"),
    "evidencia": pd.Series(dtype="string"),
})
assert periodos_acompanhamento_confirmado["chave_candidata"].isin(portas["chave_candidata"]).all()
assert periodos_acompanhamento_confirmado[["inicio", "fim"]].notna().all().all()
assert periodos_acompanhamento_confirmado["inicio"].le(periodos_acompanhamento_confirmado["fim"]).all()
assert periodos_acompanhamento_confirmado["evidencia"].str.strip().str.len().gt(0).fillna(False).all()

dias_confirmados = []
for periodo in periodos_acompanhamento_confirmado.itertuples(index=False):
    dias_confirmados.append(pd.DataFrame({
        "chave_candidata": periodo.chave_candidata,
        "data_acompanhada": pd.date_range(periodo.inicio, periodo.fim, freq="D"),
    }))
cobertura_diaria_confirmada = pd.concat(
    dias_confirmados or [pd.DataFrame({
        "chave_candidata": pd.Series(dtype="string"),
        "data_acompanhada": pd.Series(dtype="datetime64[ns]"),
    })], ignore_index=True,
).drop_duplicates()

cobertura_horizontes = cobertura_diaria_confirmada.merge(antecedencias_alvo, how="cross")
cobertura_horizontes["data"] = (
    cobertura_horizontes["data_acompanhada"]
    - pd.to_timedelta(cobertura_horizontes["dias_antes"], unit="D")
)
dias_confirmados_por_alvo = (
    cobertura_horizontes.groupby(["chave_candidata", "data"])["data_acompanhada"]
    .nunique().rename("dias_acompanhados_7d").reset_index()
)

episodios_falha_documental = episodios_consolidados.loc[
    episodios_consolidados["episodio_apto"]
].copy()
referencias_falha_documental = episodios_falha_documental[[
    "id_candidato", "chave_candidata", "inicio_avaria",
]].merge(antecedencias_alvo, how="cross")
referencias_falha_documental["data"] = (
    referencias_falha_documental["inicio_avaria"].dt.normalize()
    - pd.to_timedelta(referencias_falha_documental["dias_antes"], unit="D")
)
positivos_documentais = (
    referencias_falha_documental.groupby(["chave_candidata", "data"])["id_candidato"]
    .nunique().rename("qtd_inicios_falha_7d").reset_index()
)

rotulos_consolidados = rotulos_consolidados.merge(
    dias_confirmados_por_alvo, on=["chave_candidata", "data"], how="left", validate="one_to_one"
).merge(
    positivos_documentais, on=["chave_candidata", "data"], how="left", validate="one_to_one"
)
for campo in ["dias_acompanhados_7d", "qtd_inicios_falha_7d"]:
    rotulos_consolidados[campo] = rotulos_consolidados[campo].fillna(0).astype(int)

horizonte_completo = rotulos_consolidados["data"] + pd.Timedelta(days=DIAS_ALVO) <= DATA_FIM
tem_falha_documental = rotulos_consolidados["qtd_inicios_falha_7d"].gt(0)
negativo_sustentado = (
    horizonte_completo
    & rotulos_consolidados["dias_acompanhados_7d"].eq(DIAS_ALVO)
    & rotulos_consolidados["pendencias_no_horizonte"].eq(0)
    & ~tem_falha_documental
)

rotulos_consolidados["falha_7d"] = pd.Series(pd.NA, index=rotulos_consolidados.index, dtype="Int64")
rotulos_consolidados["motivo_rotulo"] = "acompanhamento não confirmado"
rotulos_consolidados.loc[
    rotulos_consolidados["pendencias_no_horizonte"].gt(0), "motivo_rotulo"
] = "episódio ou identificação pendente no horizonte"
rotulos_consolidados.loc[negativo_sustentado, "falha_7d"] = 0
rotulos_consolidados.loc[negativo_sustentado, "motivo_rotulo"] = "sete dias acompanhados, sem falha e sem pendência"
rotulos_consolidados.loc[horizonte_completo & tem_falha_documental, "falha_7d"] = 1
rotulos_consolidados.loc[horizonte_completo & tem_falha_documental, "motivo_rotulo"] = "início de falha documental nos próximos sete dias"
rotulos_consolidados.loc[~horizonte_completo, "motivo_rotulo"] = "horizonte incompleto"

base_alvo_falhas = rotulos_consolidados.copy()
configuracao_modelagem["objetivo_final"] = "início de falha funcional ou defeito de componente entre D+1 e D+7"
configuracao_modelagem["versao_rotulos"] = "documental_v1_conservadora"
configuracao_modelagem["escopo_falha_definido"] = True
configuracao_modelagem["alvo_final_validado"] = False
configuracao_modelagem["status_alvo_final"] = "positivos documentais; pendências preservadas; negativos exigem cobertura confirmada"


## 75. Verificação e resultado da etapa 2

A conferência verifica a unicidade de porta/dia, o horizonte de sete dias e as condições de cada classe. A tabela final preserva o motivo do rótulo; `referencias_falha_documental` permite voltar de um positivo ao episódio e `rastreabilidade_episodios` permite chegar às notas e ordens originais.

Os cortes cronológicos existentes são preservados. A seleção de amostras elegíveis e a atualização das entradas para treinamento pertencem à próxima etapa. Se não houver cobertura confirmada, o resultado terá positivos e valores ausentes, sem negativos inventados; nesse caso, ainda não constitui um conjunto pronto para treinar o classificador.


In [78]:
assert len(base_alvo_falhas) == len(base_modelagem_proxy)
assert not base_alvo_falhas.duplicated(["chave_candidata", "data"]).any()
assert set(base_alvo_falhas["falha_7d"].dropna().unique()).issubset({0, 1})
assert base_alvo_falhas.loc[base_alvo_falhas["falha_7d"].eq(1), "qtd_inicios_falha_7d"].gt(0).all()
assert base_alvo_falhas.loc[base_alvo_falhas["falha_7d"].eq(0), "dias_acompanhados_7d"].eq(7).all()
assert base_alvo_falhas.loc[base_alvo_falhas["falha_7d"].eq(0), "pendencias_no_horizonte"].eq(0).all()
assert base_alvo_falhas.loc[~horizonte_completo, "falha_7d"].isna().all()
assert (
    referencias_falha_documental["inicio_avaria"].dt.normalize()
    - referencias_falha_documental["data"]
).dt.days.between(1, 7).all()
if periodos_acompanhamento_confirmado.empty:
    assert base_alvo_falhas["falha_7d"].eq(0).sum() == 0

cortes_conferidos = base_alvo_falhas[["chave_candidata", "data", "particao"]].merge(
    base_modelagem_proxy[["chave_candidata", "data", "particao"]],
    on=["chave_candidata", "data"], validate="one_to_one", suffixes=("_atual", "_anterior"),
)
assert cortes_conferidos["particao_atual"].eq(cortes_conferidos["particao_anterior"]).all()

display(base_alvo_falhas["falha_7d"].value_counts(dropna=False).to_frame("dias_porta"))
display(pd.crosstab(base_alvo_falhas["particao"], base_alvo_falhas["motivo_rotulo"]))
display(base_alvo_falhas.loc[base_alvo_falhas["falha_7d"].eq(1), [
    "chave_candidata", "data", "falha_7d", "qtd_inicios_falha_7d", "motivo_rotulo",
]].head(7))
print("Episódios documentais aptos:", len(episodios_falha_documental))
print("Positivos:", base_alvo_falhas["falha_7d"].eq(1).sum())
print("Negativos com acompanhamento confirmado:", base_alvo_falhas["falha_7d"].eq(0).sum())
print("Sem rótulo:", base_alvo_falhas["falha_7d"].isna().sum())


,dias_porta
falha_7d,
<NA>,457128
1,14904


motivo_rotulo,episódio ou identificação pendente no horizonte,horizonte incompleto,início de falha documental nos próximos sete dias
particao,,,
teste,65873,3696,1711
treino,318897,0,11103
validacao,68662,0,2090


,chave_candidata,data,falha_7d,qtd_inicios_falha_7d,motivo_rotulo
17,BUT|1|10|PDM,2023-11-03,1,1,início de falha documental nos próximos sete dias
18,BUT|1|10|PDM,2023-11-04,1,1,início de falha documental nos próximos sete dias
19,BUT|1|10|PDM,2023-11-05,1,1,início de falha documental nos próximos sete dias
20,BUT|1|10|PDM,2023-11-06,1,2,início de falha documental nos próximos sete dias
21,BUT|1|10|PDM,2023-11-07,1,2,início de falha documental nos próximos sete dias
22,BUT|1|10|PDM,2023-11-08,1,2,início de falha documental nos próximos sete dias
23,BUT|1|10|PDM,2023-11-09,1,2,início de falha documental nos próximos sete dias


Episódios documentais aptos: 3369
Positivos: 14904
Negativos com acompanhamento confirmado: 0
Sem rótulo: 457128


## 76. Aplicação dos rótulos consolidados à base diária

A base `base_treinamento` reúne as entradas históricas existentes e o alvo `falha_7d` produzido na etapa 2. A junção utiliza **chave da porta e data**, com uma única linha para cada combinação.

Os rótulos atuais são documentais e conservadores. Eles ainda não formam um conjunto definitivo com as duas classes, pois não há negativos com acompanhamento confirmado. A referência antiga `registro_zm01_7d` e sua seleção de amostras não são copiadas para a nova base.

A coluna `indice_entrada` preserva a ligação com as entradas já pré-processadas. As linhas diárias sem rótulo permanecem como contexto histórico: uma janela válida pode precisar de seus valores de entrada.


In [79]:
colunas_entrada_base = [
    "chave_candidata", "data", "particao", "inicio_historico_porta",
] + features_temporais_finais + features_estaticas
assert len(colunas_entrada_base) == len(set(colunas_entrada_base))
assert base_modelagem_proxy.index.is_unique

base_treinamento = base_modelagem_proxy[colunas_entrada_base].copy()
base_treinamento["indice_entrada"] = base_modelagem_proxy.index

colunas_rotulos_atualizados = [
    "chave_candidata", "data", "falha_7d", "motivo_rotulo",
    "qtd_inicios_falha_7d", "dias_acompanhados_7d", "pendencias_no_horizonte",
]
base_treinamento = base_treinamento.merge(
    base_alvo_falhas[colunas_rotulos_atualizados],
    on=["chave_candidata", "data"],
    how="left",
    validate="one_to_one",
    indicator="origem_rotulo",
)
assert base_treinamento["origem_rotulo"].eq("both").all()
base_treinamento = (
    base_treinamento.drop(columns="origem_rotulo")
    .set_index("indice_entrada", drop=False)
    .sort_index()
)
assert base_treinamento.index.equals(base_modelagem_proxy.index)
pd.testing.assert_frame_equal(
    base_treinamento[["chave_candidata", "data", "particao"]].reset_index(drop=True),
    base_modelagem_proxy[["chave_candidata", "data", "particao"]].reset_index(drop=True),
)
assert "registro_zm01_7d" not in base_treinamento.columns
assert base_treinamento["motivo_rotulo"].notna().all()

print("Linhas diárias preservadas:", len(base_treinamento))
display(base_treinamento[["chave_candidata", "data", "particao", "falha_7d", "motivo_rotulo"]].head())


Linhas diárias preservadas: 472032


,chave_candidata,data,particao,falha_7d,motivo_rotulo
indice_entrada,,,,,
0,BUT|1|10|PDM,2023-10-17,treino,<NA>,episódio ou identificação pendente no horizonte
1,BUT|1|10|PDM,2023-10-18,treino,<NA>,episódio ou identificação pendente no horizonte
2,BUT|1|10|PDM,2023-10-19,treino,<NA>,episódio ou identificação pendente no horizonte
3,BUT|1|10|PDM,2023-10-20,treino,<NA>,episódio ou identificação pendente no horizonte
4,BUT|1|10|PDM,2023-10-21,treino,<NA>,episódio ou identificação pendente no horizonte


## 77. Seleção dos dias que podem servir como alvo

Uma linha pode servir como alvo quando possui rótulo conhecido, 30 dias consecutivos de contexto na mesma porta e sete dias futuros dentro dos limites permitidos.

O contexto termina no dia D e começa em D−29. A existência dessas linhas no calendário não comprova cobertura operacional dos sensores; essa limitação das entradas continua registrada na preparação anterior.

Para o treino, D+7 deve ser anterior ao início da validação. Para a validação, deve ser anterior ao início do teste. No teste, D+7 não pode ultrapassar o fim do recorte. Essas condições evitam que um alvo de treino ou validação dependa de uma ocorrência na partição seguinte.

O contexto de uma amostra de validação ou teste pode conter dias da partição anterior: esses dias já pertencem ao passado no instante da previsão. Esta etapa apenas verifica as datas; os tensores de 30 dias serão criados na etapa 4.


In [80]:
diferencas_dias = base_treinamento.groupby("chave_candidata", sort=False)["data"].diff()
assert diferencas_dias.dropna().eq(pd.Timedelta(days=1)).all()

base_treinamento["inicio_janela_30d"] = (
    base_treinamento.groupby("chave_candidata", sort=False)["data"].shift(DIAS_HISTORICO - 1)
)
base_treinamento["tem_historico_30d"] = (
    base_treinamento["inicio_janela_30d"].eq(
        base_treinamento["data"] - pd.Timedelta(days=DIAS_HISTORICO - 1)
    )
    & base_treinamento["inicio_janela_30d"].ge(base_treinamento["inicio_historico_porta"])
)

base_treinamento["fim_horizonte_7d"] = base_treinamento["data"] + pd.Timedelta(days=DIAS_ALVO)
base_treinamento["horizonte_dentro_do_recorte"] = base_treinamento["fim_horizonte_7d"].le(DATA_FIM)
base_treinamento["alvo_atravessa_corte"] = (
    (
        base_treinamento["particao"].eq("treino")
        & base_treinamento["fim_horizonte_7d"].ge(INICIO_VALIDACAO)
    )
    | (
        base_treinamento["particao"].eq("validacao")
        & base_treinamento["fim_horizonte_7d"].ge(INICIO_TESTE)
    )
)
base_treinamento["usar_como_alvo"] = (
    base_treinamento["falha_7d"].notna()
    & base_treinamento["tem_historico_30d"]
    & base_treinamento["horizonte_dentro_do_recorte"]
    & ~base_treinamento["alvo_atravessa_corte"]
)

# O motivo principal é apresentado sem apagar as demais condições.
base_treinamento["motivo_selecao"] = "apta como alvo documental"
base_treinamento.loc[
    ~base_treinamento["tem_historico_30d"], "motivo_selecao"
] = "histórico inferior a 30 dias"
base_treinamento.loc[
    base_treinamento["alvo_atravessa_corte"], "motivo_selecao"
] = "horizonte alcança a próxima partição"
base_treinamento.loc[
    ~base_treinamento["horizonte_dentro_do_recorte"], "motivo_selecao"
] = "horizonte fora do recorte"
base_treinamento.loc[
    base_treinamento["falha_7d"].isna(), "motivo_selecao"
] = "sem rótulo; motivo detalhado em motivo_rotulo"

indices_alvo = {}
alvos_por_particao = {}
linhas_resumo_atualizado = []
for particao in ["treino", "validacao", "teste"]:
    linhas_particao = base_treinamento["particao"].eq(particao)
    indices = base_treinamento.index[linhas_particao & base_treinamento["usar_como_alvo"]]
    indices_alvo[particao] = indices
    alvos_por_particao[particao] = base_treinamento.loc[indices, "falha_7d"].astype("int8")
    amostras = base_treinamento.loc[indices]
    linhas_resumo_atualizado.append({
        "particao": particao,
        "linhas_diarias": int(linhas_particao.sum()),
        "sem_rotulo": int(base_treinamento.loc[linhas_particao, "falha_7d"].isna().sum()),
        "rotuladas_nao_elegiveis": int((
            linhas_particao & base_treinamento["falha_7d"].notna()
            & ~base_treinamento["usar_como_alvo"]
        ).sum()),
        "alvos_elegiveis": len(amostras),
        "positivos": int(amostras["falha_7d"].eq(1).sum()),
        "negativos": int(amostras["falha_7d"].eq(0).sum()),
        "primeiro_alvo": amostras["data"].min(),
        "ultimo_alvo": amostras["data"].max(),
        "tem_duas_classes": set(amostras["falha_7d"].unique()) == {0, 1},
    })
resumo_base_treinamento = pd.DataFrame(linhas_resumo_atualizado)
with pd.option_context("display.max_columns", None):
    display(resumo_base_treinamento)
display(pd.crosstab(base_treinamento["particao"], base_treinamento["motivo_selecao"]))


,particao,linhas_diarias,sem_rotulo,rotuladas_nao_elegiveis,alvos_elegiveis,positivos,negativos,primeiro_alvo,ultimo_alvo,tem_duas_classes
0,treino,330000,318897,719,10384,10384,0,2023-11-15,2025-06-25,False
1,validacao,70752,68662,82,2008,2008,0,2025-07-03,2025-11-06,False
2,teste,71280,69569,0,1711,1711,0,2025-11-14,2026-03-21,False


motivo_selecao,apta como alvo documental,histórico inferior a 30 dias,horizonte alcança a próxima partição,sem rótulo; motivo detalhado em motivo_rotulo
particao,,,,
teste,1711,0,0,69569
treino,10384,587,132,318897
validacao,2008,0,82,68662


## 78. Alinhamento das entradas e conferência final

As variáveis de entrada e os cortes cronológicos não mudaram. Por isso, o imputador, o normalizador e o codificador estático já ajustados somente no treino são reaproveitados. Nenhum ajuste é realizado usando validação, teste ou os novos rótulos.

As tabelas `dados_temporais_falhas` e `dados_estaticos_falhas` mantêm todas as linhas diárias na mesma ordem de `base_treinamento`. Os índices de `indices_alvo` indicam onde terminarão as futuras sequências; eles não autorizam remover previamente os dias sem rótulo das entradas.

Somente as colunas explicitamente listadas como features poderão entrar na rede. Diagnóstico, justificativa do rótulo, contagem de falhas futuras e cobertura do horizonte permanecem fora das entradas.

A presença de alvos elegíveis não basta para iniciar o treinamento: os conjuntos ainda precisam conter positivos e negativos para treinar e avaliar o classificador binário.


In [81]:
dados_temporais_falhas = dados_temporais_prontos.loc[
    base_treinamento.index, features_temporais_finais
].copy()
dados_estaticos_falhas = dados_estaticos_prontos.loc[
    base_treinamento.index, nomes_estaticas
].copy()

assert dados_temporais_falhas.index.equals(base_treinamento.index)
assert dados_estaticos_falhas.index.equals(base_treinamento.index)
assert np.isfinite(dados_temporais_falhas.to_numpy()).all()
assert np.isfinite(dados_estaticos_falhas.to_numpy()).all()
assert dados_temporais_falhas.shape == (len(base_treinamento), len(features_temporais_finais))
assert dados_estaticos_falhas.shape == (len(base_treinamento), len(nomes_estaticas))
assert not set(features_temporais_finais + features_estaticas).intersection({
    "falha_7d", "registro_zm01_7d", "qtd_inicios_falha_7d",
    "dias_acompanhados_7d", "pendencias_no_horizonte", "motivo_rotulo",
})

for particao, indices in indices_alvo.items():
    amostras = base_treinamento.loc[indices]
    assert amostras["particao"].eq(particao).all()
    assert amostras["falha_7d"].notna().all()
    assert amostras["tem_historico_30d"].all()
    assert amostras["horizonte_dentro_do_recorte"].all()
    assert not amostras["alvo_atravessa_corte"].any()
    assert alvos_por_particao[particao].index.equals(indices)
    assert alvos_por_particao[particao].eq(amostras["falha_7d"]).all()

assert set(indices_alvo["treino"]).isdisjoint(indices_alvo["validacao"])
assert set(indices_alvo["treino"]).isdisjoint(indices_alvo["teste"])
assert set(indices_alvo["validacao"]).isdisjoint(indices_alvo["teste"])
assert resumo_base_treinamento["alvos_elegiveis"].sum() == base_treinamento["usar_como_alvo"].sum()
assert (
    resumo_base_treinamento["sem_rotulo"]
    + resumo_base_treinamento["rotuladas_nao_elegiveis"]
    + resumo_base_treinamento["alvos_elegiveis"]
).eq(resumo_base_treinamento["linhas_diarias"]).all()

base_com_duas_classes = bool(resumo_base_treinamento["tem_duas_classes"].all())
configuracao_modelagem["base_treinamento_atualizada"] = True
configuracao_modelagem["conjuntos_com_duas_classes"] = base_com_duas_classes
configuracao_modelagem["status_preparacao"] = (
    "base alinhada e com as duas classes; sequências ainda não criadas"
    if base_com_duas_classes
    else "base alinhada; faltam classes confirmadas para treinamento e avaliação"
)

print("Entradas temporais diárias:", dados_temporais_falhas.shape)
print("Entradas estáticas codificadas:", dados_estaticos_falhas.shape)
print("Alvos elegíveis:", int(base_treinamento["usar_como_alvo"].sum()))
print("Treino, validação e teste têm as duas classes:", base_com_duas_classes)
print(configuracao_modelagem["status_preparacao"])


Entradas temporais diárias: (472032, 12)
Entradas estáticas codificadas: (472032, 13)
Alvos elegíveis: 14103
Treino, validação e teste têm as duas classes: False
base alinhada; faltam classes confirmadas para treinamento e avaliação


### Resultado da etapa 3

A base diária está atualizada com os rótulos consolidados e mantém os cortes originais. A seleção foi recalculada a partir de `falha_7d`, do histórico de 30 dias e do horizonte futuro permitido.

Os objetos de referência para as próximas etapas são `base_treinamento`, `dados_temporais_falhas`, `dados_estaticos_falhas`, `indices_alvo`, `alvos_por_particao` e `preprocessadores`. Os objetos com sufixo `proxy` pertencem à referência antiga por registro ZM01.

Enquanto não houver negativos confirmados, a seleção identifica apenas alvos positivos utilizáveis sob os critérios documentais. A preparação desta etapa está concluída, mas o treinamento e a avaliação binária continuam dependentes da consolidação de negativos. A criação dos tensores de 30 dias pertence à etapa 4.


## 79. Criação das sequências de 30 dias

Cada amostra contém os valores de uma única porta entre **D−29 e D**, em ordem cronológica. O rótulo se refere ao início de falha entre D+1 e D+7. Somente os dias selecionados em `indices_alvo` encerram uma amostra.

A base atual contém **12 variáveis temporais** e **13 colunas estáticas codificadas**. Assim, o formato de cada entrada temporal é `(30, 12)`. A quantidade de 20 variáveis apresentada na arquitetura é ilustrativa; a dimensão da entrada é obtida das features efetivamente disponíveis.

Os dias sem rótulo são mantidos dentro do histórico. O rótulo necessário é o do dia D, e não o de cada dia da janela. Uma janela nunca atravessa a fronteira entre duas portas. Os contextos da validação e do teste podem incluir dias anteriores ao corte, conforme a regra da etapa 3.

A implementação utiliza arrays NumPy em `float32`, sem depender da biblioteca que será escolhida para a LSTM. As informações estáticas são obtidas na linha do dia D e permanecem separadas da sequência temporal.


In [82]:
n_features_temporais = len(features_temporais_finais)
n_features_estaticas = len(nomes_estaticas)
assert DIAS_HISTORICO == 30
assert dados_temporais_falhas.index.equals(base_treinamento.index)
assert dados_estaticos_falhas.index.equals(base_treinamento.index)
assert list(dados_temporais_falhas.columns) == features_temporais_finais
assert list(dados_estaticos_falhas.columns) == nomes_estaticas

valores_temporais = dados_temporais_falhas.to_numpy(dtype="float32")
valores_estaticos = dados_estaticos_falhas.to_numpy(dtype="float32")
chaves_das_linhas = base_treinamento["chave_candidata"].to_numpy()
datas_das_linhas = base_treinamento["data"].to_numpy(dtype="datetime64[ns]")

sequencias = {}
for particao in ["treino", "validacao", "teste"]:
    indices = indices_alvo[particao]
    # O índice original identifica a linha; get_indexer encontra sua posição no array.
    posicoes_fim = base_treinamento.index.get_indexer(indices)
    assert (posicoes_fim >= 0).all()
    assert base_treinamento.loc[indices, "usar_como_alvo"].all()
    assert base_treinamento.loc[indices, "particao"].eq(particao).all()

    X_temporal = np.empty(
        (len(indices), DIAS_HISTORICO, n_features_temporais), dtype="float32"
    )
    for numero_amostra, posicao_fim in enumerate(posicoes_fim):
        posicao_inicio = posicao_fim - DIAS_HISTORICO + 1
        assert posicao_inicio >= 0

        chaves_janela = chaves_das_linhas[posicao_inicio:posicao_fim + 1]
        datas_janela = datas_das_linhas[posicao_inicio:posicao_fim + 1]
        assert len(datas_janela) == DIAS_HISTORICO
        assert (chaves_janela == chaves_das_linhas[posicao_fim]).all()
        assert (np.diff(datas_janela) == np.timedelta64(1, "D")).all()
        assert datas_janela[0] == datas_janela[-1] - np.timedelta64(DIAS_HISTORICO - 1, "D")

        X_temporal[numero_amostra] = valores_temporais[posicao_inicio:posicao_fim + 1]

    X_estatico = valores_estaticos[posicoes_fim].copy()
    assert alvos_por_particao[particao].index.equals(indices)
    y = alvos_por_particao[particao].to_numpy(dtype="float32").reshape(-1, 1)

    metadados = base_treinamento.loc[indices, [
        "indice_entrada", "chave_candidata", "inicio_janela_30d",
        "data", "fim_horizonte_7d", "particao", "falha_7d",
    ]].reset_index(drop=True)
    metadados.index.name = "amostra"
    metadados["posicao_inicio"] = posicoes_fim - DIAS_HISTORICO + 1
    metadados["posicao_fim"] = posicoes_fim
    metadados["inicio_horizonte"] = metadados["data"] + pd.Timedelta(days=1)
    assert pd.to_datetime(datas_das_linhas[metadados["posicao_inicio"]]).equals(
        pd.DatetimeIndex(metadados["inicio_janela_30d"])
    )

    sequencias[particao] = {
        "X_temporal": X_temporal,
        "X_estatico": X_estatico,
        "y": y,
        "metadados": metadados,
    }

print("Variáveis temporais por dia:", n_features_temporais)
print("Colunas estáticas codificadas:", n_features_estaticas)


Variáveis temporais por dia: 12
Colunas estáticas codificadas: 13


## 80. Conferência dos tensores e rastreabilidade

As três entradas correspondentes à mesma posição representam a mesma amostra:

- `X_temporal[i]`: histórico de 30 dias da porta.
- `X_estatico[i]`: informações estáticas codificadas da porta no dia D.
- `y[i]`: rótulo para D+1 a D+7, com formato `(1,)`.

A linha `metadados.iloc[i]` informa a porta, as datas e a linha original. O resumo apresenta as dimensões reais de cada conjunto. O código verifica o alinhamento das entradas, a ausência de valores não finitos e o respeito aos cortes temporais.

O vetor de rótulos contém valores 0 ou 1; ele não contém probabilidades previstas. A probabilidade será produzida pela camada Sigmoid após a implementação e o treinamento da rede.


In [83]:
linhas_resumo_sequencias = []
for particao, conjunto in sequencias.items():
    X_temporal = conjunto["X_temporal"]
    X_estatico = conjunto["X_estatico"]
    y = conjunto["y"]
    metadados = conjunto["metadados"]
    quantidade = len(indices_alvo[particao])
    posicoes_fim = metadados["posicao_fim"].to_numpy(dtype=int)
    posicoes_inicio = metadados["posicao_inicio"].to_numpy(dtype=int)

    assert X_temporal.shape == (quantidade, DIAS_HISTORICO, n_features_temporais)
    assert X_estatico.shape == (quantidade, n_features_estaticas)
    assert y.shape == (quantidade, 1)
    assert len(metadados) == quantidade
    for array in [X_temporal, X_estatico, y]:
        assert array.dtype == np.float32
        assert np.isfinite(array).all()
    assert np.isin(y, [0, 1]).all()
    np.testing.assert_array_equal(X_temporal[:, 0, :], valores_temporais[posicoes_inicio])
    np.testing.assert_array_equal(X_temporal[:, -1, :], valores_temporais[posicoes_fim])
    np.testing.assert_array_equal(X_estatico, valores_estaticos[posicoes_fim])
    np.testing.assert_array_equal(y[:, 0], metadados["falha_7d"].to_numpy(dtype="float32"))

    assert metadados["inicio_horizonte"].gt(metadados["data"]).all()
    assert (metadados["fim_horizonte_7d"] - metadados["data"]).dt.days.eq(7).all()
    if particao == "treino":
        assert metadados["fim_horizonte_7d"].lt(INICIO_VALIDACAO).all()
    elif particao == "validacao":
        assert metadados["data"].ge(INICIO_VALIDACAO).all()
        assert metadados["fim_horizonte_7d"].lt(INICIO_TESTE).all()
    else:
        assert metadados["data"].ge(INICIO_TESTE).all()
        assert metadados["fim_horizonte_7d"].le(DATA_FIM).all()

    linhas_resumo_sequencias.append({
        "particao": particao,
        "X_temporal": X_temporal.shape,
        "X_estatico": X_estatico.shape,
        "y": y.shape,
        "positivos": int((y == 1).sum()),
        "negativos": int((y == 0).sum()),
        "memoria_arrays_MiB": round(
            (X_temporal.nbytes + X_estatico.nbytes + y.nbytes) / (1024 ** 2), 2
        ),
    })

resumo_sequencias = pd.DataFrame(linhas_resumo_sequencias)
display(resumo_sequencias)
assert sum(len(conjunto["y"]) for conjunto in sequencias.values()) == int(
    base_treinamento["usar_como_alvo"].sum()
)

# A amostra exibe os 30 dias na mesma ordem da entrada que será entregue à LSTM.
if len(sequencias["treino"]["y"]) > 0:
    metadado_exemplo = sequencias["treino"]["metadados"].iloc[0]
    janela_exemplo = pd.DataFrame(
        sequencias["treino"]["X_temporal"][0], columns=features_temporais_finais
    )
    janela_exemplo.insert(0, "data", pd.date_range(
        metadado_exemplo["inicio_janela_30d"], metadado_exemplo["data"], freq="D"
    ))
    display(metadado_exemplo.to_frame("valor"))
    with pd.option_context("display.max_rows", 30):
        display(janela_exemplo)


,particao,X_temporal,X_estatico,y,positivos,negativos,memoria_arrays_MiB
0,treino,"(10384, 30, 12)","(10384, 13)","(10384, 1)",10384,0,14.81
1,validacao,"(2008, 30, 12)","(2008, 13)","(2008, 1)",2008,0,2.86
2,teste,"(1711, 30, 12)","(1711, 13)","(1711, 1)",1711,0,2.44


,valor
indice_entrada,297
chave_candidata,BUT|1|10|PDM
inicio_janela_30d,2024-07-11 00:00:00
data,2024-08-09 00:00:00
fim_horizonte_7d,2024-08-16 00:00:00
particao,treino
falha_7d,1
posicao_inicio,268
posicao_fim,297
inicio_horizonte,2024-08-10 00:00:00


,data,qtd_alarmes_registrados_1d,qtd_alarmes_registrados_7d,qtd_alarmes_registrados_14d,variacao_alarmes_registrados_7d,qtd_fo_7d,...,dias_com_registro_7d,dias_com_registro_14d,dias_desde_ultima_zm01,dias_desde_ultima_zm02,sem_historico_zm01,sem_historico_zm02
0,2024-07-11,-0.156522,-0.229077,0.104352,-0.628557,-0.073539,...,-0.573139,0.159380,0.828463,-0.265901,0.0,0.0
1,2024-07-12,-0.156522,-0.229077,0.104352,-0.628557,-0.073539,...,-0.573139,0.159380,0.836950,-0.258409,0.0,0.0
2,2024-07-13,-0.156522,-0.229077,0.104352,-0.628557,-0.073539,...,-0.573139,0.159380,0.845436,-0.250917,0.0,0.0
3,2024-07-14,-0.156522,-0.229077,-0.080290,-0.315354,-0.073539,...,-0.573139,-0.232887,0.853922,-0.243425,0.0,0.0
4,2024-07-15,-0.156522,-0.229077,-0.080290,-0.315354,-0.073539,...,-0.573139,-0.232887,0.862408,-0.235933,0.0,0.0
5,2024-07-16,-0.156522,-0.229077,-0.080290,-0.315354,-0.073539,...,-0.573139,-0.232887,0.870894,-0.228441,0.0,0.0
6,2024-07-17,-0.156522,-0.229077,-0.080290,-0.315354,-0.073539,...,-0.573139,-0.232887,0.879381,-0.220949,0.0,0.0
7,2024-07-18,-0.156522,-0.229077,-0.264933,-0.002152,-0.073539,...,-0.573139,-0.625153,0.887867,-0.213457,0.0,0.0
8,2024-07-19,-0.156522,-0.229077,-0.264933,-0.002152,-0.073539,...,-0.573139,-0.625153,0.896353,-0.205965,0.0,0.0
9,2024-07-20,-0.156522,-0.229077,-0.264933,-0.002152,-0.073539,...,-0.573139,-0.625153,0.904839,-0.198473,0.0,0.0


## 81. Exemplo de lote e conexão com a arquitetura

Com `batch_size=256`, um lote completo terá os formatos abaixo:

```text
Entrada temporal: (256, 30, 12) → LSTM(64) → representação (256, 64)
Entrada estática: (256, 13)
Concatenação após normalização e dropout: (256, 77)
Dense(32, ReLU) → Dropout(0,3) → Dense(1, Sigmoid)
Saída esperada: (256, 1)
```

As 64 unidades da LSTM não precisam coincidir com as 12 variáveis de entrada. As 13 colunas estáticas são a codificação de estação e plataforma, e não 13 novos campos brutos.

O lote abaixo demonstra o formato usando as primeiras amostras de treino, sem executar uma rede. No treinamento futuro, um eventual embaralhamento deverá aplicar a mesma ordem às duas entradas e aos rótulos. Os dias dentro de cada sequência permanecem em ordem cronológica.

O último lote de cada conjunto pode ter menos de 256 amostras; isso não exige descartar nem repetir amostras.


In [84]:
BATCH_SIZE = 256
assert BATCH_SIZE > 0
quantidade_lote = min(BATCH_SIZE, len(sequencias["treino"]["y"]))

lote_exemplo = {
    "X_temporal": sequencias["treino"]["X_temporal"][:quantidade_lote],
    "X_estatico": sequencias["treino"]["X_estatico"][:quantidade_lote],
    "y": sequencias["treino"]["y"][:quantidade_lote],
    "metadados": sequencias["treino"]["metadados"].iloc[:quantidade_lote],
}
print("Lote temporal:", lote_exemplo["X_temporal"].shape)
print("Lote estático:", lote_exemplo["X_estatico"].shape)
print("Rótulos do lote:", lote_exemplo["y"].shape)

resumo_lotes = []
for particao, conjunto in sequencias.items():
    quantidade = len(conjunto["y"])
    resumo_lotes.append({
        "particao": particao,
        "amostras": quantidade,
        "lotes": (quantidade + BATCH_SIZE - 1) // BATCH_SIZE,
        "amostras_ultimo_lote": ((quantidade - 1) % BATCH_SIZE) + 1 if quantidade else 0,
    })
display(pd.DataFrame(resumo_lotes))

configuracao_modelagem["sequencias_criadas"] = True
configuracao_modelagem["formato_janela"] = [DIAS_HISTORICO, n_features_temporais]
configuracao_modelagem["n_features_estaticas"] = n_features_estaticas
configuracao_modelagem["batch_size_proposto"] = BATCH_SIZE
configuracao_modelagem["status_preparacao"] = (
    "sequências criadas; implementação e treinamento da rede pendentes"
    if base_com_duas_classes
    else "sequências criadas; treinamento binário ainda depende de negativos confirmados"
)
print(configuracao_modelagem["status_preparacao"])


Lote temporal: (256, 30, 12)
Lote estático: (256, 13)
Rótulos do lote: (256, 1)


,particao,amostras,lotes,amostras_ultimo_lote
0,treino,10384,41,144
1,validacao,2008,8,216
2,teste,1711,7,175


sequências criadas; treinamento binário ainda depende de negativos confirmados


### Resultado da etapa 4

As sequências estão organizadas no dicionário `sequencias`, separado em `treino`, `validacao` e `teste`. Cada conjunto contém `X_temporal`, `X_estatico`, `y` e `metadados`, alinhados na mesma ordem.

Os arrays são criados em memória ao executar o notebook. O arquivo preserva o código, as verificações e os resultados exibidos. A etapa não altera rótulos nem ajusta novamente os pré-processadores.

A criação das entradas para a LSTM está concluída. A rede ainda não foi implementada nem treinada, e a ausência de negativos confirmados continua impedindo o treinamento binário com esses conjuntos.


## 82. Implementação da arquitetura LSTM

A implementação utiliza PyTorch, disponível no ambiente do projeto. A classe abaixo representa a rede proposta:

```text
(30, 12) → LSTM(64) → LayerNormalization → Dropout(0,3)
                                               ↓
                           concatenação com 13 valores estáticos
                                               ↓
                        Dense(32, ReLU) → Dropout(0,3)
                                               ↓
                                  Dense(1, Sigmoid)
```

A LSTM fornece a representação do último dia da sequência. A concatenação reúne seus 64 valores e as 13 colunas estáticas; a camada seguinte recebe 77 valores. As dimensões de entrada são obtidas das sequências, preservando as features definidas nas etapas anteriores.

A execução inicial verifica somente o formato da saída. Os pesos ainda são aleatórios, portanto os valores dessa verificação não representam estimativas aprendidas de risco de falha.


In [85]:
import copy
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score

SEMENTE = 42
EPOCAS = 30
PACIENCIA = 5
TAXA_APRENDIZADO = 0.001
LIMIAR_INICIAL = 0.5

torch.manual_seed(SEMENTE)
torch.set_num_threads(min(4, torch.get_num_threads()))
dispositivo = torch.device("cpu")

class LSTMPortas(nn.Module):
    def __init__(self, n_temporais, n_estaticas):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_temporais, hidden_size=64, batch_first=True)
        self.normalizacao = nn.LayerNorm(64)
        self.dropout_temporal = nn.Dropout(0.3)
        self.dense = nn.Linear(64 + n_estaticas, 32)
        self.relu = nn.ReLU()
        self.dropout_dense = nn.Dropout(0.3)
        self.saida = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, temporal, estatico):
        historico, _ = self.lstm(temporal)
        ultimo_dia = historico[:, -1, :]
        resumo = self.dropout_temporal(self.normalizacao(ultimo_dia))
        combinado = torch.cat([resumo, estatico], dim=1)
        combinado = self.dropout_dense(self.relu(self.dense(combinado)))
        return self.sigmoid(self.saida(combinado))

modelo = LSTMPortas(n_features_temporais, n_features_estaticas).to(dispositivo)
modelo.eval()
quantidade_verificacao = min(BATCH_SIZE, len(sequencias["treino"]["y"]))
if quantidade_verificacao > 0:
    with torch.no_grad():
        saida_verificacao = modelo(
            torch.from_numpy(sequencias["treino"]["X_temporal"][:quantidade_verificacao]).to(dispositivo),
            torch.from_numpy(sequencias["treino"]["X_estatico"][:quantidade_verificacao]).to(dispositivo),
        )
    assert saida_verificacao.shape == (quantidade_verificacao, 1)
    assert torch.isfinite(saida_verificacao).all()
    assert ((saida_verificacao >= 0) & (saida_verificacao <= 1)).all()
    print("Formato da saída com pesos iniciais:", tuple(saida_verificacao.shape))

print("PyTorch:", torch.__version__)
print(modelo)
print("Parâmetros treináveis:", sum(p.numel() for p in modelo.parameters() if p.requires_grad))


Formato da saída com pesos iniciais: (256, 1)
PyTorch: 2.12.0+cpu
LSTMPortas(
  (lstm): LSTM(12, 64, batch_first=True)
  (normalizacao): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
  (dropout_temporal): Dropout(p=0.3, inplace=False)
  (dense): Linear(in_features=77, out_features=32, bias=True)
  (relu): ReLU()
  (dropout_dense): Dropout(p=0.3, inplace=False)
  (saida): Linear(in_features=32, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)
Parâmetros treináveis: 22625


## 83. Condições de treinamento e preparação dos lotes

O treinamento binário exige exemplos com e sem falha no treino. A validação também precisa das duas classes para permitir as comparações desta etapa. A conferência abaixo considera somente treino e validação; o teste fica reservado à avaliação final.

Com apenas rótulos 1, reduzir a perda pode significar simplesmente aprender a responder “falha” em todos os casos. Por isso, a execução não ajusta pesos enquanto faltarem negativos confirmados. Isso não é resolvido convertendo registros incertos em zero.

O lote contém as duas entradas e o rótulo juntos. Apenas a ordem das amostras de treino é embaralhada; os 30 dias de cada sequência preservam sua ordem. A validação mantém a ordem original, e o último lote incompleto é preservado.


In [86]:
conferencia_classes_treinamento = []
for particao in ["treino", "validacao"]:
    conjunto = sequencias[particao]
    y_conjunto = conjunto["y"]
    quantidade = len(y_conjunto)
    assert conjunto["X_temporal"].shape == (quantidade, DIAS_HISTORICO, n_features_temporais)
    assert conjunto["X_estatico"].shape == (quantidade, n_features_estaticas)
    assert y_conjunto.shape == (quantidade, 1)
    for campo in ["X_temporal", "X_estatico", "y"]:
        assert conjunto[campo].dtype == np.float32
        assert np.isfinite(conjunto[campo]).all()
    assert np.isin(y_conjunto, [0, 1]).all()

    conferencia_classes_treinamento.append({
        "particao": particao,
        "positivos": int((y_conjunto == 1).sum()),
        "negativos": int((y_conjunto == 0).sum()),
        "tem_duas_classes": set(np.unique(y_conjunto)) == {0, 1},
    })
conferencia_classes_treinamento = pd.DataFrame(conferencia_classes_treinamento)
pode_treinar = bool(conferencia_classes_treinamento["tem_duas_classes"].all())
display(conferencia_classes_treinamento)

carregadores = {}
if pode_treinar:
    for particao in ["treino", "validacao"]:
        conjunto = sequencias[particao]
        dataset = TensorDataset(
            torch.from_numpy(conjunto["X_temporal"]),
            torch.from_numpy(conjunto["X_estatico"]),
            torch.from_numpy(conjunto["y"]),
        )
        carregadores[particao] = DataLoader(
            dataset,
            batch_size=BATCH_SIZE,
            shuffle=(particao == "treino"),
            drop_last=False,
            num_workers=0,
            generator=torch.Generator().manual_seed(SEMENTE),
        )
else:
    print("Treinamento pendente: treino e validação precisam de positivos e negativos confirmados.")


,particao,positivos,negativos,tem_duas_classes
0,treino,10384,0,False
1,validacao,2008,0,False


Treinamento pendente: treino e validação precisam de positivos e negativos confirmados.


## 84. Treinamento e acompanhamento por época

A função de perda é a entropia cruzada binária (`BCELoss`), aplicada à saída Sigmoid. O otimizador Adam utiliza taxa inicial de 0,001. Esta primeira versão não aplica pesos de classe nem reamostragem.

Em cada época, a rede ajusta os pesos somente nos lotes de treino. Depois, a validação utiliza `eval()` e `no_grad()`: os dropouts são desativados e os pesos não são atualizados.

O histórico registra perda de treino, perda de validação, Average Precision (AP), precisão, recall e F1 na validação. AP resume a qualidade da ordenação dos alertas ao longo de diferentes limiares; as outras três métricas utilizam inicialmente 0,5. A escolha operacional do limiar pertence à etapa de avaliação.

O treinamento termina após cinco épocas sem redução da perda de validação ou ao atingir 30 épocas. Os pesos da melhor época são copiados em memória e restaurados ao final. Nenhuma métrica do teste participa dessa decisão.


In [87]:
assert EPOCAS > 0 and PACIENCIA > 0 and TAXA_APRENDIZADO > 0
assert 0 < LIMIAR_INICIAL < 1
historico_epocas = []
treinamento_executado = False
melhores_pesos = None
melhores_probabilidades_validacao = None
melhor_epoca = None
melhor_perda_validacao = float("inf")
epocas_sem_melhora = 0
predicoes_validacao = pd.DataFrame()

if pode_treinar:
    # Uma execução desta célula inicia um treinamento novo e reproduzível na CPU.
    torch.manual_seed(SEMENTE)
    modelo = LSTMPortas(n_features_temporais, n_features_estaticas).to(dispositivo)
    carregadores["treino"].generator.manual_seed(SEMENTE)
    criterio = nn.BCELoss()
    otimizador = torch.optim.Adam(modelo.parameters(), lr=TAXA_APRENDIZADO)

    for epoca in range(1, EPOCAS + 1):
        modelo.train()
        soma_perda_treino = 0.0
        quantidade_treino = 0
        for temporal, estatico, alvo in carregadores["treino"]:
            temporal, estatico, alvo = temporal.to(dispositivo), estatico.to(dispositivo), alvo.to(dispositivo)
            otimizador.zero_grad()
            probabilidade = modelo(temporal, estatico)
            perda = criterio(probabilidade, alvo)
            assert torch.isfinite(perda)
            perda.backward()
            nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)
            otimizador.step()
            soma_perda_treino += perda.item() * len(alvo)
            quantidade_treino += len(alvo)

        modelo.eval()
        soma_perda_validacao = 0.0
        quantidade_validacao = 0
        probabilidades_validacao = []
        with torch.no_grad():
            for temporal, estatico, alvo in carregadores["validacao"]:
                temporal, estatico, alvo = temporal.to(dispositivo), estatico.to(dispositivo), alvo.to(dispositivo)
                probabilidade = modelo(temporal, estatico)
                perda = criterio(probabilidade, alvo)
                assert torch.isfinite(perda)
                soma_perda_validacao += perda.item() * len(alvo)
                quantidade_validacao += len(alvo)
                probabilidades_validacao.append(probabilidade.cpu().numpy())

        perda_treino = soma_perda_treino / quantidade_treino
        perda_validacao = soma_perda_validacao / quantidade_validacao
        probabilidades_validacao = np.concatenate(probabilidades_validacao).ravel()
        y_validacao = sequencias["validacao"]["y"].ravel()
        classes_validacao = (probabilidades_validacao >= LIMIAR_INICIAL).astype(int)
        assert len(probabilidades_validacao) == len(y_validacao)
        assert np.isfinite(probabilidades_validacao).all()

        historico_epocas.append({
            "epoca": epoca,
            "perda_treino": perda_treino,
            "perda_validacao": perda_validacao,
            "ap_validacao": average_precision_score(y_validacao, probabilidades_validacao),
            "precisao_validacao": precision_score(y_validacao, classes_validacao, zero_division=0),
            "recall_validacao": recall_score(y_validacao, classes_validacao, zero_division=0),
            "f1_validacao": f1_score(y_validacao, classes_validacao, zero_division=0),
        })
        print(
            f"Época {epoca:02d} | perda treino: {perda_treino:.4f} | "
            f"perda validação: {perda_validacao:.4f} | "
            f"AP validação: {historico_epocas[-1]['ap_validacao']:.4f}"
        )

        if perda_validacao < melhor_perda_validacao:
            melhor_perda_validacao = perda_validacao
            melhor_epoca = epoca
            melhores_pesos = copy.deepcopy(modelo.state_dict())
            melhores_probabilidades_validacao = probabilidades_validacao.copy()
            epocas_sem_melhora = 0
        else:
            epocas_sem_melhora += 1
            if epocas_sem_melhora >= PACIENCIA:
                print("Parada antecipada por ausência de melhora na validação.")
                break

    modelo.load_state_dict(melhores_pesos)
    modelo.eval()
    treinamento_executado = True
    predicoes_validacao = sequencias["validacao"]["metadados"].copy()
    assert len(predicoes_validacao) == len(melhores_probabilidades_validacao)
    predicoes_validacao["probabilidade_falha_7d"] = melhores_probabilidades_validacao
    predicoes_validacao["classe_limiar_inicial"] = (
        melhores_probabilidades_validacao >= LIMIAR_INICIAL
    ).astype("int8")
else:
    print("Nenhuma época executada: faltam as duas classes nos conjuntos de desenvolvimento.")

historico_treinamento = pd.DataFrame(historico_epocas, columns=[
    "epoca", "perda_treino", "perda_validacao", "ap_validacao",
    "precisao_validacao", "recall_validacao", "f1_validacao",
])
configuracao_modelagem["arquitetura_implementada"] = True
configuracao_modelagem["biblioteca_rede"] = "PyTorch"
configuracao_modelagem["treinamento_executado"] = treinamento_executado
configuracao_modelagem["melhor_epoca"] = melhor_epoca
configuracao_modelagem["parametros_treinamento"] = {
    "semente": SEMENTE, "epocas_maximas": EPOCAS, "paciencia": PACIENCIA,
    "taxa_aprendizado": TAXA_APRENDIZADO, "batch_size": BATCH_SIZE,
    "limiar_inicial": LIMIAR_INICIAL, "criterio_selecao": "menor perda de validação",
}
configuracao_modelagem["status_preparacao"] = (
    "modelo treinado; pesos da melhor época restaurados; avaliação final pendente"
    if treinamento_executado
    else "arquitetura e rotina prontas; treinamento pendente de negativos confirmados"
)


Nenhuma época executada: faltam as duas classes nos conjuntos de desenvolvimento.


## 85. Curvas de validação e situação do modelo

Quando houver treinamento, os gráficos mostram a evolução das perdas e da AP na validação. A proporção de positivos da validação aparece como referência para AP de uma ordenação sem informação útil. Nenhuma previsão ou curva é fabricada quando o treinamento não pode ser executado.

O resumo da melhor época corresponde aos pesos restaurados. `predicoes_validacao` mantém as probabilidades dessa mesma época ligadas aos metadados, para a futura análise de limiar. A avaliação do teste e a exportação do modelo pertencem à etapa seguinte.


In [88]:
if treinamento_executado:
    import matplotlib.pyplot as plt

    assert not historico_treinamento.empty
    assert melhor_epoca in historico_treinamento["epoca"].values
    assert historico_treinamento["perda_validacao"].min() == melhor_perda_validacao
    display(historico_treinamento)
    display(historico_treinamento.loc[historico_treinamento["epoca"].eq(melhor_epoca)])

    figura, eixos = plt.subplots(1, 2, figsize=(12, 4))
    eixos[0].plot(historico_treinamento["epoca"], historico_treinamento["perda_treino"], label="Treino")
    eixos[0].plot(historico_treinamento["epoca"], historico_treinamento["perda_validacao"], label="Validação")
    eixos[0].axvline(melhor_epoca, color="gray", linestyle=":", label="Melhor época")
    eixos[0].set(xlabel="Época", ylabel="Perda binária", title="Evolução da perda")
    eixos[0].legend()

    eixos[1].plot(historico_treinamento["epoca"], historico_treinamento["ap_validacao"], label="AP da validação")
    eixos[1].axhline(float(sequencias["validacao"]["y"].mean()), color="gray", linestyle="--", label="Proporção de positivos")
    eixos[1].set(xlabel="Época", ylabel="Average Precision", title="Ordenação dos alertas", ylim=(0, 1.05))
    eixos[1].legend()
    figura.tight_layout()
    plt.show()
    plt.close(figura)
    print("Melhor época restaurada:", melhor_epoca)
else:
    assert historico_treinamento.empty
    assert predicoes_validacao.empty
    assert melhores_pesos is None
    print("Sem métricas de validação: o modelo ainda não foi treinado.")

print(configuracao_modelagem["status_preparacao"])


Sem métricas de validação: o modelo ainda não foi treinado.
arquitetura e rotina prontas; treinamento pendente de negativos confirmados


### Situação da etapa 5

A arquitetura proposta, a rotina de treinamento e o acompanhamento da validação estão implementados. A execução usa somente treino e validação e restaura os melhores pesos quando houver treinamento.

Com os rótulos atuais, o treinamento permanece pendente de negativos confirmados. A estrutura da rede pode ser verificada, mas não há modelo treinado nem desempenho de validação a apresentar.

Após a inclusão fundamentada de negativos, a reexecução das etapas de rotulagem, seleção e sequências atualizará os conjuntos. A rotina desta etapa poderá então executar o treinamento e produzir o histórico. Os pesos treinados permanecerão em memória até a etapa de exportação.
